# Notebook 09 — Modelling Dataset and Experimental Protocol

## Purpose

This notebook defines the forecasting experiment **before any predictive
model is trained**.

The objective is to convert the previously validated SOLETE preprocessing,
Gregorian, astronomical, and Panchang feature pipelines into a controlled
time-series forecasting experiment.

The central research question is:

> **Can deterministic Vedic / lunisolar calendar features improve
> short-term renewable-energy forecasting beyond historical measurements,
> conventional Gregorian time features, and standard astronomical
> features?**

A secondary question is:

> **If Panchang features improve forecasting, do they provide information
> beyond the underlying continuous solar and lunar astronomical quantities
> from which the calendar is derived?**

---

## Experimental Principle

The experiment will use a controlled feature-ablation design.

The feature families are introduced progressively.

### F0 — Forecasting Baseline Features

Historical / autoregressive renewable-energy measurements and appropriate
meteorological predictors.

This represents the conventional forecasting baseline without explicit
calendar or astronomical information.

### F1 — Gregorian Time Features

F0 plus deterministic Gregorian calendar features such as:

- hour / minute of day;
- day of week;
- day of year;
- weekend indicator;
- daily cyclic encoding;
- weekly cyclic encoding;
- annual cyclic encoding.

### F2 — Solar Astronomical Features

F1 plus independently computed deterministic solar geometry.

The validated F2 solar feature set contains:

- solar elevation;
- cosine of solar zenith;
- sine of solar azimuth;
- cosine of solar azimuth;
- daylight indicator.

### F3 — Continuous Lunar / Sun–Moon Astronomy

F2 plus the validated continuous lunar and Sun–Moon astronomical features:

- Moon–Sun phase sine;
- Moon–Sun phase cosine;
- Moon longitude sine;
- Moon longitude cosine;
- Sun ecliptic longitude sine;
- Sun ecliptic longitude cosine;
- Moon ecliptic latitude;
- Moon illumination fraction;
- Earth–Moon distance.

F3 is especially important because it provides a continuous astronomical
control before introducing the Panchang representation.

### F4 — Vedic / Panchang Features

F3 plus the validated categorical Panchang quantities:

- Tithi;
- Paksha;
- Karana;
- Nakshatra;
- Yoga.

This allows the comparison:

$$
F3 \rightarrow F4
$$

to test whether the structured Panchang representation contributes useful
forecasting information beyond the continuous Sun–Moon astronomy already
available to the model.

---

## Hypotheses

### Null Hypothesis — H0

Adding Vedic / Panchang calendar features does not produce a meaningful
out-of-sample forecasting improvement beyond historical, Gregorian, solar,
and continuous lunar astronomical information.

### Alternative Hypothesis — H1

Adding Vedic / Panchang calendar features produces a meaningful
out-of-sample forecasting improvement beyond those baseline feature
families.

No improvement will be assumed in advance.

---

## Forecasting and Validation Rules

The experiment will follow the following principles:

1. **Chronological validation only**

   Training observations must always precede validation and test
   observations.

   Ordinary random train/test splitting and ordinary shuffled k-fold
   cross-validation will not be used.

2. **No future-information leakage**

   Every predictor must be evaluated according to whether it would actually
   be available at the forecast issue time.

3. **Deterministic future-known features are allowed**

   Gregorian calendar features, solar astronomy, lunar astronomy, and
   Panchang quantities can be calculated in advance for future timestamps.

4. **Observed meteorological variables require special treatment**

   Same-timestamp observed quantities such as GHI, POA irradiance,
   temperature, humidity, and wind conditions cannot automatically be
   treated as future-known predictors.

   Their use must depend on the forecasting scenario:

   - historical lagged observations; or
   - explicitly assumed meteorological forecasts.

5. **Feature engineering must be separated from model fitting**

   Scaling, categorical encoding, feature selection, and other learned
   transformations must be fitted using training data only.

6. **The experiment definition must be fixed before examining comparative
   model results**

   Forecast horizons, splits, metrics, feature families, and baseline
   definitions will be decided in this notebook before the main F0–F4
   comparison is performed.

---

## Reusable Pipeline Available

The exploratory notebook logic has already been transferred into reusable
source modules:

- `src/data_loader.py`
- `src/preprocessing.py`
- `src/astronomy_features.py`
- `src/vedic_calendar.py`

These modules reproduce the previously validated notebook calculations and
will be used to construct the modelling dataset in this notebook.

---

## Decisions Still Required

Before generating targets or fitting models, this notebook must explicitly
define:

1. primary renewable-energy target;
2. forecast horizon or horizons;
3. forecast-information scenario;
4. autoregressive lag structure;
5. chronological train / validation / test periods;
6. baseline forecasting methods;
7. evaluation metrics;
8. categorical encoding strategy for F4;
9. exact model-input columns for F0, F1, F2, F3, and F4.

These decisions will be made before model comparison begins.

# 1. Forecast Target Definition

## Primary Target

The primary forecasting target for the initial controlled experiment is:

`P_Solar[kW]`

representing photovoltaic power output in the SOLETE dataset.

The initial modelling experiments will therefore estimate future solar
power:

$$
P_{\text{Solar}}(t+h)
$$

using information available at forecast issue time $t$, where $h$
denotes the forecast horizon.

---

## Why Solar Power Is the Primary Target

Solar power is selected first because it provides the clearest setting for
testing the progressive feature hierarchy developed in this project.

The feature families contain increasingly detailed deterministic temporal
and astronomical information:

$$
\text{Gregorian time}
\rightarrow
\text{solar astronomy}
\rightarrow
\text{lunar astronomy}
\rightarrow
\text{Panchang}
$$

For photovoltaic generation, solar geometry has an explicit physical
relationship with the daily and seasonal availability of solar radiation.

This makes the controlled comparison:

$$
F0 \rightarrow F1 \rightarrow F2 \rightarrow F3 \rightarrow F4
$$

particularly interpretable.

The experiment can first establish whether:

1. Gregorian timing improves the historical / meteorological baseline;
2. independently computed solar geometry provides additional information;
3. lunar and Sun–Moon astronomy adds anything beyond solar geometry;
4. Panchang categories add anything beyond the underlying continuous
   astronomical variables.

---

## Secondary Target

The SOLETE wind-turbine power variable:

`P_Gaia[kW]`

will be retained as a secondary target for a later experiment.

Using wind power as a second target can test whether any observed advantage
from lunar or Panchang features is specific to photovoltaic generation or
also appears in a different renewable-energy process.

The solar and wind experiments should remain separate rather than combining
their targets into a single modelling task.

---

## Target Convention

For the primary experiment:

- target variable: `P_Solar[kW]`;
- modelling resolution: **5 minutes**;
- target at forecast issue time: $P_{\text{Solar}}(t)$;
- forecast target: $P_{\text{Solar}}(t+h)$;
- $h > 0$ must always hold for the forecasting experiment.

No same-timestamp prediction will be treated as forecasting.

The exact value or values of $h$ will be defined next.

# 2. Forecast Horizon Definition

## Multi-Horizon Short-Term Forecasting

The primary modelling resolution is 5 minutes.

The controlled forecasting experiment will evaluate three future horizons:

| Forecast horizon | 5-minute steps |
|---|---:|
| 15 minutes | 3 |
| 30 minutes | 6 |
| 60 minutes | 12 |

The corresponding targets are:

$$
y_t^{(15)} =
P_{\text{Solar}}(t + 15\text{ min})
$$

$$
y_t^{(30)} =
P_{\text{Solar}}(t + 30\text{ min})
$$

$$
y_t^{(60)} =
P_{\text{Solar}}(t + 60\text{ min})
$$

---

## Primary Horizon

The primary forecast horizon will be:

**60 minutes ahead**

or equivalently:

**12 steps ahead at 5-minute resolution.**

This horizon is selected as the main comparison point because it provides
a meaningful short-term forecasting task while reducing the dominance of
very-near-term persistence.

At extremely short horizons, recent PV power observations may already
provide most of the predictive information, potentially masking smaller
contributions from deterministic temporal, astronomical, or Panchang
features.

A 60-minute horizon gives the deterministic feature families more
opportunity to contribute while still remaining within the short-term
forecasting setting.

---

## Secondary Horizons

The following additional horizons will be evaluated:

- **15 minutes ahead**
- **30 minutes ahead**

These horizons allow the experiment to test whether the contribution of
the different feature families changes with forecast distance.

For example, an improvement that appears only at 60 minutes but not at
15 minutes would indicate a different forecasting role than an
improvement that is consistent across all horizons.

---

## Optional 5-Minute Sanity Check

A 5-minute-ahead forecast may later be included as a diagnostic experiment.

However, it will not be treated as the primary research horizon because
persistence and very recent autoregressive information are expected to be
particularly strong at this timescale.

---

## Direct Forecasting Convention

Each horizon will initially be treated as a separate **direct forecasting
problem**.

For a forecast issued at time $t$:

- input features are constructed using information available at or before
  $t$;
- deterministic future-known calendar / astronomical information may be
  included according to the experimental feature-family definition;
- the target is shifted forward by the selected horizon.

For example, for the 60-minute experiment:

$$
X_t
\rightarrow
P_{\text{Solar}}(t + 60\text{ min})
$$

rather than recursively predicting each intermediate 5-minute step.

This avoids compounding recursive forecast errors and keeps the F0–F4
feature-ablation comparison straightforward.

---

## Horizon Policy

The feature definitions, chronological splits, model classes, and
evaluation metrics should remain consistent across the 15-, 30-, and
60-minute experiments.

The **60-minute horizon will be treated as the primary reported result**,
while the 15- and 30-minute horizons will provide supporting
multi-horizon evidence.

# 3. Forecast-Information Availability and Leakage Policy

## Forecast Issue Time

For every training example, let:

- $t$ = forecast issue time;
- $h$ = forecast horizon;
- $t+h$ = forecast-valid time.

The prediction task is:

$$
X_t
\rightarrow
P_{\text{Solar}}(t+h)
$$

Only information that would legitimately be available when the forecast is
issued at time $t$ may be used as an observed predictor.

---

## Primary Information Scenario

The primary experiment will use a:

**measurement-history + deterministic future-known information scenario**

This means that historical observations available at or before the forecast
issue time may be used, while future observed meteorological measurements
are prohibited.

Deterministic calendar and astronomical quantities are treated differently
because they can be calculated in advance.

---

## Observed Variables Allowed at Forecast Issue Time

Observed SOLETE measurements may be used only when their timestamp satisfies:

$$
\text{observation time} \leq t
$$

Examples include historical or current values of:

- `P_Solar[kW]`;
- temperature;
- humidity;
- pressure;
- wind speed;
- wind direction;
- GHI;
- POA irradiance.

Their exact lag structure will be defined later.

An observation recorded at the forecast issue time $t$ may be used if it
is assumed to be available when the forecast is issued.

No observed measurement with timestamp greater than $t$ may enter the input
feature vector.

---

## Prohibited Future Observations

For a forecast of:

$$
P_{\text{Solar}}(t+h)
$$

the following observed values at $t+h$, or at any intermediate future
time after $t$, are **not available** in the primary experiment:

- future observed `P_Solar[kW]`;
- future observed GHI;
- future observed POA irradiance;
- future observed temperature;
- future observed humidity;
- future observed pressure;
- future observed wind speed;
- future observed wind direction.

Using these measured future values would introduce look-ahead leakage.

In particular, same-valid-time observed GHI or POA must not be used to
predict solar power at that same future valid time.

---

## Deterministic Future-Known Features

The following feature families can be calculated exactly in advance:

- Gregorian calendar features;
- solar astronomical geometry;
- lunar and Sun–Moon astronomical quantities;
- Panchang quantities.

Therefore these deterministic features may be evaluated at the
**forecast-valid timestamp $t+h$** even though the forecast is issued at
time $t$.

For example, in the 60-minute experiment:

$$
X_t
\rightarrow
P_{\text{Solar}}(t + 60\text{ min})
$$

the F1–F4 deterministic features will describe:

$$
t + 60\text{ min}
$$

rather than only the issue timestamp $t$.

This does not constitute future-information leakage because these quantities
are deterministic functions of time and can be calculated before the future
observation occurs.

---

## Feature-Time Convention

The modelling table will distinguish two types of predictors.

### Historical / Observed Predictors

Evaluated at:

$$
t,\;
t-5\text{ min},\;
t-10\text{ min},\;
\ldots
$$

depending on the selected autoregressive lag structure.

### Deterministic Predictors

Evaluated primarily at the forecast-valid time:

$$
t+h
$$

This includes F1, F2, F3, and F4 calendar / astronomical information.

---

## Why This Distinction Matters

Without this distinction, the experiment could unintentionally compare:

- historical measurements describing the present;
- astronomical features describing the present;
- a target describing the future.

Instead, evaluating deterministic features at $t+h$ allows them to describe
the actual future time being forecast while still respecting the
information available at issue time.

This is particularly important for solar geometry because the Sun's
position can change substantially over a 60-minute forecast horizon.

---

## Optional Future Scenario

A separate experiment could later assume that operational weather forecasts
are available for $t+h$.

Such a scenario could include forecasted:

- irradiance;
- temperature;
- humidity;
- wind variables.

However, actual future SOLETE observations must never be substituted for
weather forecasts and described as operationally available information.

The primary experiment will therefore remain based on:

**historical observed measurements + future-known deterministic features.**

# 4. Historical / Autoregressive Input Window

## Forecast-Issue-Time History

The primary F0 baseline will use a fixed **60-minute recent-history window**
at the native 5-minute SOLETE resolution.

For a forecast issued at time $t$, the recent-history timestamps are:

$$
t,\;
t-5\text{ min},\;
t-10\text{ min},\;
\ldots,\;
t-60\text{ min}
$$

This corresponds to:

$$
13
$$

observed time points including the current issue-time observation.

The corresponding lag indices are:

$$
0,1,2,\ldots,12
$$

where one lag step represents 5 minutes.

---

## Observed Variables in the Recent-History Window

The recent-history window will initially contain the following observed
quantities:

| Variable | Role |
|---|---|
| `P_Solar[kW]` | autoregressive PV history |
| `GHI[kW1m2]` | historical global horizontal irradiance |
| `POA Irr[kW1m2]` | historical plane-of-array irradiance |
| `TEMPERATURE[degC]` | historical temperature |
| `Humidity_clean` | cleaned historical humidity |
| `Pressure_clean[mbar]` | cleaned historical pressure |
| `WIND_SPEED[m1s]` | historical wind speed |
| `wind_direction_sin` | circular wind-direction component |
| `wind_direction_cos` | circular wind-direction component |

Only observations at or before $t$ are included.

No measured value after the forecast issue time is allowed.

---

## Seasonal Autoregressive Reference

In addition to the recent 60-minute history, the baseline will include one
daily seasonal PV reference:

$$
P_{\text{Solar}}(t-24\text{ h})
$$

At 5-minute resolution this corresponds to:

$$
288
$$

time steps.

This feature provides the baseline model with a simple same-time-previous-day
reference without introducing future information.

The 24-hour lag will initially be applied only to `P_Solar[kW]`.

---

## Why a 60-Minute Recent Window?

A one-hour history provides information about:

- current PV production level;
- recent ramping behavior;
- recent irradiance conditions;
- recent meteorological state;
- short-term cloud-related changes.

It is also long enough to support the primary 60-minute-ahead forecasting
task without creating an unnecessarily large historical feature space.

Using the complete 5-minute sequence rather than only a few manually
selected lags avoids choosing individual lag positions after examining
model results.

---

## Common History Across Forecast Horizons

The same historical input definition will be used for all primary forecast
horizons:

$$
15,\;30,\;60\text{ minutes}
$$

Thus the comparison across horizons changes the future target time, not the
amount of historical information exposed to the model.

For every horizon:

$$
\text{observed history} \leq t
$$

while the target is:

$$
P_{\text{Solar}}(t+h)
$$

---

## F0 Baseline Definition

The initial F0 feature family therefore consists of:

$$
\text{recent observed history}
+
P_{\text{Solar}}(t-24\text{ h})
$$

with no explicit Gregorian, astronomical, or Panchang variables.

The later feature families will be nested extensions:

$$
F0 \subset F1 \subset F2 \subset F3 \subset F4
$$

so that improvements can be attributed to the newly introduced feature
family rather than to a change in historical information.

---

## Scope Decision

Longer history windows, additional daily lags, rolling statistics, or
alternative lag selections will not be introduced into the primary
F0–F4 comparison before the initial results are obtained.

They may later be evaluated as sensitivity analyses if necessary.

# 5. Chronological Train / Validation / Test Split

## Split Strategy

The forecasting experiment will use a fixed chronological holdout design.

No observations will be randomly shuffled between training, validation, and
test sets.

The SOLETE 5-minute dataset spans approximately:

`2018-06-01` to `2019-09-01`

The primary split will be:

| Split | Period |
|---|---|
| Training | 2018-06-01 to 2019-03-31 |
| Validation | 2019-04-01 to 2019-05-31 |
| Test | 2019-06-01 to 2019-08-31 |

The final timestamp at `2019-09-01 00:00 UTC` belongs to the dataset
boundary and will not be treated as a separate September evaluation period.

---

## Why This Split Is Used

The training period contains approximately ten months of historical data,
including:

- summer;
- autumn;
- winter;
- early spring.

It also contains the previous summer of 2018, which is important because
the final test period covers summer 2019.

The validation period covers April and May 2019 and will be used for:

- model selection;
- hyperparameter selection;
- early stopping where applicable;
- preprocessing choices that must be learned from data.

The final test period covers June through August 2019.

The test set must remain untouched during model development and tuning.

---

## Split Assignment Uses Forecast-Valid Time

Each modelling example is associated with:

- issue time: $t$;
- forecast horizon: $h$;
- forecast-valid time: $t+h$.

The split assignment will be based on the **forecast-valid time**:

$$
t_{\text{valid}} = t+h
$$

rather than only the issue time $t$.

For example, if a 60-minute forecast is issued at:

`2019-05-31 23:30 UTC`

its target occurs at:

`2019-06-01 00:30 UTC`

and therefore the example belongs to the **test set**, not the validation
set.

This prevents targets from crossing chronological split boundaries.

---

## Split Boundaries

Using half-open time intervals, the intended split definition is:

### Training

$$
2018\text{-}06\text{-}01
\leq
t_{\text{valid}}
<
2019\text{-}04\text{-}01
$$

### Validation

$$
2019\text{-}04\text{-}01
\leq
t_{\text{valid}}
<
2019\text{-}06\text{-}01
$$

### Test

$$
2019\text{-}06\text{-}01
\leq
t_{\text{valid}}
<
2019\text{-}09\text{-}01
$$

All timestamps are interpreted in UTC.

---

## Historical Features Near Split Boundaries

Historical predictors for a modelling row may come from times earlier than
the row's forecast-valid split.

This is allowed because those observations would genuinely have been
available at forecast issue time.

For example, the first test forecast may legitimately use historical
measurements from the end of the validation period.

However:

- no future observed value may cross backward into an earlier split;
- model fitting must use training targets only;
- validation targets may be used only for model selection;
- test targets must remain completely unseen until final evaluation.

---

## Initial-Lag Loss

The F0 definition includes:

$$
P_{\text{Solar}}(t-24\text{ h})
$$

and a 60-minute recent-history window.

Therefore the earliest modelling rows will naturally be unavailable until
sufficient historical context exists.

These initial rows will be dropped after lag construction rather than filled
artificially.

---

## Fixed Split Policy Across Horizons

The same calendar boundaries will be used for all forecast horizons:

- 15 minutes;
- 30 minutes;
- 60 minutes.

Because split membership is determined by $t+h$, the exact issue-time rows
included in each split may differ slightly by horizon.

This preserves a consistent forecast-valid evaluation period across all
horizons.

---

## Test-Set Policy

The test set will be used only after:

1. feature construction is finalized;
2. model classes are selected;
3. hyperparameters are chosen using training and validation data;
4. preprocessing transformations are fitted without access to test targets.

No modelling decision should be changed in response to test-set
performance during the primary experiment.

# 6. Evaluation Metrics

## Primary Metric — Mean Absolute Error

The primary forecasting metric will be **Mean Absolute Error (MAE)**.

For $N$ forecast observations:

$$
\text{MAE}
=
\frac{1}{N}
\sum_{i=1}^{N}
\left|
y_i-\hat{y}_i
\right|
$$

where:

- $y_i$ is the observed solar power;
- $\hat{y}_i$ is the corresponding forecast.

MAE will be reported in:

`kW`

Because the target itself is measured in kilowatts, MAE provides a directly
interpretable measure of typical forecast error.

Lower values indicate better forecasting performance.

---

## Secondary Metric — Root Mean Squared Error

Root Mean Squared Error (RMSE) will also be reported:

$$
\text{RMSE}
=
\sqrt{
\frac{1}{N}
\sum_{i=1}^{N}
\left(
y_i-\hat{y}_i
\right)^2
}
$$

RMSE gives greater weight to large forecast errors than MAE.

Reporting both MAE and RMSE therefore distinguishes:

- typical absolute forecast error;
- sensitivity to larger forecast failures.

RMSE will also be reported in:

`kW`

---

## Normalized Error Metrics

To make the results easier to interpret relative to the PV system size,
normalized versions of MAE and RMSE will also be reported.

The SOLETE PV system has a nominal capacity of approximately:

$$
P_{\text{rated}} = 10\text{ kW}
$$

Normalized MAE is defined as:

$$
\text{nMAE}
=
\frac{\text{MAE}}{P_{\text{rated}}}
\times 100
$$

and normalized RMSE as:

$$
\text{nRMSE}
=
\frac{\text{RMSE}}{P_{\text{rated}}}
\times 100
$$

These quantities will be reported as percentages of nominal PV capacity.

---

## Supplementary Metric — Coefficient of Determination

The coefficient of determination, $R^2$, may also be reported as a
supplementary descriptive metric:

$$
R^2
=
1
-
\frac{
\sum_i (y_i-\hat{y}_i)^2
}{
\sum_i (y_i-\bar{y})^2
}
$$

However, $R^2$ will not be used as the primary criterion for comparing
feature families.

The principal comparison will remain based on forecast-error metrics.

---

## Metrics Not Used as Primary Measures

### MAPE

Mean Absolute Percentage Error will **not** be used as a primary metric.

Solar-power observations contain many zero or near-zero values, particularly
during nighttime.

Because MAPE contains division by the observed target:

$$
\left|
\frac{y_i-\hat{y}_i}{y_i}
\right|
$$

it becomes undefined or unstable when $y_i$ is zero or very small.

Therefore MAPE could give misleading conclusions for this dataset.

---

## All-Time and Daylight Evaluation

Solar power has a strong day/night structure.

A model can obtain very small nighttime errors simply by predicting values
close to zero.

Therefore forecast performance will be reported for two evaluation scopes.

### 1. All valid timestamps

This measures operational performance across the complete chronological
evaluation period.

### 2. Daylight timestamps

A separate daylight evaluation will include observations for which the
deterministically calculated solar position at the forecast-valid time
satisfies:

$$
\text{solar elevation} > 0^\circ
$$

This daylight subset focuses the evaluation on periods where meaningful
photovoltaic generation is physically possible.

The daylight definition is based on deterministic solar geometry rather
than on the observed target value, avoiding target-dependent filtering.

---

## Feature-Ablation Improvement

For each model and forecast horizon, the effect of adding a new feature
family will be measured using the change in forecast error.

For example, the relative MAE change from F3 to F4 is:

$$
\Delta_{\text{MAE}}^{F3 \rightarrow F4}
=
\frac{
\text{MAE}_{F3}
-
\text{MAE}_{F4}
}{
\text{MAE}_{F3}
}
\times 100
$$

A positive value indicates lower MAE after adding the F4 Panchang features.

The same comparison can be performed for:

$$
F0 \rightarrow F1
$$

$$
F1 \rightarrow F2
$$

$$
F2 \rightarrow F3
$$

$$
F3 \rightarrow F4
$$

The most important comparison for the main research question is:

$$
F3 \rightarrow F4
$$

because F3 already contains the continuous astronomical information from
which the Panchang quantities are derived.

---

## Metric Policy Across Horizons

The same evaluation metrics will be used for:

- 15-minute forecasts;
- 30-minute forecasts;
- 60-minute forecasts.

The primary reported comparison will use:

**60-minute-ahead MAE**

with RMSE, normalized errors, daylight-only performance, and shorter
forecast horizons providing supporting evidence.

No evaluation metric will be selected or removed based on which one produces
the most favorable F4 result.

# 7. Non-ML Forecasting Baselines

## Purpose of Reference Baselines

Before evaluating machine-learning models, the experiment will establish
simple forecasting benchmarks.

These baselines provide reference performance against which the F0–F4
models can be compared.

A complex model should not be considered useful merely because it produces
numerically reasonable forecasts; it should also improve upon simple
forecasting rules that require little or no model fitting.

---

## Baseline B1 — Persistence

The first baseline assumes that the most recently observed photovoltaic
power remains unchanged over the forecast horizon.

For a forecast issued at time $t$:

$$
\hat{P}_{\text{Solar}}(t+h)
=
P_{\text{Solar}}(t)
$$

This baseline uses only information that is available at the forecast issue
time.

It will be evaluated independently for:

- 15-minute forecasts;
- 30-minute forecasts;
- 60-minute forecasts.

Persistence is expected to be particularly competitive at short horizons
because PV output often changes gradually over consecutive 5-minute
intervals.

---

## Baseline B2 — Daily Persistence

The second baseline uses the photovoltaic output observed at the same
forecast-valid clock time on the previous day.

For a target at $t+h$:

$$
\hat{P}_{\text{Solar}}(t+h)
=
P_{\text{Solar}}(t+h-24\text{ h})
$$

At the selected forecast horizons, this historical value is already
available when the forecast is issued at time $t$.

For example, for a 60-minute-ahead forecast:

$$
\hat{P}_{\text{Solar}}(t+60\text{ min})
=
P_{\text{Solar}}(t+60\text{ min}-24\text{ h})
$$

Daily persistence provides a simple seasonal reference because photovoltaic
generation is strongly related to time of day and solar position.

---

## Difference Between F0's 24-Hour Lag and Daily Persistence

The F0 modelling feature set includes:

$$
P_{\text{Solar}}(t-24\text{ h})
$$

as an input predictor.

Daily persistence is instead a complete forecast rule using:

$$
P_{\text{Solar}}(t+h-24\text{ h})
$$

The two quantities are therefore not identical when $h>0$.

The F0 model may learn how to combine its historical inputs with other
observed variables, whereas daily persistence contains no fitted model.

---

## Optional Baseline B3 — Training-Set Mean

A simple constant baseline may also be retained as a diagnostic reference.

For each modelling split, the prediction is based on the mean target value
estimated from the training data only:

$$
\hat{y}
=
\bar{y}_{\text{train}}
$$

This baseline is not expected to be competitive for solar forecasting, but
it can provide a useful lower reference for checking that the modelling
pipeline is functioning correctly.

The validation or test target distribution must not be used when estimating
this value.

---

## Baseline Evaluation

Each baseline will be evaluated using the same metrics as the learned
models:

- MAE;
- RMSE;
- nMAE;
- nRMSE;
- supplementary $R^2$ where appropriate.

Performance will be reported for:

1. all valid timestamps;
2. daylight-only timestamps.

The same chronological validation and test periods will be used.

---

## Baseline Role in the Experiment

The reference hierarchy is therefore:

$$
\text{Persistence}
$$

$$
\text{Daily Persistence}
$$

followed by the learned feature-family experiments:

$$
F0
\rightarrow
F1
\rightarrow
F2
\rightarrow
F3
\rightarrow
F4
$$

The simple baselines answer:

> Can the learned forecasting model outperform straightforward historical
> reference rules?

The controlled F0–F4 comparison then answers:

> Once a forecasting model is established, what additional information is
> contributed by Gregorian, solar, lunar, and Panchang feature families?

These are distinct questions and will be reported separately.

# 8. F4 Panchang Categorical Encoding Strategy

## Raw F4 Representation

The validated raw F4 feature family contains five categorical variables:

| Feature | Number of categories |
|---|---:|
| Tithi | 30 |
| Paksha | 2 |
| Karana | 11 |
| Nakshatra | 27 |
| Yoga | 27 |

The raw variables are stored as categorical values rather than continuous
numeric measurements.

---

## Primary Encoding — One-Hot Encoding

The primary F4 experiment will use **one-hot encoding**.

For a categorical variable with $K$ possible states, one-hot encoding
creates $K$ binary indicator variables.

For example, Tithi will be represented conceptually as:

$$
\text{Tithi}
\rightarrow
[
I(T_1),
I(T_2),
\ldots,
I(T_{30})
]
$$

where exactly one Tithi indicator is active for each observation.

The same principle will be used for:

- Paksha;
- Karana;
- Nakshatra;
- Yoga.

---

## Why Integer Encoding Is Not Used as the Primary Representation

Although Tithi, Nakshatra, and Yoga have conventional numbers, those numbers
must not automatically be interpreted as ordinary continuous magnitudes.

For example, encoding:

$$
\text{Ashwini}=1,\quad
\text{Bharani}=2,\quad
\ldots,\quad
\text{Revati}=27
$$

and passing those values directly to a model could incorrectly imply that:

- Nakshatra 20 is numerically twice Nakshatra 10;
- the distance between every neighboring category is meaningful in an
  ordinary linear sense;
- Nakshatra 27 is far from Nakshatra 1 despite the circular boundary.

Similar problems apply to Tithi and Yoga.

Therefore the category numbers may be retained for auditing and naming, but
they will not be used directly as continuous model inputs in the primary
experiment.

---

## Why One-Hot Encoding Fits the F3 → F4 Experiment

F3 already contains continuous astronomical representations such as:

- Moon-Sun phase;
- solar and lunar longitudes;
- lunar illumination;
- lunar latitude;
- Earth-Moon distance.

The purpose of F4 is therefore not to introduce another continuous angular
representation.

Instead, F4 tests whether the **structured Panchang state representation**
adds useful information beyond the continuous astronomical variables.

One-hot encoding preserves this distinction:

$$
F3
=
\text{continuous astronomy}
$$

$$
F4
=
F3
+
\text{categorical Panchang states}
$$

---

## Fixed Category Vocabulary

The possible Panchang categories are defined deterministically by the
calendar system rather than learned from the target data.

The canonical category sets are therefore known in advance:

- 30 Tithis;
- 2 Pakshas;
- 11 Karanas;
- 27 Nakshatras;
- 27 Yogas.

The modelling pipeline should use these fixed category vocabularies.

This avoids accidental dependence on category ordering discovered from the
validation or test sets.

---

## Number of F4 Indicator Columns

Using full one-hot encoding gives:

$$
30 + 2 + 11 + 27 + 27 = 97
$$

binary Panchang indicator columns.

Therefore F4 introduces up to:

$$
97
$$

additional binary features beyond F3.

This dimensionality is modest relative to the number of available
5-minute observations.

---

## Full Encoding Without Arbitrary Ordinal Assumptions

The primary experiment will retain all one-hot categories rather than
dropping categories purely to create an artificial numeric ordering.

For models that require special handling of redundant indicator columns,
that issue will be addressed within the model pipeline rather than by
changing the semantic definition of F4.

---

## Target Encoding Is Excluded

Target encoding will not be used in the primary F4 experiment.

Target encoding would estimate category values from historical
`P_Solar[kW]` observations and would therefore introduce an additional
learned transformation.

It would also make leakage control and chronological validation more
complicated.

The primary F4 experiment is intended to test the calendar categories
themselves rather than target-derived category statistics.

---

## Learned Embeddings Are Deferred

Neural-network embeddings may later be considered for advanced models.

They will not be used in the initial controlled experiment because they add
another learned representation layer and would make the basic F3 → F4
comparison less transparent.

---

## Circular Encoding Is Not the Primary F4 Encoding

Some Panchang variables possess cyclic structure.

For example:

$$
T_{30}
\rightarrow
T_1
$$

at the lunar-cycle boundary.

However, the primary F4 experiment will not encode these categories using
additional sine/cosine variables.

Continuous astronomical cyclic information is already present in F3.

Adding another continuous cyclic representation inside F4 could blur the
intended distinction between:

- underlying continuous astronomy;
- discrete Panchang state representation.

Circular Panchang encodings may later be studied as a sensitivity analysis.

---

## Encoding and Data Leakage

No target values are required for one-hot encoding.

Any software encoder used in the modelling pipeline must nevertheless be
configured without using validation or test target information.

The preferred implementation will use:

- the fixed canonical Panchang category vocabulary;
- one-hot output;
- safe handling of unexpected categories.

This keeps preprocessing deterministic and reproducible.

---

## Final F4 Encoding Policy

The primary experiment therefore uses:

**full one-hot encoding of Tithi, Paksha, Karana, Nakshatra, and Yoga**

with:

- no ordinal integer interpretation;
- no target encoding;
- no learned embeddings;
- no additional Panchang sine/cosine encoding in the primary comparison.

This preserves the intended controlled comparison:

$$
F3
\rightarrow
F4
$$

as:

$$
\text{continuous astronomy}
\rightarrow
\text{continuous astronomy + discrete Panchang states}
$$

# 9. Exact F0–F4 Feature Definitions

## Modelling-Row Convention

Each modelling row represents one forecast issued at time $t$ for a target
at forecast-valid time:

$$
t_{\text{valid}} = t+h
$$

The modelling table will therefore distinguish:

- **issue-time / historical observed features**, available at or before $t$;
- **future-known deterministic features**, evaluated at $t+h$;
- **target**, observed at $t+h$.

The target is:

$$
y_t^{(h)} = P_{\text{Solar}}(t+h)
$$

for forecast horizons of 15, 30, and 60 minutes.

---

## Metadata Columns

The modelling dataset may retain the following columns for indexing,
splitting, and auditing:

- `issue_time`
- `valid_time`
- `horizon_minutes`
- `target_P_Solar_kW`
- `split`

These columns are metadata or targets and are **not automatically model
features**.

---

# F0 — Historical / Autoregressive Baseline

## Recent-History Variables

The following nine observed quantities are included:

1. `P_Solar[kW]`
2. `GHI[kW1m2]`
3. `POA Irr[kW1m2]`
4. `TEMPERATURE[degC]`
5. `Humidity_clean`
6. `Pressure_clean[mbar]`
7. `WIND_SPEED[m1s]`
8. `wind_direction_sin`
9. `wind_direction_cos`

Each quantity is sampled at:

$$
t,\;
t-5,\;
t-10,\;
\ldots,\;
t-60\text{ min}
$$

corresponding to lag steps:

$$
0,1,\ldots,12
$$

At 13 timestamps for 9 variables, the recent-history block contains:

$$
9 \times 13 = 117
$$

features.

---

## Lag Naming Convention

Lagged columns will follow a consistent pattern.

For example:

```text
P_Solar_kw_lag_0
P_Solar_kw_lag_1
...
P_Solar_kw_lag_12

# 10. Initial Machine-Learning Model Family

## Purpose

The first controlled F0–F4 experiment will use a small set of standard
machine-learning models before introducing deep-learning or decomposition
methods.

The objective at this stage is not to find the most complex possible model.

Instead, the goal is to determine whether the progressive feature families:

$$
F0 \rightarrow F1 \rightarrow F2 \rightarrow F3 \rightarrow F4
$$

provide useful out-of-sample forecasting information under consistent
modelling conditions.

Using more than one model class also helps determine whether any observed
feature benefit is specific to a particular learning algorithm.

---

# Model M1 — Ridge Regression

The first learned model will be **Ridge Regression**.

Ridge regression estimates a linear relationship between the predictors and
future photovoltaic power while applying L2 regularization.

Conceptually:

$$
\hat{y}
=
\beta_0
+
\sum_{j=1}^{p}\beta_j x_j
$$

with the regularized objective:

$$
\sum_i
\left(
y_i-\hat{y}_i
\right)^2
+
\alpha
\sum_j
\beta_j^2
$$

where $\alpha$ controls the strength of regularization.

---

## Why Ridge Regression Is Included

Ridge regression provides a useful first reference because it is:

- computationally efficient;
- reproducible;
- relatively interpretable;
- suitable for high-dimensional feature sets;
- compatible with one-hot encoded Panchang variables;
- resistant to instability caused by correlated predictors.

Correlation is expected in this project because several feature families
describe related temporal or astronomical structure.

Examples include:

- hour of day and solar elevation;
- Moon phase and illumination;
- neighboring lagged observations;
- multiple one-hot Panchang categories.

Regularization therefore makes Ridge more appropriate than unregularized
ordinary least squares.

---

## Ridge Preprocessing

Continuous numerical predictors will be standardized using statistics
estimated from the training data only.

The transformation must therefore follow:

$$
\text{fit scaler on training data}
$$

then:

$$
\text{transform training, validation, and test data}
$$

Validation and test observations must not influence the fitted scaling
parameters.

One-hot categorical features do not require magnitude standardization in
the same sense as continuous variables.

The final implementation will use a leakage-safe modelling pipeline.

---

# Model M2 — Histogram Gradient Boosting

The second learned model will be:

**Histogram Gradient Boosting Regression**

implemented using scikit-learn's:

`HistGradientBoostingRegressor`

This model represents nonlinear relationships by sequentially constructing
decision trees that correct errors made by the previous ensemble.

---

## Why Histogram Gradient Boosting Is Included

Solar-power forecasting relationships are unlikely to be entirely linear.

Examples include:

- nonlinear dependence on solar elevation;
- changing irradiance-to-power relationships;
- interactions between recent PV history and solar geometry;
- weather-dependent production ramps;
- interactions between astronomical or calendar states.

A tree-based boosting model can learn nonlinearities and interactions
without manually specifying them.

Histogram-based boosting is also computationally suitable for the
approximately:

`131,617`

five-minute observations in the SOLETE modelling dataset.

---

## Scaling for Histogram Gradient Boosting

Tree-based models do not generally require standardization of continuous
features.

Therefore the same raw numerical values may be used for the histogram
gradient boosting model after the required cleaning and feature engineering.

The underlying F0–F4 semantic feature definitions must nevertheless remain
identical to those used by Ridge Regression.

---

# Why Two Initial Model Families Are Used

The two initial models represent substantially different modelling
assumptions.

### Ridge Regression

Tests whether the added features contribute useful information through
approximately linear relationships.

### Histogram Gradient Boosting

Tests whether the added features contribute through nonlinear relationships
or feature interactions.

If an F3 → F4 improvement appears under both models, the result is less
likely to be an artifact of one specific modelling architecture.

If an improvement occurs only under one model, that difference itself is
informative and should be reported rather than hidden.

---

# Model Comparison Policy

For every model class, the same feature hierarchy will be evaluated:

$$
F0,\;F1,\;F2,\;F3,\;F4
$$

The same chronological train / validation / test periods will also be used.

Therefore comparisons such as:

$$
F3 \rightarrow F4
$$

will always be made **within the same model class**.

For example:

$$
\text{Ridge}(F3)
\quad\text{vs}\quad
\text{Ridge}(F4)
$$

and:

$$
\text{HGBR}(F3)
\quad\text{vs}\quad
\text{HGBR}(F4)
$$

rather than comparing different models across different feature families.

---

# Hyperparameter Policy

Hyperparameters will be selected using the chronological training and
validation periods only.

The final test set will not be used for hyperparameter selection.

The initial tuning space should remain deliberately modest.

The purpose of the primary experiment is controlled feature comparison,
not exhaustive model optimization.

Where possible, the same hyperparameter-selection procedure will be used
across F0–F4.

---

# Models Deferred to Later Stages

The following approaches are intentionally deferred until after the initial
controlled tabular experiment:

- ordinary LSTM;
- BiLSTM;
- GRU;
- temporal convolutional networks;
- Transformers;
- VMD-based hybrid models;
- learned Panchang embeddings;
- XGBoost / LightGBM / CatBoost;
- large hyperparameter searches.

These methods may later be evaluated if the controlled baseline experiments
justify additional complexity.

In particular, VMD will not be introduced until the ordinary forecasting
pipeline and F0–F4 feature-ablation results are established.

No physical interpretation will be assigned automatically to VMD modes.

---

# Initial Modelling Hierarchy

The experimental hierarchy is therefore:

$$
\text{Persistence baselines}
$$

$$
\downarrow
$$

$$
\text{Ridge Regression}
$$

$$
\downarrow
$$

$$
\text{Histogram Gradient Boosting}
$$

with each learned model evaluated using:

$$
F0 \rightarrow F1 \rightarrow F2 \rightarrow F3 \rightarrow F4
$$

Only after these controlled experiments are complete will more advanced
forecasting architectures be considered.

# 11. Hyperparameter Selection and Model-Fitting Protocol

## Primary Validation Strategy

The primary experiment will use the fixed chronological validation period
defined earlier:

| Split | Period |
|---|---|
| Training | 2018-06-01 to 2019-03-31 |
| Validation | 2019-04-01 to 2019-05-31 |
| Test | 2019-06-01 to 2019-08-31 |

Hyperparameters will be selected using:

$$
\text{training data}
\rightarrow
\text{fit model}
\rightarrow
\text{validation data}
\rightarrow
\text{select hyperparameters}
$$

The test set will not participate in this process.

---

## Why a Fixed Chronological Validation Period Is Used

The main objective of the initial experiment is controlled comparison of:

$$
F0 \rightarrow F1 \rightarrow F2 \rightarrow F3 \rightarrow F4
$$

across several forecast horizons and two model families.

Using one fixed chronological validation period keeps this comparison:

- transparent;
- reproducible;
- computationally manageable;
- consistent across all feature families.

Random cross-validation will not be used because it would mix future and
past observations.

---

## Hyperparameter Selection Criterion

The primary hyperparameter-selection criterion will be:

**validation MAE**

for the corresponding forecast horizon.

Thus, for each candidate configuration:

$$
\text{best configuration}
=
\arg\min
\left(
\text{Validation MAE}
\right)
$$

RMSE and other metrics may also be recorded, but they will not replace MAE
as the primary tuning criterion.

---

## Separate Tuning by Model and Forecast Horizon

Hyperparameters may be tuned separately for:

- Ridge Regression;
- Histogram Gradient Boosting;

and for each forecast horizon:

- 15 minutes;
- 30 minutes;
- 60 minutes.

This is necessary because the optimal model complexity may change as the
forecast horizon changes.

---

## Feature-Family Fairness

Within a given model class and forecast horizon, the same hyperparameter
search procedure will be applied to:

$$
F0,\;F1,\;F2,\;F3,\;F4
$$

The search space must not be enlarged selectively for a feature family after
its performance is observed.

This ensures that an apparent feature-family improvement is not simply the
result of giving one configuration more tuning effort than another.

---

# Ridge Regression Tuning

The primary Ridge hyperparameter is the L2 regularization strength:

$$
\alpha
$$

A modest logarithmic candidate grid will be used:

```text
0.01
0.1
1
10
100

# 12. Final Experimental Protocol Freeze

## Primary Forecasting Task

The primary experiment forecasts:

$$
P_{\text{Solar}}(t+h)
$$

at 5-minute resolution.

The evaluated forecast horizons are:

- 15 minutes;
- 30 minutes;
- 60 minutes.

The **60-minute horizon** is the primary reported horizon.

---

## Forecasting Scenario

The primary experiment uses:

**historical observed measurements + future-known deterministic features**

Observed variables may only come from:

$$
\text{time} \leq t
$$

where $t$ is the forecast issue time.

Deterministic Gregorian, solar, lunar, and Panchang variables may be evaluated
at:

$$
t+h
$$

because they are known in advance.

Future observed meteorological measurements are prohibited.

---

## Historical Input Definition

F0 uses a 60-minute recent-history window at 5-minute resolution:

$$
t,\;t-5,\;t-10,\ldots,t-60\text{ min}
$$

for the following nine observed variables:

1. `P_Solar[kW]`
2. `GHI[kW1m2]`
3. `POA Irr[kW1m2]`
4. `TEMPERATURE[degC]`
5. `Humidity_clean`
6. `Pressure_clean[mbar]`
7. `WIND_SPEED[m1s]`
8. `wind_direction_sin`
9. `wind_direction_cos`

F0 also contains:

$$
P_{\text{Solar}}(t-24\text{ h})
$$

as a daily seasonal reference.

---

## Feature-Ablation Hierarchy

The primary feature hierarchy is fixed as:

$$
F0 \subset F1 \subset F2 \subset F3 \subset F4
$$

### F0

Historical / autoregressive observed features.

Approximate dimensionality:

$$
118
$$

### F1

F0 plus 15 Gregorian features evaluated at $t+h$.

Approximate dimensionality:

$$
133
$$

### F2

F1 plus 5 deterministic solar-geometry features.

Approximate dimensionality:

$$
138
$$

### F3

F2 plus 9 continuous lunar / Sun–Moon astronomical features.

Approximate dimensionality:

$$
147
$$

### F4

F3 plus one-hot encoded Panchang variables:

- Tithi;
- Paksha;
- Karana;
- Nakshatra;
- Yoga.

The full Panchang block contains:

$$
97
$$

binary indicators.

Approximate F4 dimensionality:

$$
244
$$

---

## Panchang Encoding

The primary F4 representation uses:

**full one-hot encoding**

The primary experiment will not use:

- ordinal Panchang numbers as continuous inputs;
- target encoding;
- learned embeddings;
- additional Panchang sine/cosine encodings.

This preserves the intended distinction:

$$
F3 = \text{continuous astronomy}
$$

$$
F4 = F3 + \text{discrete Panchang states}
$$

---

## Chronological Data Splits

Split membership is determined by forecast-valid time:

$$
t_{\text{valid}} = t+h
$$

### Training

`2018-06-01` to `2019-03-31`

### Validation

`2019-04-01` to `2019-05-31`

### Test

`2019-06-01` to `2019-08-31`

No random train/test split will be used.

---

## Reference Forecasts

The non-ML baselines are:

### Persistence

$$
\hat{P}_{\text{Solar}}(t+h)
=
P_{\text{Solar}}(t)
$$

### Daily Persistence

$$
\hat{P}_{\text{Solar}}(t+h)
=
P_{\text{Solar}}(t+h-24\text{ h})
$$

An optional training-mean baseline may also be retained as a pipeline
diagnostic.

---

## Initial Learned Models

The first controlled experiment will use:

1. Ridge Regression;
2. Histogram Gradient Boosting Regression.

More advanced models are deferred until the basic feature-ablation experiment
is complete.

---

## Model Selection

Hyperparameters will be selected using validation MAE only.

The test set will not be used for:

- feature selection;
- hyperparameter selection;
- scaling decisions;
- encoding decisions;
- early stopping;
- experimental redesign.

After hyperparameters are selected, the final model may be refitted using the
combined training and validation period before one final test evaluation.

---

## Evaluation Metrics

The primary metric is:

**MAE in kW**

Supporting metrics are:

- RMSE;
- nMAE;
- nRMSE;
- supplementary $R^2$.

Performance will be reported for:

1. all valid forecast timestamps;
2. daylight-only timestamps.

MAPE is excluded as a primary metric because solar power contains many zero
or near-zero observations.

---

## Primary Scientific Comparison

The most important ablation is:

$$
F3 \rightarrow F4
$$

because F3 already contains the continuous astronomical quantities underlying
the Panchang representation.

Therefore the primary research test is whether adding discrete Panchang
states improves forecasting beyond continuous solar and lunar astronomy.

No improvement is assumed in advance.

---

## Protocol Freeze

The following decisions are now fixed before model results are examined:

- target;
- forecast horizons;
- forecast-information scenario;
- historical lag window;
- F0–F4 feature definitions;
- Panchang encoding;
- chronological splits;
- forecasting baselines;
- initial model classes;
- hyperparameter-selection procedure;
- evaluation metrics.

Any later departure from this protocol must be documented explicitly as a
separate sensitivity analysis or follow-up experiment.

The next stage is therefore implementation of the modelling dataset according
to this frozen protocol.

In [1]:
# ============================================================
# Notebook 09 — Modelling Dataset and Experimental Protocol
# Step 1: Load the canonical 5-minute SOLETE dataset
# ============================================================

from pathlib import Path
import sys

import pandas as pd

# ------------------------------------------------------------
# 1. Locate the project root
# ------------------------------------------------------------
# This notebook is stored inside:
#
#   Project/notebooks/
#
# Depending on how VS Code launches the Jupyter kernel,
# Path.cwd() may be either the project directory itself or a
# directory somewhere inside it.
#
# Rather than assuming the current working directory, search
# upward until we find the project directory containing both:
#
#   src/
#   solete_dataset/
# ------------------------------------------------------------

search_locations = [Path.cwd(), *Path.cwd().parents]

PROJECT_ROOT = next(
    path
    for path in search_locations
    if (path / "src").is_dir()
    and (path / "solete_dataset").is_dir()
)

print("Project root:")
print(PROJECT_ROOT)


# ------------------------------------------------------------
# 2. Make the reusable src package importable
# ------------------------------------------------------------

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_solete


# ------------------------------------------------------------
# 3. Locate the 5-minute HDF5 dataset
# ------------------------------------------------------------
# We deliberately search for the file rather than hard-coding a
# filename here. This prevents the notebook from depending on an
# assumed file name while still requiring exactly one 5-minute
# HDF5 dataset to be present.
# ------------------------------------------------------------

DATA_DIR = PROJECT_ROOT / "solete_dataset"

five_minute_files = [
    path
    for path in DATA_DIR.rglob("*.h5")
    if "5min" in path.name.lower().replace("_", "")
]

print("\n5-minute HDF5 candidates:")
for path in five_minute_files:
    print(" -", path.relative_to(PROJECT_ROOT))

assert len(five_minute_files) == 1, (
    "Expected exactly one 5-minute SOLETE .h5 file, "
    f"but found {len(five_minute_files)}."
)

DATA_PATH_5MIN = five_minute_files[0]


# ------------------------------------------------------------
# 4. Load the dataset using our reusable loader
# ------------------------------------------------------------

df_5min = load_solete(DATA_PATH_5MIN)


# ------------------------------------------------------------
# 5. Minimal loading sanity check
# ------------------------------------------------------------

print("\nLoaded dataset:")
print(f"Shape: {df_5min.shape}")
print(f"Start: {df_5min.index.min()}")
print(f"End:   {df_5min.index.max()}")

display(df_5min.head())

Project root:
/Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project

5-minute HDF5 candidates:
 - solete_dataset/SOLETE_Pombo_5min.h5

Loaded dataset:
Shape: (131617, 11)
Start: 2018-06-01 00:00:00+00:00
End:   2019-09-01 00:00:00+00:00


,TEMPERATURE[degC],HUMIDITY[%],WIND_SPEED[m1s],WIND_DIR[deg],GHI[kW1m2],POA Irr[kW1m2],P_Gaia[kW],P_Solar[kW],Pressure[mbar],Azimuth[deg],Elevation[deg]
Timestamp,,,,,,,,,,,
2018-06-01 00:00:00+00:00,14.561204,0.7,1.293645,111.635452,0.0,0.0,0.046917,0.0,1017.559985,0.0,0.0
2018-06-01 00:05:00+00:00,14.487667,0.7,1.442333,91.236667,0.0,0.0,0.000267,0.0,1017.539990,0.0,0.0
2018-06-01 00:10:00+00:00,14.400000,0.7,1.408667,104.423333,0.0,0.0,0.000000,0.0,1017.493668,0.0,0.0
2018-06-01 00:15:00+00:00,14.240000,0.7,1.379333,103.976667,0.0,0.0,0.039050,0.0,1017.466342,0.0,0.0
2018-06-01 00:20:00+00:00,14.200000,0.7,1.453333,96.606667,0.0,0.0,0.003157,0.0,1017.533658,0.0,0.0


In [2]:
# ============================================================
# Step 2: Apply the validated observational preprocessing
# ============================================================

from src.preprocessing import build_cleaned_solete_features


# ------------------------------------------------------------
# 1. Build the cleaned observational dataset
# ------------------------------------------------------------
# This function:
#
#   - preserves all original SOLETE columns;
#   - corrects the known pressure artifact;
#   - cleans/wraps wind direction to [0, 360);
#   - creates sine/cosine wind-direction components;
#   - applies the validated causal humidity cleaning.
#
# IMPORTANT:
# We intentionally use build_cleaned_solete_features() here,
# rather than build_preprocessed_solete_features().
#
# Gregorian features will later be generated separately at the
# forecast-valid timestamp (t + h), according to the frozen
# experimental protocol.
# ------------------------------------------------------------

df_clean = build_cleaned_solete_features(df_5min)


# ------------------------------------------------------------
# 2. Verify the resulting dataset
# ------------------------------------------------------------

print("Cleaned dataset:")
print(f"Shape: {df_clean.shape}")
print(f"Start: {df_clean.index.min()}")
print(f"End:   {df_clean.index.max()}")

print("\nMissing values:")
print(df_clean.isna().sum().sum())

print("\nAdded cleaned columns:")
added_columns = [
    column
    for column in df_clean.columns
    if column not in df_5min.columns
]

for column in added_columns:
    print(" -", column)

display(df_clean.head())

Cleaned dataset:
Shape: (131617, 16)
Start: 2018-06-01 00:00:00+00:00
End:   2019-09-01 00:00:00+00:00

Missing values:
0

Added cleaned columns:
 - Pressure_clean[mbar]
 - Humidity_clean
 - Wind_dir_clean[deg]
 - wind_direction_sin
 - wind_direction_cos


,TEMPERATURE[degC],HUMIDITY[%],WIND_SPEED[m1s],WIND_DIR[deg],GHI[kW1m2],POA Irr[kW1m2],P_Gaia[kW],P_Solar[kW],Pressure[mbar],Azimuth[deg],Elevation[deg],Pressure_clean[mbar],Humidity_clean,Wind_dir_clean[deg],wind_direction_sin,wind_direction_cos
Timestamp,,,,,,,,,,,,,,,,
2018-06-01 00:00:00+00:00,14.561204,0.7,1.293645,111.635452,0.0,0.0,0.046917,0.0,1017.559985,0.0,0.0,1017.559985,0.7,111.635452,0.929549,-0.368700
2018-06-01 00:05:00+00:00,14.487667,0.7,1.442333,91.236667,0.0,0.0,0.000267,0.0,1017.539990,0.0,0.0,1017.539990,0.7,91.236667,0.999767,-0.021582
2018-06-01 00:10:00+00:00,14.400000,0.7,1.408667,104.423333,0.0,0.0,0.000000,0.0,1017.493668,0.0,0.0,1017.493668,0.7,104.423333,0.968482,-0.249084
2018-06-01 00:15:00+00:00,14.240000,0.7,1.379333,103.976667,0.0,0.0,0.039050,0.0,1017.466342,0.0,0.0,1017.466342,0.7,103.976667,0.970394,-0.241527
2018-06-01 00:20:00+00:00,14.200000,0.7,1.453333,96.606667,0.0,0.0,0.003157,0.0,1017.533658,0.0,0.0,1017.533658,0.7,96.606667,0.993359,-0.115053


In [3]:
# ============================================================
# Step 3: Construct the F0 historical / autoregressive block
# ============================================================

# ------------------------------------------------------------
# 1. Verify that row-wise shifting is safe
# ------------------------------------------------------------
# The lag construction below uses pandas .shift(), so the
# dataframe must represent a complete chronological 5-minute
# sequence.
# ------------------------------------------------------------

assert df_clean.index.is_monotonic_increasing, (
    "The 5-minute index is not chronologically ordered."
)

assert not df_clean.index.has_duplicates, (
    "Duplicate timestamps exist in the 5-minute dataset."
)

time_differences = df_clean.index.to_series().diff().dropna()

assert (time_differences == pd.Timedelta(minutes=5)).all(), (
    "The dataset is not a complete 5-minute grid."
)

print("5-minute timeline check: PASSED")


# ------------------------------------------------------------
# 2. Define the nine observed variables frozen for F0
# ------------------------------------------------------------
# Keys:
#   modelling-friendly base names
#
# Values:
#   corresponding columns in the cleaned SOLETE dataframe
# ------------------------------------------------------------

F0_HISTORY_SOURCES = {
    "P_Solar_kw": "P_Solar[kW]",
    "GHI_kw1m2": "GHI[kW1m2]",
    "POA_Irr_kw1m2": "POA Irr[kW1m2]",
    "temperature_degC": "TEMPERATURE[degC]",
    "humidity_clean": "Humidity_clean",
    "pressure_clean_mbar": "Pressure_clean[mbar]",
    "wind_speed_m1s": "WIND_SPEED[m1s]",
    "wind_direction_sin": "wind_direction_sin",
    "wind_direction_cos": "wind_direction_cos",
}


# ------------------------------------------------------------
# 3. Define the recent-history lag structure
# ------------------------------------------------------------
# At 5-minute resolution:
#
#   lag_0  = t
#   lag_1  = t - 5 min
#   ...
#   lag_12 = t - 60 min
# ------------------------------------------------------------

RECENT_LAGS = list(range(13))


# ------------------------------------------------------------
# 4. Build the recent-history feature dictionary
# ------------------------------------------------------------

f0_columns = {}

for feature_name, source_column in F0_HISTORY_SOURCES.items():
    for lag in RECENT_LAGS:
        column_name = f"{feature_name}_lag_{lag}"

        f0_columns[column_name] = df_clean[source_column].shift(lag)


# ------------------------------------------------------------
# 5. Add the frozen 24-hour seasonal PV lag
# ------------------------------------------------------------
# 24 hours × 60 minutes / 5-minute resolution = 288 steps.
# ------------------------------------------------------------

DAILY_LAG_STEPS = 288

f0_columns["P_Solar_kw_lag_288"] = (
    df_clean["P_Solar[kW]"].shift(DAILY_LAG_STEPS)
)


# ------------------------------------------------------------
# 6. Assemble the F0 historical feature dataframe
# ------------------------------------------------------------

df_f0_history = pd.DataFrame(
    f0_columns,
    index=df_clean.index,
)


# ------------------------------------------------------------
# 7. Verify the dimensionality against the frozen protocol
# ------------------------------------------------------------
# Expected:
#
#   9 variables × 13 recent lags = 117
#   + 1 daily PV lag            =   1
#                                  ---
#                                  118 features
# ------------------------------------------------------------

EXPECTED_F0_FEATURES = 118

assert df_f0_history.shape[1] == EXPECTED_F0_FEATURES, (
    f"Expected {EXPECTED_F0_FEATURES} F0 features, "
    f"but found {df_f0_history.shape[1]}."
)


# ------------------------------------------------------------
# 8. Inspect missingness caused by lag construction
# ------------------------------------------------------------
# Missing values at the beginning are expected because the
# 24-hour lag is unavailable for the first 288 observations.
#
# We do NOT fill these values artificially.
# ------------------------------------------------------------

complete_f0_rows = df_f0_history.notna().all(axis=1)

print("\nF0 historical feature block:")
print(f"Shape before dropping initial lag rows: {df_f0_history.shape}")
print(f"Number of F0 features: {df_f0_history.shape[1]}")
print(f"Complete rows: {complete_f0_rows.sum()}")
print(f"Rows requiring initial-history removal: {(~complete_f0_rows).sum()}")

print("\nFirst complete F0 timestamp:")
print(df_f0_history.index[complete_f0_rows][0])

print("\nLast F0 timestamp:")
print(df_f0_history.index[-1])

display(df_f0_history.loc[complete_f0_rows].head())

5-minute timeline check: PASSED

F0 historical feature block:
Shape before dropping initial lag rows: (131617, 118)
Number of F0 features: 118
Complete rows: 131329
Rows requiring initial-history removal: 288

First complete F0 timestamp:
2018-06-02 00:00:00+00:00

Last F0 timestamp:
2019-09-01 00:00:00+00:00


,P_Solar_kw_lag_0,P_Solar_kw_lag_1,P_Solar_kw_lag_2,P_Solar_kw_lag_3,P_Solar_kw_lag_4,P_Solar_kw_lag_5,P_Solar_kw_lag_6,P_Solar_kw_lag_7,P_Solar_kw_lag_8,P_Solar_kw_lag_9,...,wind_direction_cos_lag_4,wind_direction_cos_lag_5,wind_direction_cos_lag_6,wind_direction_cos_lag_7,wind_direction_cos_lag_8,wind_direction_cos_lag_9,wind_direction_cos_lag_10,wind_direction_cos_lag_11,wind_direction_cos_lag_12,P_Solar_kw_lag_288
Timestamp,,,,,,,,,,,,,,,,,,,,,
2018-06-02 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.300595,0.327768,0.428935,0.346335,0.996195,0.993082,0.922021,0.681998,0.681998,0.0
2018-06-02 00:05:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.292372,0.300595,0.327768,0.428935,0.346335,0.996195,0.993082,0.922021,0.681998,0.0
2018-06-02 00:10:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.292372,0.292372,0.300595,0.327768,0.428935,0.346335,0.996195,0.993082,0.922021,0.0
2018-06-02 00:15:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.292372,0.292372,0.292372,0.300595,0.327768,0.428935,0.346335,0.996195,0.993082,0.0
2018-06-02 00:20:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.963263,0.292372,0.292372,0.292372,0.300595,0.327768,0.428935,0.346335,0.996195,0.0


In [4]:
# ============================================================
# Step 4: Construct horizon-specific forecast targets
# ============================================================

# ------------------------------------------------------------
# 1. Define the frozen forecast horizons
# ------------------------------------------------------------
# At 5-minute resolution:
#
#   15 minutes =  3 steps
#   30 minutes =  6 steps
#   60 minutes = 12 steps
# ------------------------------------------------------------

FORECAST_HORIZONS = {
    15: 3,
    30: 6,
    60: 12,
}


# ------------------------------------------------------------
# 2. Build one target table for each forecast horizon
# ------------------------------------------------------------
# Each row is indexed by forecast issue time t.
#
# For horizon h:
#
#   issue_time = t
#   valid_time = t + h
#   target     = P_Solar(t + h)
#
# The target is constructed with a negative shift because we
# want a future value to appear on the current issue-time row.
# ------------------------------------------------------------

target_tables = {}

for horizon_minutes, horizon_steps in FORECAST_HORIZONS.items():

    df_target = pd.DataFrame(
        index=df_clean.index
    )

    # Keep issue time explicitly for later auditing.
    df_target["issue_time"] = df_target.index

    # Deterministic features will later be evaluated at this
    # forecast-valid timestamp.
    df_target["valid_time"] = (
        df_target["issue_time"]
        + pd.Timedelta(minutes=horizon_minutes)
    )

    # Future PV target:
    #
    # row at t receives P_Solar from t + h.
    df_target["target_P_Solar_kW"] = (
        df_clean["P_Solar[kW]"].shift(-horizon_steps)
    )

    # Store horizon metadata explicitly.
    df_target["horizon_minutes"] = horizon_minutes

    target_tables[horizon_minutes] = df_target


# ------------------------------------------------------------
# 3. Verify each target alignment
# ------------------------------------------------------------

for horizon_minutes, horizon_steps in FORECAST_HORIZONS.items():

    df_target = target_tables[horizon_minutes]

    target_available = df_target["target_P_Solar_kW"].notna()

    expected_available_rows = len(df_clean) - horizon_steps

    assert target_available.sum() == expected_available_rows, (
        f"{horizon_minutes}-minute target row count mismatch."
    )

    # The final available target should point to the final
    # timestamp in the original SOLETE dataset.
    last_issue_time = df_target.index[target_available][-1]
    last_valid_time = df_target.loc[last_issue_time, "valid_time"]

    assert last_valid_time == df_clean.index[-1], (
        f"{horizon_minutes}-minute final target alignment failed."
    )

    # Direct value-level alignment check.
    expected_target = df_clean.loc[
        last_valid_time,
        "P_Solar[kW]"
    ]

    actual_target = df_target.loc[
        last_issue_time,
        "target_P_Solar_kW"
    ]

    assert actual_target == expected_target, (
        f"{horizon_minutes}-minute target value alignment failed."
    )

    print(f"\n{horizon_minutes}-minute horizon:")
    print(f"  Steps ahead:              {horizon_steps}")
    print(f"  Available target rows:    {target_available.sum()}")
    print(f"  First issue time:         {df_target.index[0]}")
    print(f"  First valid time:         {df_target['valid_time'].iloc[0]}")
    print(f"  Last valid issue time:    {last_issue_time}")
    print(f"  Last valid target time:   {last_valid_time}")


15-minute horizon:
  Steps ahead:              3
  Available target rows:    131614
  First issue time:         2018-06-01 00:00:00+00:00
  First valid time:         2018-06-01 00:15:00+00:00
  Last valid issue time:    2019-08-31 23:45:00+00:00
  Last valid target time:   2019-09-01 00:00:00+00:00

30-minute horizon:
  Steps ahead:              6
  Available target rows:    131611
  First issue time:         2018-06-01 00:00:00+00:00
  First valid time:         2018-06-01 00:30:00+00:00
  Last valid issue time:    2019-08-31 23:30:00+00:00
  Last valid target time:   2019-09-01 00:00:00+00:00

60-minute horizon:
  Steps ahead:              12
  Available target rows:    131605
  First issue time:         2018-06-01 00:00:00+00:00
  First valid time:         2018-06-01 01:00:00+00:00
  Last valid issue time:    2019-08-31 23:00:00+00:00
  Last valid target time:   2019-09-01 00:00:00+00:00


In [5]:
# ============================================================
# Step 5: Combine F0 history, targets, and chronological splits
# ============================================================

# ------------------------------------------------------------
# 1. Define the frozen chronological split boundaries
# ------------------------------------------------------------

TRAIN_START = pd.Timestamp("2018-06-01 00:00:00", tz="UTC")
VALID_START = pd.Timestamp("2019-04-01 00:00:00", tz="UTC")
TEST_START = pd.Timestamp("2019-06-01 00:00:00", tz="UTC")
TEST_END = pd.Timestamp("2019-09-01 00:00:00", tz="UTC")


# ------------------------------------------------------------
# 2. Build one base modelling table per forecast horizon
# ------------------------------------------------------------
# Each table contains:
#
#   - issue-time index
#   - F0 historical features
#   - issue_time metadata
#   - valid_time metadata
#   - target
#   - horizon metadata
#   - chronological split label
#
# Deterministic F1-F4 features are NOT added yet.
# ------------------------------------------------------------

base_modelling_tables = {}

for horizon_minutes in FORECAST_HORIZONS:

    # Start with F0 historical predictors.
    df_model = df_f0_history.copy()

    # Add the previously validated target metadata.
    df_model = df_model.join(
        target_tables[horizon_minutes][
            [
                "issue_time",
                "valid_time",
                "target_P_Solar_kW",
                "horizon_minutes",
            ]
        ]
    )

    # --------------------------------------------------------
    # Keep only rows for which:
    #
    #   1. all 118 F0 historical features are available;
    #   2. the future target exists;
    #   3. valid_time lies inside the frozen experiment period.
    #
    # TEST_END is exclusive, so the single dataset-boundary
    # timestamp 2019-09-01 00:00 UTC is not part of the test set.
    # --------------------------------------------------------

    usable_mask = (
        df_model[df_f0_history.columns].notna().all(axis=1)
        & df_model["target_P_Solar_kW"].notna()
        & (df_model["valid_time"] >= TRAIN_START)
        & (df_model["valid_time"] < TEST_END)
    )

    df_model = df_model.loc[usable_mask].copy()

    # --------------------------------------------------------
    # Assign split according to FORECAST-VALID time, not issue
    # time.
    # --------------------------------------------------------

    df_model["split"] = pd.NA

    df_model.loc[
        (df_model["valid_time"] >= TRAIN_START)
        & (df_model["valid_time"] < VALID_START),
        "split",
    ] = "train"

    df_model.loc[
        (df_model["valid_time"] >= VALID_START)
        & (df_model["valid_time"] < TEST_START),
        "split",
    ] = "validation"

    df_model.loc[
        (df_model["valid_time"] >= TEST_START)
        & (df_model["valid_time"] < TEST_END),
        "split",
    ] = "test"

    # Every retained row must belong to exactly one split.
    assert df_model["split"].notna().all(), (
        f"Unassigned split rows found for "
        f"{horizon_minutes}-minute horizon."
    )

    base_modelling_tables[horizon_minutes] = df_model


# ------------------------------------------------------------
# 3. Report row counts and timing for each horizon
# ------------------------------------------------------------

for horizon_minutes, df_model in base_modelling_tables.items():

    print(f"\n{horizon_minutes}-minute horizon:")
    print(f"  Total usable rows: {len(df_model)}")

    split_counts = (
        df_model["split"]
        .value_counts()
        .reindex(["train", "validation", "test"])
    )

    print("  Split counts:")
    print(split_counts.to_string())

    print(
        f"  First issue time: "
        f"{df_model['issue_time'].iloc[0]}"
    )

    print(
        f"  First valid time: "
        f"{df_model['valid_time'].iloc[0]}"
    )

    print(
        f"  Last issue time:  "
        f"{df_model['issue_time'].iloc[-1]}"
    )

    print(
        f"  Last valid time:  "
        f"{df_model['valid_time'].iloc[-1]}"
    )


# ------------------------------------------------------------
# 4. Sanity check the split boundary itself
# ------------------------------------------------------------

df_60 = base_modelling_tables[60]

print("\n60-minute boundary examples:")

print("\nLast training row:")
display(
    df_60.loc[df_60["split"] == "train",
              ["issue_time", "valid_time", "target_P_Solar_kW"]]
    .tail(1)
)

print("\nFirst validation row:")
display(
    df_60.loc[df_60["split"] == "validation",
              ["issue_time", "valid_time", "target_P_Solar_kW"]]
    .head(1)
)

print("\nFirst test row:")
display(
    df_60.loc[df_60["split"] == "test",
              ["issue_time", "valid_time", "target_P_Solar_kW"]]
    .head(1)
)


15-minute horizon:
  Total usable rows: 131325
  Split counts:
split
train         87261
validation    17568
test          26496
  First issue time: 2018-06-02 00:00:00+00:00
  First valid time: 2018-06-02 00:15:00+00:00
  Last issue time:  2019-08-31 23:40:00+00:00
  Last valid time:  2019-08-31 23:55:00+00:00

30-minute horizon:
  Total usable rows: 131322
  Split counts:
split
train         87258
validation    17568
test          26496
  First issue time: 2018-06-02 00:00:00+00:00
  First valid time: 2018-06-02 00:30:00+00:00
  Last issue time:  2019-08-31 23:25:00+00:00
  Last valid time:  2019-08-31 23:55:00+00:00

60-minute horizon:
  Total usable rows: 131316
  Split counts:
split
train         87252
validation    17568
test          26496
  First issue time: 2018-06-02 00:00:00+00:00
  First valid time: 2018-06-02 01:00:00+00:00
  Last issue time:  2019-08-31 22:55:00+00:00
  Last valid time:  2019-08-31 23:55:00+00:00

60-minute boundary examples:

Last training row:


,issue_time,valid_time,target_P_Solar_kW
Timestamp,,,
2019-03-31 22:55:00+00:00,2019-03-31 22:55:00+00:00,2019-03-31 23:55:00+00:00,0.0



First validation row:


,issue_time,valid_time,target_P_Solar_kW
Timestamp,,,
2019-03-31 23:00:00+00:00,2019-03-31 23:00:00+00:00,2019-04-01 00:00:00+00:00,0.0



First test row:


,issue_time,valid_time,target_P_Solar_kW
Timestamp,,,
2019-05-31 23:00:00+00:00,2019-05-31 23:00:00+00:00,2019-06-01 00:00:00+00:00,0.001667


In [6]:
# ============================================================
# Step 6: Build F1 Gregorian features at forecast-valid time
# ============================================================

from src.preprocessing import build_gregorian_features


# ------------------------------------------------------------
# 1. Build one forecast-valid Gregorian feature table for
#    each forecast horizon
# ------------------------------------------------------------
#
# IMPORTANT:
#
# The reusable Gregorian function accepts a DatetimeIndex and
# calculates the 15 validated deterministic UTC features.
#
# For forecasting, these features must describe:
#
#     valid_time = t + h
#
# rather than the issue time t.
#
# This is allowed because calendar information is deterministic
# and known in advance.
# ------------------------------------------------------------

gregorian_feature_tables = {}

for horizon_minutes, df_model in base_modelling_tables.items():

    # Extract the future forecast-valid timestamps.
    valid_times = pd.DatetimeIndex(
        df_model["valid_time"],
        name="valid_time",
    )

    # Generate the validated 15 Gregorian features at t + h.
    df_gregorian = build_gregorian_features(valid_times)

    # --------------------------------------------------------
    # Verify that the feature builder preserved the exact
    # forecast-valid timestamp sequence.
    # --------------------------------------------------------

    assert df_gregorian.index.equals(valid_times), (
        f"Gregorian timestamp alignment failed for "
        f"{horizon_minutes}-minute horizon."
    )

    assert df_gregorian.shape[1] == 15, (
        f"Expected 15 Gregorian features, but found "
        f"{df_gregorian.shape[1]}."
    )

    assert df_gregorian.isna().sum().sum() == 0, (
        f"Missing Gregorian values found for "
        f"{horizon_minutes}-minute horizon."
    )

    # --------------------------------------------------------
    # Prefix the columns with 'valid_' to make their timing
    # explicit inside the modelling dataset.
    # --------------------------------------------------------

    df_gregorian = df_gregorian.add_prefix("valid_")

    # The feature values describe valid_time, but modelling
    # rows are indexed by issue time.
    #
    # After validating the timestamp alignment above, assign
    # the issue-time index so the features can later be joined
    # safely to the corresponding modelling rows.
    df_gregorian.index = df_model.index

    gregorian_feature_tables[horizon_minutes] = df_gregorian


# ------------------------------------------------------------
# 2. Inspect all three horizons
# ------------------------------------------------------------

for horizon_minutes, df_gregorian in gregorian_feature_tables.items():

    print(f"\n{horizon_minutes}-minute horizon:")
    print(f"  Shape:          {df_gregorian.shape}")
    print(
        f"  Missing values: "
        f"{df_gregorian.isna().sum().sum()}"
    )

    print(
        f"  First issue:    "
        f"{df_gregorian.index[0]}"
    )

    print(
        f"  First valid:    "
        f"{base_modelling_tables[horizon_minutes]['valid_time'].iloc[0]}"
    )

    print(
        f"  Valid hour:     "
        f"{df_gregorian['valid_utc_hour'].iloc[0]}"
    )

    print(
        f"  Valid minute:   "
        f"{df_gregorian['valid_utc_minute'].iloc[0]}"
    )


# ------------------------------------------------------------
# 3. Explicit 60-minute alignment sanity check
# ------------------------------------------------------------
#
# The first usable 60-minute modelling row has:
#
#   issue_time = 2018-06-02 00:00 UTC
#   valid_time = 2018-06-02 01:00 UTC
#
# Therefore its Gregorian features must describe 01:00,
# not 00:00.
# ------------------------------------------------------------

df_greg_60 = gregorian_feature_tables[60]
df_model_60 = base_modelling_tables[60]

assert (
    df_model_60["valid_time"].iloc[0]
    == pd.Timestamp("2018-06-02 01:00:00", tz="UTC")
)

assert df_greg_60["valid_utc_hour"].iloc[0] == 1
assert df_greg_60["valid_utc_minute"].iloc[0] == 0

print("\n60-minute forecast-valid alignment check: PASSED")

display(df_greg_60.head())


15-minute horizon:
  Shape:          (131325, 15)
  Missing values: 0
  First issue:    2018-06-02 00:00:00+00:00
  First valid:    2018-06-02 00:15:00+00:00
  Valid hour:     0
  Valid minute:   15

30-minute horizon:
  Shape:          (131322, 15)
  Missing values: 0
  First issue:    2018-06-02 00:00:00+00:00
  First valid:    2018-06-02 00:30:00+00:00
  Valid hour:     0
  Valid minute:   30

60-minute horizon:
  Shape:          (131316, 15)
  Missing values: 0
  First issue:    2018-06-02 00:00:00+00:00
  First valid:    2018-06-02 01:00:00+00:00
  Valid hour:     1
  Valid minute:   0

60-minute forecast-valid alignment check: PASSED


,valid_utc_year,valid_utc_month,valid_utc_day,valid_utc_day_of_week,valid_utc_day_of_year,valid_utc_hour,valid_utc_minute,valid_utc_minute_of_day,valid_utc_is_weekend,valid_utc_time_sin,valid_utc_time_cos,valid_utc_weekday_sin,valid_utc_weekday_cos,valid_utc_year_sin,valid_utc_year_cos
Timestamp,,,,,,,,,,,,,,,
2018-06-02 00:00:00+00:00,2018,6,2,5,153,1,0,60,1,0.258819,0.965926,-0.974928,-0.222521,0.500621,-0.865667
2018-06-02 00:05:00+00:00,2018,6,2,5,153,1,5,65,1,0.279829,0.960050,-0.974928,-0.222521,0.500569,-0.865696
2018-06-02 00:10:00+00:00,2018,6,2,5,153,1,10,70,1,0.300706,0.953717,-0.974928,-0.222521,0.500518,-0.865726
2018-06-02 00:15:00+00:00,2018,6,2,5,153,1,15,75,1,0.321439,0.946930,-0.974928,-0.222521,0.500466,-0.865756
2018-06-02 00:20:00+00:00,2018,6,2,5,153,1,20,80,1,0.342020,0.939693,-0.974928,-0.222521,0.500414,-0.865786


In [7]:
# ============================================================
# Step 7: Build F2 solar astronomy at forecast-valid time
# ============================================================

from src.astronomy_features import build_solar_f2_features


# ------------------------------------------------------------
# 1. Build one solar-feature table for each forecast horizon
# ------------------------------------------------------------
#
# Important timing convention:
#
#   modelling valid time = t + h
#
# The reusable astronomy function internally evaluates each
# SOLETE timestamp at:
#
#   SOLETE timestamp + 2 minutes
#
# This represents the approximate midpoint of the underlying
# 5-minute aggregation interval established during Notebook 07.
#
# Therefore:
#
#   DO NOT manually add another +2 minutes here.
# ------------------------------------------------------------

solar_feature_tables = {}

for horizon_minutes, df_model in base_modelling_tables.items():

    # Forecast-valid SOLETE timestamps.
    valid_times = pd.DatetimeIndex(
        df_model["valid_time"],
        name="valid_time",
    )

    # Build the validated five F2 solar features.
    #
    # Internally:
    #
    #   valid_time
    #       -> valid_time + 2 minutes
    #       -> pvlib solar-position calculation
    #
    # The returned dataframe retains valid_time as its index.
    df_solar = build_solar_f2_features(valid_times)

    # --------------------------------------------------------
    # Validate the output before changing its index.
    # --------------------------------------------------------

    assert df_solar.index.equals(valid_times), (
        f"Solar timestamp alignment failed for "
        f"{horizon_minutes}-minute horizon."
    )

    assert df_solar.shape[1] == 5, (
        f"Expected 5 F2 solar features, but found "
        f"{df_solar.shape[1]}."
    )

    assert df_solar.isna().sum().sum() == 0, (
        f"Missing solar values found for "
        f"{horizon_minutes}-minute horizon."
    )

    # Expected canonical F2 feature names.
    expected_solar_columns = [
        "solar_elevation_deg",
        "solar_cos_zenith",
        "solar_azimuth_sin",
        "solar_azimuth_cos",
        "solar_daylight",
    ]

    assert list(df_solar.columns) == expected_solar_columns, (
        "Unexpected F2 solar feature columns."
    )

    # --------------------------------------------------------
    # Prefix with valid_ so the feature timing remains explicit
    # inside the modelling table.
    # --------------------------------------------------------

    df_solar = df_solar.add_prefix("valid_")

    # Values describe valid_time, but rows in the modelling
    # dataset are indexed by issue time.
    df_solar.index = df_model.index

    solar_feature_tables[horizon_minutes] = df_solar


# ------------------------------------------------------------
# 2. Report dimensionality and alignment for each horizon
# ------------------------------------------------------------

for horizon_minutes, df_solar in solar_feature_tables.items():

    df_model = base_modelling_tables[horizon_minutes]

    print(f"\n{horizon_minutes}-minute horizon:")
    print(f"  Shape:          {df_solar.shape}")
    print(
        f"  Missing values: "
        f"{df_solar.isna().sum().sum()}"
    )

    print(
        f"  First issue:    "
        f"{df_model['issue_time'].iloc[0]}"
    )

    print(
        f"  First valid:    "
        f"{df_model['valid_time'].iloc[0]}"
    )

    print(
        f"  Astronomy eval: "
        f"{df_model['valid_time'].iloc[0] + pd.Timedelta(minutes=2)}"
    )


# ------------------------------------------------------------
# 3. Explicit 60-minute timing check
# ------------------------------------------------------------
#
# First 60-minute modelling row:
#
#   issue_time       = 2018-06-02 00:00 UTC
#   valid_time       = 2018-06-02 01:00 UTC
#   astronomy eval   = 2018-06-02 01:02 UTC
# ------------------------------------------------------------

df_solar_60 = solar_feature_tables[60]
df_model_60 = base_modelling_tables[60]

first_valid_60 = df_model_60["valid_time"].iloc[0]
first_astro_60 = first_valid_60 + pd.Timedelta(minutes=2)

assert first_valid_60 == pd.Timestamp(
    "2018-06-02 01:00:00",
    tz="UTC",
)

assert first_astro_60 == pd.Timestamp(
    "2018-06-02 01:02:00",
    tz="UTC",
)

print("\n60-minute astronomy timing check: PASSED")

print("\nFirst 60-minute forecast solar features:")
print(f"Issue time:      {df_model_60['issue_time'].iloc[0]}")
print(f"Valid time:      {first_valid_60}")
print(f"Astronomy time:  {first_astro_60}")

display(df_solar_60.head())


15-minute horizon:
  Shape:          (131325, 5)
  Missing values: 0
  First issue:    2018-06-02 00:00:00+00:00
  First valid:    2018-06-02 00:15:00+00:00
  Astronomy eval: 2018-06-02 00:17:00+00:00

30-minute horizon:
  Shape:          (131322, 5)
  Missing values: 0
  First issue:    2018-06-02 00:00:00+00:00
  First valid:    2018-06-02 00:30:00+00:00
  Astronomy eval: 2018-06-02 00:32:00+00:00

60-minute horizon:
  Shape:          (131316, 5)
  Missing values: 0
  First issue:    2018-06-02 00:00:00+00:00
  First valid:    2018-06-02 01:00:00+00:00
  Astronomy eval: 2018-06-02 01:02:00+00:00

60-minute astronomy timing check: PASSED

First 60-minute forecast solar features:
Issue time:      2018-06-02 00:00:00+00:00
Valid time:      2018-06-02 01:00:00+00:00
Astronomy time:  2018-06-02 01:02:00+00:00


,valid_solar_elevation_deg,valid_solar_cos_zenith,valid_solar_azimuth_sin,valid_solar_azimuth_cos,valid_solar_daylight
Timestamp,,,,,
2018-06-02 00:00:00+00:00,-8.571247,-0.149039,0.441480,0.897271,0
2018-06-02 00:05:00+00:00,-8.253567,-0.143554,0.459015,0.888428,0
2018-06-02 00:10:00+00:00,-7.923620,-0.137853,0.476304,0.879281,0
2018-06-02 00:15:00+00:00,-7.581581,-0.131938,0.493341,0.869836,0
2018-06-02 00:20:00+00:00,-7.227632,-0.125812,0.510120,0.860103,0


In [8]:
# ============================================================
# Step 8: Inspect the reusable F3 lunar-feature API
# ============================================================

import inspect
import src.astronomy_features as astronomy_features


# ------------------------------------------------------------
# Find functions related to the lunar / F3 feature pipeline.
#
# We inspect the module rather than guessing the exact function
# signature, particularly because the lunar calculations also
# depend on the local JPL ephemeris file.
# ------------------------------------------------------------

candidate_names = [
    name
    for name in dir(astronomy_features)
    if any(
        keyword in name.lower()
        for keyword in ["lunar", "moon", "f3", "ephemeris"]
    )
    and callable(getattr(astronomy_features, name))
]


print("Relevant astronomy functions:\n")

for name in candidate_names:
    function = getattr(astronomy_features, name)

    try:
        signature = inspect.signature(function)
    except (TypeError, ValueError):
        signature = "(signature unavailable)"

    print(f"{name}{signature}")

Relevant astronomy functions:

build_lunar_f3_features(timestamps: 'pd.DatetimeIndex', ephemeris) -> 'pd.DataFrame'
load_jpl_ephemeris(ephemeris_path: 'str | Path')
moon_ecliptic_latitude_and_distance(ephemeris, skyfield_times) -> 'tuple[np.ndarray, np.ndarray]'
moon_illumination_fraction(ephemeris, skyfield_times) -> 'np.ndarray'
moon_sun_phase_angle_deg(sun_longitude_deg: 'float | np.ndarray | pd.Series', moon_longitude_deg: 'float | np.ndarray | pd.Series') -> 'float | np.ndarray | pd.Series'


In [9]:
# ============================================================
# Step 9: Build F3 lunar / Sun-Moon astronomical features
# ============================================================

from src.astronomy_features import (
    load_jpl_ephemeris,
    build_lunar_f3_features,
)


# ------------------------------------------------------------
# 1. Locate and load the local JPL ephemeris
# ------------------------------------------------------------
# Notebook 07 established DE440s as the astronomical ephemeris
# used by the reusable lunar feature pipeline.
#
# The file is stored locally and is intentionally not tracked
# by Git because of its size.
# ------------------------------------------------------------

EPHEMERIS_PATH = PROJECT_ROOT / "ephemeris" / "de440s.bsp"

assert EPHEMERIS_PATH.is_file(), (
    f"JPL ephemeris file not found:\n{EPHEMERIS_PATH}"
)

print("JPL ephemeris:")
print(EPHEMERIS_PATH.relative_to(PROJECT_ROOT))


# Load the ephemeris once and reuse the same object.
ephemeris = load_jpl_ephemeris(EPHEMERIS_PATH)

print("\nEphemeris loaded successfully.")


# ------------------------------------------------------------
# 2. Compute F3 lunar features ONCE for the complete canonical
#    5-minute SOLETE timestamp grid
# ------------------------------------------------------------
#
# This is more efficient than recalculating the same astronomy
# separately for the 15-, 30-, and 60-minute forecast tables.
#
# IMPORTANT:
#
# build_lunar_f3_features() internally follows the validated
# SOLETE astronomy-time convention:
#
#     SOLETE timestamp + 2 minutes
#
# Therefore we pass the original canonical timestamp grid and
# DO NOT manually add another +2 minutes.
# ------------------------------------------------------------

df_lunar_full = build_lunar_f3_features(
    df_clean.index,
    ephemeris,
)


# ------------------------------------------------------------
# 3. Validate the complete F3 lunar table
# ------------------------------------------------------------

expected_lunar_columns = [
    "moon_phase_sin",
    "moon_phase_cos",
    "moon_longitude_sin",
    "moon_longitude_cos",
    "sun_ecliptic_longitude_sin",
    "sun_ecliptic_longitude_cos",
    "moon_ecliptic_latitude_deg",
    "moon_illumination_fraction",
    "earth_moon_distance_km",
]

assert df_lunar_full.index.equals(df_clean.index), (
    "Full F3 lunar timestamp alignment failed."
)

assert list(df_lunar_full.columns) == expected_lunar_columns, (
    "Unexpected F3 lunar feature columns."
)

assert df_lunar_full.shape == (len(df_clean), 9), (
    f"Unexpected F3 lunar shape: {df_lunar_full.shape}"
)

assert df_lunar_full.isna().sum().sum() == 0, (
    "Missing values found in the full F3 lunar feature table."
)

print("\nFull F3 lunar table:")
print(f"Shape:          {df_lunar_full.shape}")
print(f"Missing values: {df_lunar_full.isna().sum().sum()}")
print(f"Start:          {df_lunar_full.index.min()}")
print(f"End:            {df_lunar_full.index.max()}")


# ------------------------------------------------------------
# 4. Select F3 features at each horizon's forecast-valid times
# ------------------------------------------------------------
#
# For every modelling row:
#
#     issue time = t
#     valid time = t + h
#
# We select the already-computed astronomy corresponding to
# valid_time.
#
# The selected values therefore describe the future target
# interval while remaining deterministic and future-known.
# ------------------------------------------------------------

lunar_feature_tables = {}

for horizon_minutes, df_model in base_modelling_tables.items():

    valid_times = pd.DatetimeIndex(
        df_model["valid_time"],
        name="valid_time",
    )

    # Select exact forecast-valid rows from the canonical
    # full-timeline lunar feature table.
    df_lunar = df_lunar_full.loc[valid_times].copy()

    # Verify that selection preserved the requested order.
    assert df_lunar.index.equals(valid_times), (
        f"F3 lunar valid-time alignment failed for "
        f"{horizon_minutes}-minute horizon."
    )

    assert df_lunar.shape[1] == 9

    assert df_lunar.isna().sum().sum() == 0

    # Prefix columns so their timing is explicit.
    df_lunar = df_lunar.add_prefix("valid_")

    # Values describe valid_time, but modelling rows themselves
    # are indexed by forecast issue time.
    df_lunar.index = df_model.index

    lunar_feature_tables[horizon_minutes] = df_lunar


# ------------------------------------------------------------
# 5. Report all horizon-specific F3 tables
# ------------------------------------------------------------

for horizon_minutes, df_lunar in lunar_feature_tables.items():

    df_model = base_modelling_tables[horizon_minutes]

    print(f"\n{horizon_minutes}-minute horizon:")
    print(f"  Shape:          {df_lunar.shape}")
    print(
        f"  Missing values: "
        f"{df_lunar.isna().sum().sum()}"
    )

    print(
        f"  First issue:    "
        f"{df_model['issue_time'].iloc[0]}"
    )

    print(
        f"  First valid:    "
        f"{df_model['valid_time'].iloc[0]}"
    )

    print(
        f"  Astronomy eval: "
        f"{df_model['valid_time'].iloc[0] + pd.Timedelta(minutes=2)}"
    )


# ------------------------------------------------------------
# 6. Explicit 60-minute alignment check
# ------------------------------------------------------------

df_lunar_60 = lunar_feature_tables[60]
df_model_60 = base_modelling_tables[60]

first_valid_60 = df_model_60["valid_time"].iloc[0]

assert first_valid_60 == pd.Timestamp(
    "2018-06-02 01:00:00",
    tz="UTC",
)

# Confirm that the first modelling F3 row is numerically equal
# to the canonical full-timeline astronomy at that valid time.
canonical_first_60 = (
    df_lunar_full
    .loc[first_valid_60]
    .to_numpy()
)

model_first_60 = (
    df_lunar_60
    .iloc[0]
    .to_numpy()
)

assert (
    abs(canonical_first_60 - model_first_60) < 1e-12
).all(), (
    "60-minute F3 value alignment failed."
)

print("\n60-minute F3 alignment check: PASSED")

print("\nFirst 60-minute forecast lunar features:")
print(f"Issue time:      {df_model_60['issue_time'].iloc[0]}")
print(f"Valid time:      {first_valid_60}")
print(
    f"Astronomy time:  "
    f"{first_valid_60 + pd.Timedelta(minutes=2)}"
)

display(df_lunar_60.head())

JPL ephemeris:
ephemeris/de440s.bsp

Ephemeris loaded successfully.

Full F3 lunar table:
Shape:          (131617, 9)
Missing values: 0
Start:          2018-06-01 00:00:00+00:00
End:            2019-09-01 00:00:00+00:00

15-minute horizon:
  Shape:          (131325, 9)
  Missing values: 0
  First issue:    2018-06-02 00:00:00+00:00
  First valid:    2018-06-02 00:15:00+00:00
  Astronomy eval: 2018-06-02 00:17:00+00:00

30-minute horizon:
  Shape:          (131322, 9)
  Missing values: 0
  First issue:    2018-06-02 00:00:00+00:00
  First valid:    2018-06-02 00:30:00+00:00
  Astronomy eval: 2018-06-02 00:32:00+00:00

60-minute horizon:
  Shape:          (131316, 9)
  Missing values: 0
  First issue:    2018-06-02 00:00:00+00:00
  First valid:    2018-06-02 01:00:00+00:00
  Astronomy eval: 2018-06-02 01:02:00+00:00

60-minute F3 alignment check: PASSED

First 60-minute forecast lunar features:
Issue time:      2018-06-02 00:00:00+00:00
Valid time:      2018-06-02 01:00:00+00:00
Astronom

,valid_moon_phase_sin,valid_moon_phase_cos,valid_moon_longitude_sin,valid_moon_longitude_cos,valid_sun_ecliptic_longitude_sin,valid_sun_ecliptic_longitude_cos,valid_moon_ecliptic_latitude_deg,valid_moon_illumination_fraction,valid_earth_moon_distance_km
Timestamp,,,,,,,,,
2018-06-02 00:00:00+00:00,-0.617980,-0.786194,-0.941826,0.336101,0.948162,0.317789,1.544368,0.893493,405091.945014
2018-06-02 00:05:00+00:00,-0.618498,-0.785787,-0.941585,0.336776,0.948180,0.317734,1.540866,0.893291,405094.028839
2018-06-02 00:10:00+00:00,-0.619015,-0.785379,-0.941343,0.337451,0.948198,0.317679,1.537363,0.893089,405096.101941
2018-06-02 00:15:00+00:00,-0.619532,-0.784972,-0.941101,0.338125,0.948217,0.317624,1.533859,0.892887,405098.164313
2018-06-02 00:20:00+00:00,-0.620049,-0.784563,-0.940859,0.338799,0.948235,0.317568,1.530355,0.892684,405100.215948


In [10]:
# ============================================================
# Step 10: Inspect the reusable F4 Panchang construction APIs
# ============================================================

import inspect

import src.astronomy_features as astronomy_features
import src.vedic_calendar as vedic_calendar


# ------------------------------------------------------------
# 1. Astronomy helper required by F4
# ------------------------------------------------------------
# F4 needs the raw tropical ecliptic longitudes of:
#
#   - Sun
#   - Moon
#
# evaluated using the same validated astronomy convention used
# throughout Notebook 07 and Notebook 08.
# ------------------------------------------------------------

astronomy_api_names = [
    "tropical_longitudes_for_solete",
]


print("Astronomy API:\n")

for name in astronomy_api_names:
    function = getattr(astronomy_features, name)
    print(f"{name}{inspect.signature(function)}")


# ------------------------------------------------------------
# 2. Vedic-calendar helpers required by F4
# ------------------------------------------------------------
# We need:
#
#   1. canonical Lahiri ayanamsha for each UTC timestamp;
#   2. the high-level builder that converts the validated
#      Sun/Moon geometry into the five raw Panchang features.
# ------------------------------------------------------------

vedic_api_names = [
    "lahiri_ayanamsha_for_utc_times",
    "build_panchang_f4_raw",
]


print("\nVedic-calendar API:\n")

for name in vedic_api_names:
    function = getattr(vedic_calendar, name)
    print(f"{name}{inspect.signature(function)}")

Astronomy API:

tropical_longitudes_for_solete(timestamps: 'pd.DatetimeIndex', ephemeris) -> 'pd.DataFrame'

Vedic-calendar API:

lahiri_ayanamsha_for_utc_times(timestamps: 'pd.DatetimeIndex') -> 'np.ndarray'
build_panchang_f4_raw(sun_tropical_longitude_deg: 'pd.Series', moon_tropical_longitude_deg: 'pd.Series', ayanamsha_deg: 'pd.Series') -> 'pd.DataFrame'


In [12]:
# ============================================================
# Step 11: Build canonical F4 Panchang features
# ============================================================

import numpy as np

from src.astronomy_features import (
    solete_astronomy_reference_times,
    tropical_longitudes_for_solete,
)

from src.vedic_calendar import (
    lahiri_ayanamsha_for_utc_times,
    build_panchang_f4_raw,
)


# ------------------------------------------------------------
# 1. Compute raw tropical Sun/Moon longitudes for the complete
#    canonical 5-minute SOLETE timeline
# ------------------------------------------------------------
#
# tropical_longitudes_for_solete() internally applies the
# validated SOLETE astronomy convention:
#
#     SOLETE timestamp + 2 minutes
#
# Therefore the returned Sun and Moon tropical longitudes
# correspond to the approximate midpoint of each 5-minute
# aggregation interval.
# ------------------------------------------------------------

df_tropical_full = tropical_longitudes_for_solete(
    df_clean.index,
    ephemeris,
)


# ------------------------------------------------------------
# 2. Verify the tropical-longitude output
# ------------------------------------------------------------

expected_tropical_columns = [
    "sun_tropical_longitude_deg",
    "moon_tropical_longitude_deg",
]

assert df_tropical_full.index.equals(df_clean.index), (
    "Tropical-longitude timestamp alignment failed."
)

assert list(df_tropical_full.columns) == expected_tropical_columns, (
    "Unexpected tropical-longitude columns."
)

assert df_tropical_full.shape == (len(df_clean), 2), (
    f"Unexpected tropical-longitude shape: "
    f"{df_tropical_full.shape}"
)

assert df_tropical_full.isna().sum().sum() == 0, (
    "Missing values found in tropical longitudes."
)

print("Full tropical-longitude table:")
print(f"Shape:          {df_tropical_full.shape}")
print(
    f"Missing values: "
    f"{df_tropical_full.isna().sum().sum()}"
)


# ------------------------------------------------------------
# 3. Construct the SAME +2-minute astronomy reference times
#    for the Lahiri ayanamsha calculation
# ------------------------------------------------------------
#
# Unlike tropical_longitudes_for_solete(), the reusable
# lahiri_ayanamsha_for_utc_times() function is generic and
# operates on the UTC timestamps supplied to it.
#
# Therefore we explicitly generate:
#
#     SOLETE timestamp + 2 minutes
#
# so that:
#
#   Sun tropical longitude
#   Moon tropical longitude
#   Lahiri ayanamsha
#
# all refer to exactly the same astronomical instant.
# ------------------------------------------------------------

astronomy_reference_times = solete_astronomy_reference_times(
    df_clean.index
)

assert len(astronomy_reference_times) == len(df_clean)

assert astronomy_reference_times[0] == (
    df_clean.index[0] + pd.Timedelta(minutes=2)
)

print("\nFirst SOLETE timestamp:")
print(df_clean.index[0])

print("First astronomy reference time:")
print(astronomy_reference_times[0])


# ------------------------------------------------------------
# 4. Compute canonical extended Lahiri ayanamsha
# ------------------------------------------------------------
#
# Notebook 08 established the canonical convention using the
# Swiss Ephemeris extended Lahiri calculation.
#
# The returned NumPy array contains one ayanamsha value for
# every +2-minute astronomy reference timestamp.
# ------------------------------------------------------------

ayanamsha_values = lahiri_ayanamsha_for_utc_times(
    astronomy_reference_times
)

assert len(ayanamsha_values) == len(df_clean)

assert np.isfinite(ayanamsha_values).all(), (
    "Non-finite Lahiri ayanamsha values found."
)

# Store the values as a Series indexed by the corresponding
# SOLETE timestamps. The VALUES refer to timestamp + 2 min,
# while the index identifies the SOLETE aggregation interval.
lahiri_ayanamsha_full = pd.Series(
    ayanamsha_values,
    index=df_clean.index,
    name="lahiri_ayanamsha_deg",
)


# ------------------------------------------------------------
# 5. Validate the known canonical Notebook 08 reference point
# ------------------------------------------------------------
#
# For SOLETE timestamp:
#
#     2018-06-01 00:00 UTC
#
# astronomy is evaluated at:
#
#     2018-06-01 00:02 UTC
#
# The validated canonical Lahiri ayanamsha is approximately:
#
#     24.110343268621794 degrees
# ------------------------------------------------------------

KNOWN_FIRST_AYANAMSHA = 24.110343268621794

assert np.isclose(
    lahiri_ayanamsha_full.iloc[0],
    KNOWN_FIRST_AYANAMSHA,
    atol=1e-10,
    rtol=0.0,
), (
    "Canonical first-point Lahiri ayanamsha check failed."
)

print("\nLahiri ayanamsha:")
print(
    f"First value: "
    f"{lahiri_ayanamsha_full.iloc[0]}"
)
print(
    f"Minimum:     "
    f"{lahiri_ayanamsha_full.min()}"
)
print(
    f"Maximum:     "
    f"{lahiri_ayanamsha_full.max()}"
)

print("\nCanonical Lahiri reference check: PASSED")


# ------------------------------------------------------------
# 6. Build the raw five-column F4 Panchang table
# ------------------------------------------------------------
#
# Inputs:
#
#   tropical Sun longitude
#   tropical Moon longitude
#   canonical extended Lahiri ayanamsha
#
# Output:
#
#   tithi
#   paksha
#   karana
#   nakshatra
#   yoga
# ------------------------------------------------------------

df_panchang_full = build_panchang_f4_raw(
    sun_tropical_longitude_deg=(
        df_tropical_full["sun_tropical_longitude_deg"]
    ),
    moon_tropical_longitude_deg=(
        df_tropical_full["moon_tropical_longitude_deg"]
    ),
    ayanamsha_deg=lahiri_ayanamsha_full,
)


# ------------------------------------------------------------
# 7. Validate the complete raw F4 table
# ------------------------------------------------------------

expected_panchang_columns = [
    "tithi",
    "paksha",
    "karana",
    "nakshatra",
    "yoga",
]

assert df_panchang_full.index.equals(df_clean.index), (
    "F4 Panchang timestamp alignment failed."
)

assert list(df_panchang_full.columns) == expected_panchang_columns, (
    "Unexpected raw F4 Panchang columns."
)

assert df_panchang_full.shape == (len(df_clean), 5), (
    f"Unexpected raw F4 shape: "
    f"{df_panchang_full.shape}"
)

assert df_panchang_full.isna().sum().sum() == 0, (
    "Missing values found in raw F4 Panchang features."
)

print("\nFull raw F4 Panchang table:")
print(f"Shape:          {df_panchang_full.shape}")
print(
    f"Missing values: "
    f"{df_panchang_full.isna().sum().sum()}"
)


# ------------------------------------------------------------
# 8. Verify the canonical first Panchang reference case
# ------------------------------------------------------------
#
# Notebook 08 established that the first SOLETE timestamp,
# evaluated astronomically at 00:02 UTC, must produce:
#
#   Tithi      = 18
#   Paksha     = Krishna
#   Karana     = Vanija
#   Nakshatra  = Mula
#   Yoga       = Shubha
# ------------------------------------------------------------

first_panchang = df_panchang_full.iloc[0]

assert int(first_panchang["tithi"]) == 18
assert first_panchang["paksha"] == "Krishna"
assert first_panchang["karana"] == "Vanija"
assert first_panchang["nakshatra"] == "Mula"
assert first_panchang["yoga"] == "Shubha"

print("\nCanonical first Panchang reference check: PASSED")

print("\nFirst raw F4 row:")
display(df_panchang_full.head(1))


# ------------------------------------------------------------
# 9. Select F4 categories at each forecast-valid time
# ------------------------------------------------------------
#
# As with F1-F3, Panchang features describe:
#
#     valid_time = t + h
#
# rather than the forecast issue time.
# ------------------------------------------------------------

panchang_feature_tables = {}

for horizon_minutes, df_model in base_modelling_tables.items():

    valid_times = pd.DatetimeIndex(
        df_model["valid_time"],
        name="valid_time",
    )

    # Select the already-computed Panchang state corresponding
    # exactly to each future forecast-valid timestamp.
    df_panchang = df_panchang_full.loc[valid_times].copy()

    assert df_panchang.index.equals(valid_times), (
        f"F4 valid-time alignment failed for "
        f"{horizon_minutes}-minute horizon."
    )

    assert df_panchang.shape[1] == 5

    assert df_panchang.isna().sum().sum() == 0

    # Prefix raw categorical columns to make their forecast
    # timing explicit.
    df_panchang = df_panchang.add_prefix("valid_")

    # Values describe valid_time, but modelling rows themselves
    # remain indexed by issue time.
    df_panchang.index = df_model.index

    panchang_feature_tables[horizon_minutes] = df_panchang


# ------------------------------------------------------------
# 10. Report horizon-specific raw F4 tables
# ------------------------------------------------------------

for horizon_minutes, df_panchang in (
    panchang_feature_tables.items()
):

    df_model = base_modelling_tables[horizon_minutes]

    print(f"\n{horizon_minutes}-minute horizon:")
    print(f"  Shape:          {df_panchang.shape}")
    print(
        f"  Missing values: "
        f"{df_panchang.isna().sum().sum()}"
    )

    print(
        f"  First issue:    "
        f"{df_model['issue_time'].iloc[0]}"
    )

    print(
        f"  First valid:    "
        f"{df_model['valid_time'].iloc[0]}"
    )


# ------------------------------------------------------------
# 11. Explicit 60-minute F4 alignment check
# ------------------------------------------------------------

df_panchang_60 = panchang_feature_tables[60]
df_model_60 = base_modelling_tables[60]

first_valid_60 = df_model_60["valid_time"].iloc[0]


# Canonical Panchang row at the forecast-valid timestamp.
canonical_f4_first_60 = df_panchang_full.loc[first_valid_60]


# First modelling Panchang row.
#
# Its column names contain the "valid_" prefix, whereas the
# canonical full-timeline table uses the original raw names.
model_f4_first_60 = df_panchang_60.iloc[0].copy()

model_f4_first_60.index = (
    model_f4_first_60.index
    .str.removeprefix("valid_")
)


# Now both Series have identical semantic labels and values.
assert canonical_f4_first_60.equals(
    model_f4_first_60
), (
    "60-minute F4 value alignment failed."
)


print("\n60-minute F4 alignment check: PASSED")

print("\nCanonical F4 at first 60-minute valid time:")
display(canonical_f4_first_60.to_frame("canonical"))

print("\nModelling F4 at the same valid time:")
display(model_f4_first_60.to_frame("model"))

print("\nFirst five 60-minute forecast-valid Panchang rows:")
display(df_panchang_60.head())

Full tropical-longitude table:
Shape:          (131617, 2)
Missing values: 0

First SOLETE timestamp:
2018-06-01 00:00:00+00:00
First astronomy reference time:
2018-06-01 00:02:00+00:00

Lahiri ayanamsha:
First value: 24.110343268621794
Minimum:     24.110343268621794
Maximum:     24.12737225615406

Canonical Lahiri reference check: PASSED

Full raw F4 Panchang table:
Shape:          (131617, 5)
Missing values: 0

Canonical first Panchang reference check: PASSED

First raw F4 row:


,tithi,paksha,karana,nakshatra,yoga
Timestamp,,,,,
2018-06-01 00:00:00+00:00,18,Krishna,Vanija,Mula,Shubha



15-minute horizon:
  Shape:          (131325, 5)
  Missing values: 0
  First issue:    2018-06-02 00:00:00+00:00
  First valid:    2018-06-02 00:15:00+00:00

30-minute horizon:
  Shape:          (131322, 5)
  Missing values: 0
  First issue:    2018-06-02 00:00:00+00:00
  First valid:    2018-06-02 00:30:00+00:00

60-minute horizon:
  Shape:          (131316, 5)
  Missing values: 0
  First issue:    2018-06-02 00:00:00+00:00
  First valid:    2018-06-02 01:00:00+00:00

60-minute F4 alignment check: PASSED

Canonical F4 at first 60-minute valid time:


,canonical
tithi,19
paksha,Krishna
karana,Bava
nakshatra,Purva Ashadha
yoga,Shukla



Modelling F4 at the same valid time:


,model
tithi,19
paksha,Krishna
karana,Bava
nakshatra,Purva Ashadha
yoga,Shukla



First five 60-minute forecast-valid Panchang rows:


,valid_tithi,valid_paksha,valid_karana,valid_nakshatra,valid_yoga
Timestamp,,,,,
2018-06-02 00:00:00+00:00,19,Krishna,Bava,Purva Ashadha,Shukla
2018-06-02 00:05:00+00:00,19,Krishna,Bava,Purva Ashadha,Shukla
2018-06-02 00:10:00+00:00,19,Krishna,Bava,Purva Ashadha,Shukla
2018-06-02 00:15:00+00:00,19,Krishna,Bava,Purva Ashadha,Shukla
2018-06-02 00:20:00+00:00,19,Krishna,Bava,Purva Ashadha,Shukla


In [13]:
# ============================================================
# Step 12: Assemble the nested F0-F4 feature-family tables
# ============================================================

# ------------------------------------------------------------
# 1. Storage structure
# ------------------------------------------------------------
#
# feature_family_tables[horizon][family]
#
# Example:
#
#   feature_family_tables[60]["F3"]
#
# returns the complete F3 model-input dataframe for the
# 60-minute forecast horizon.
# ------------------------------------------------------------

feature_family_tables = {}


# ------------------------------------------------------------
# 2. Assemble each feature family for every forecast horizon
# ------------------------------------------------------------
#
# Frozen hierarchy:
#
#   F0 = historical / autoregressive features
#   F1 = F0 + Gregorian
#   F2 = F1 + solar astronomy
#   F3 = F2 + lunar / Sun-Moon astronomy
#   F4_raw = F3 + raw Panchang categories
#
# F4 is called F4_raw here because the five Panchang columns
# have not yet been one-hot encoded.
# ------------------------------------------------------------

for horizon_minutes, df_model in base_modelling_tables.items():

    issue_index = df_model.index

    # --------------------------------------------------------
    # F0
    # --------------------------------------------------------

    df_f0 = df_f0_history.loc[issue_index].copy()

    # --------------------------------------------------------
    # F1 = F0 + 15 Gregorian features
    # --------------------------------------------------------

    df_f1 = pd.concat(
        [
            df_f0,
            gregorian_feature_tables[horizon_minutes],
        ],
        axis=1,
    )

    # --------------------------------------------------------
    # F2 = F1 + 5 solar-astronomy features
    # --------------------------------------------------------

    df_f2 = pd.concat(
        [
            df_f1,
            solar_feature_tables[horizon_minutes],
        ],
        axis=1,
    )

    # --------------------------------------------------------
    # F3 = F2 + 9 lunar / Sun-Moon features
    # --------------------------------------------------------

    df_f3 = pd.concat(
        [
            df_f2,
            lunar_feature_tables[horizon_minutes],
        ],
        axis=1,
    )

    # --------------------------------------------------------
    # F4_raw = F3 + 5 raw categorical Panchang features
    # --------------------------------------------------------

    df_f4_raw = pd.concat(
        [
            df_f3,
            panchang_feature_tables[horizon_minutes],
        ],
        axis=1,
    )

    feature_family_tables[horizon_minutes] = {
        "F0": df_f0,
        "F1": df_f1,
        "F2": df_f2,
        "F3": df_f3,
        "F4_raw": df_f4_raw,
    }


# ------------------------------------------------------------
# 3. Expected dimensions before F4 one-hot encoding
# ------------------------------------------------------------

EXPECTED_RAW_FEATURE_COUNTS = {
    "F0": 118,
    "F1": 133,
    "F2": 138,
    "F3": 147,
    "F4_raw": 152,   # 147 + 5 raw categorical columns
}


# ------------------------------------------------------------
# 4. Validate dimensions, alignment, and missingness
# ------------------------------------------------------------

for horizon_minutes, family_tables in feature_family_tables.items():

    expected_rows = len(base_modelling_tables[horizon_minutes])

    print(f"\n{horizon_minutes}-minute horizon:")

    for family_name, df_family in family_tables.items():

        expected_features = EXPECTED_RAW_FEATURE_COUNTS[
            family_name
        ]

        assert len(df_family) == expected_rows, (
            f"{horizon_minutes}-minute {family_name}: "
            "row-count mismatch."
        )

        assert df_family.shape[1] == expected_features, (
            f"{horizon_minutes}-minute {family_name}: "
            f"expected {expected_features} features, "
            f"found {df_family.shape[1]}."
        )

        assert df_family.index.equals(
            base_modelling_tables[horizon_minutes].index
        ), (
            f"{horizon_minutes}-minute {family_name}: "
            "issue-time index mismatch."
        )

        assert df_family.isna().sum().sum() == 0, (
            f"{horizon_minutes}-minute {family_name}: "
            "missing values found."
        )

        print(
            f"  {family_name:<6} "
            f"shape = {df_family.shape}"
        )


# ------------------------------------------------------------
# 5. Verify strict nested feature hierarchy
# ------------------------------------------------------------
#
# Every earlier family's columns must appear unchanged at the
# start of the next family.
# ------------------------------------------------------------

for horizon_minutes, family_tables in feature_family_tables.items():

    assert list(
        family_tables["F1"].columns[:118]
    ) == list(
        family_tables["F0"].columns
    )

    assert list(
        family_tables["F2"].columns[:133]
    ) == list(
        family_tables["F1"].columns
    )

    assert list(
        family_tables["F3"].columns[:138]
    ) == list(
        family_tables["F2"].columns
    )

    assert list(
        family_tables["F4_raw"].columns[:147]
    ) == list(
        family_tables["F3"].columns
    )

print("\nNested F0 -> F1 -> F2 -> F3 -> F4_raw checks: PASSED")


# ------------------------------------------------------------
# 6. Inspect the five raw F4 categorical columns
# ------------------------------------------------------------

RAW_PANCHANG_COLUMNS = [
    "valid_tithi",
    "valid_paksha",
    "valid_karana",
    "valid_nakshatra",
    "valid_yoga",
]

assert (
    list(feature_family_tables[60]["F4_raw"].columns[-5:])
    == RAW_PANCHANG_COLUMNS
)

print("\nRaw F4 categorical columns:")
for column in RAW_PANCHANG_COLUMNS:
    print(" -", column)


# ------------------------------------------------------------
# 7. Display a compact 60-minute example
# ------------------------------------------------------------

print("\nFirst 60-minute F4_raw modelling row:")

display(
    feature_family_tables[60]["F4_raw"][
        RAW_PANCHANG_COLUMNS
    ].head()
)


15-minute horizon:
  F0     shape = (131325, 118)
  F1     shape = (131325, 133)
  F2     shape = (131325, 138)
  F3     shape = (131325, 147)
  F4_raw shape = (131325, 152)

30-minute horizon:
  F0     shape = (131322, 118)
  F1     shape = (131322, 133)
  F2     shape = (131322, 138)
  F3     shape = (131322, 147)
  F4_raw shape = (131322, 152)

60-minute horizon:
  F0     shape = (131316, 118)
  F1     shape = (131316, 133)
  F2     shape = (131316, 138)
  F3     shape = (131316, 147)
  F4_raw shape = (131316, 152)

Nested F0 -> F1 -> F2 -> F3 -> F4_raw checks: PASSED

Raw F4 categorical columns:
 - valid_tithi
 - valid_paksha
 - valid_karana
 - valid_nakshatra
 - valid_yoga

First 60-minute F4_raw modelling row:


,valid_tithi,valid_paksha,valid_karana,valid_nakshatra,valid_yoga
Timestamp,,,,,
2018-06-02 00:00:00+00:00,19,Krishna,Bava,Purva Ashadha,Shukla
2018-06-02 00:05:00+00:00,19,Krishna,Bava,Purva Ashadha,Shukla
2018-06-02 00:10:00+00:00,19,Krishna,Bava,Purva Ashadha,Shukla
2018-06-02 00:15:00+00:00,19,Krishna,Bava,Purva Ashadha,Shukla
2018-06-02 00:20:00+00:00,19,Krishna,Bava,Purva Ashadha,Shukla


In [14]:
# ============================================================
# Step 13: Inspect canonical Panchang category vocabularies
# ============================================================

import src.vedic_calendar as vedic_calendar


# ------------------------------------------------------------
# Find module-level constants related to Panchang category
# names.
#
# We want the one-hot encoder to use the same canonical
# vocabulary as the validated calendar implementation rather
# than maintaining a second independent spelling/order list
# inside this notebook.
# ------------------------------------------------------------

candidate_constants = [
    name
    for name in dir(vedic_calendar)
    if any(
        keyword in name.upper()
        for keyword in [
            "KARANA",
            "NAKSHATRA",
            "YOGA",
        ]
    )
    and name.isupper()
]


print("Candidate Panchang constants:\n")

for name in candidate_constants:
    value = getattr(vedic_calendar, name)

    if isinstance(value, (list, tuple)):
        print(f"{name}:")
        print(value)
        print()

Candidate Panchang constants:

KARANA_SEQUENCE:
('Kimstughna', 'Bava', 'Balava', 'Kaulava', 'Taitila', 'Gara', 'Vanija', 'Vishti', 'Bava', 'Balava', 'Kaulava', 'Taitila', 'Gara', 'Vanija', 'Vishti', 'Bava', 'Balava', 'Kaulava', 'Taitila', 'Gara', 'Vanija', 'Vishti', 'Bava', 'Balava', 'Kaulava', 'Taitila', 'Gara', 'Vanija', 'Vishti', 'Bava', 'Balava', 'Kaulava', 'Taitila', 'Gara', 'Vanija', 'Vishti', 'Bava', 'Balava', 'Kaulava', 'Taitila', 'Gara', 'Vanija', 'Vishti', 'Bava', 'Balava', 'Kaulava', 'Taitila', 'Gara', 'Vanija', 'Vishti', 'Bava', 'Balava', 'Kaulava', 'Taitila', 'Gara', 'Vanija', 'Vishti', 'Shakuni', 'Chatushpada', 'Naga')

MOVABLE_KARANAS:
('Bava', 'Balava', 'Kaulava', 'Taitila', 'Gara', 'Vanija', 'Vishti')

NAKSHATRA_NAMES:
('Ashwini', 'Bharani', 'Krittika', 'Rohini', 'Mrigashirsha', 'Ardra', 'Punarvasu', 'Pushya', 'Ashlesha', 'Magha', 'Purva Phalguni', 'Uttara Phalguni', 'Hasta', 'Chitra', 'Swati', 'Vishakha', 'Anuradha', 'Jyeshtha', 'Mula', 'Purva Ashadha', 'Uttara Ashadh

In [15]:
# ============================================================
# Step 14: One-hot encode the Panchang F4 feature block
# ============================================================

# ------------------------------------------------------------
# 1. Define the fixed canonical category vocabularies
# ------------------------------------------------------------
#
# These vocabularies are NOT learned from the modelling data.
#
# They come from the deterministic Panchang system itself.
# This guarantees that:
#
#   - train / validation / test use the same categories;
#   - all forecast horizons use the same categories;
#   - F4 dimensionality is fixed;
#   - a temporarily absent category does not change columns.
# ------------------------------------------------------------

TITHI_CATEGORIES = tuple(range(1, 31))

PAKSHA_CATEGORIES = (
    "Shukla",
    "Krishna",
)

# KARANA_SEQUENCE contains repeated movable Karanas across the
# 60 half-Tithi slots.
#
# dict.fromkeys(...) removes repetitions while preserving the
# canonical first-appearance order.
KARANA_CATEGORIES = tuple(
    dict.fromkeys(vedic_calendar.KARANA_SEQUENCE)
)

NAKSHATRA_CATEGORIES = tuple(
    vedic_calendar.NAKSHATRA_NAMES
)

YOGA_CATEGORIES = tuple(
    vedic_calendar.YOGA_NAMES
)


# ------------------------------------------------------------
# 2. Verify canonical vocabulary sizes
# ------------------------------------------------------------

assert len(TITHI_CATEGORIES) == 30
assert len(PAKSHA_CATEGORIES) == 2
assert len(KARANA_CATEGORIES) == 11
assert len(NAKSHATRA_CATEGORIES) == 27
assert len(YOGA_CATEGORIES) == 27

TOTAL_PANCHANG_INDICATORS = (
    len(TITHI_CATEGORIES)
    + len(PAKSHA_CATEGORIES)
    + len(KARANA_CATEGORIES)
    + len(NAKSHATRA_CATEGORIES)
    + len(YOGA_CATEGORIES)
)

assert TOTAL_PANCHANG_INDICATORS == 97

print("Canonical Panchang vocabulary sizes:")
print(f"  Tithi:      {len(TITHI_CATEGORIES)}")
print(f"  Paksha:     {len(PAKSHA_CATEGORIES)}")
print(f"  Karana:     {len(KARANA_CATEGORIES)}")
print(f"  Nakshatra:  {len(NAKSHATRA_CATEGORIES)}")
print(f"  Yoga:       {len(YOGA_CATEGORIES)}")
print(f"  Total:      {TOTAL_PANCHANG_INDICATORS}")


# ------------------------------------------------------------
# 3. Inspect the derived canonical Karana vocabulary
# ------------------------------------------------------------

print("\nCanonical Karana categories:")
print(KARANA_CATEGORIES)


# ------------------------------------------------------------
# 4. Helper: fixed-vocabulary Panchang one-hot encoding
# ------------------------------------------------------------

def encode_panchang_fixed_vocabulary(
    df_panchang: pd.DataFrame,
) -> pd.DataFrame:
    """
    Convert the five raw forecast-valid Panchang columns into
    exactly 97 deterministic one-hot indicator columns.

    The category vocabularies are fixed in advance and are not
    learned from the supplied dataframe.
    """

    required_columns = [
        "valid_tithi",
        "valid_paksha",
        "valid_karana",
        "valid_nakshatra",
        "valid_yoga",
    ]

    assert list(df_panchang.columns) == required_columns, (
        "Unexpected raw Panchang column structure."
    )

    # --------------------------------------------------------
    # Before converting to pandas Categoricals, explicitly
    # verify that every observed value belongs to the canonical
    # vocabulary.
    #
    # This prevents an unexpected category from silently being
    # converted to NaN.
    # --------------------------------------------------------

    vocabulary_map = {
        "valid_tithi": TITHI_CATEGORIES,
        "valid_paksha": PAKSHA_CATEGORIES,
        "valid_karana": KARANA_CATEGORIES,
        "valid_nakshatra": NAKSHATRA_CATEGORIES,
        "valid_yoga": YOGA_CATEGORIES,
    }

    for column, categories in vocabulary_map.items():

        observed = set(df_panchang[column].unique())
        allowed = set(categories)

        unexpected = observed - allowed

        assert not unexpected, (
            f"Unexpected categories in {column}: "
            f"{sorted(unexpected)}"
        )

    # --------------------------------------------------------
    # Convert each raw variable to a Categorical with an
    # explicitly fixed category set.
    # --------------------------------------------------------

    df_categorical = pd.DataFrame(
        index=df_panchang.index
    )

    for column, categories in vocabulary_map.items():

        df_categorical[column] = pd.Categorical(
            df_panchang[column],
            categories=categories,
            ordered=False,
        )

    # --------------------------------------------------------
    # Full one-hot encoding.
    #
    # Because the Categorical vocabularies were explicitly
    # supplied above, pandas creates columns for every canonical
    # state even if a category were absent from this particular
    # modelling interval.
    # --------------------------------------------------------

    df_encoded = pd.get_dummies(
        df_categorical,
        prefix={
            "valid_tithi": "valid_tithi",
            "valid_paksha": "valid_paksha",
            "valid_karana": "valid_karana",
            "valid_nakshatra": "valid_nakshatra",
            "valid_yoga": "valid_yoga",
        },
        prefix_sep="_",
        dtype="int8",
    )

    assert df_encoded.shape[1] == 97, (
        f"Expected 97 Panchang indicators, "
        f"found {df_encoded.shape[1]}."
    )

    assert df_encoded.isna().sum().sum() == 0

    return df_encoded


# ------------------------------------------------------------
# 5. Encode F4 for each forecast horizon
# ------------------------------------------------------------

panchang_onehot_tables = {}

for horizon_minutes in FORECAST_HORIZONS:

    df_raw_panchang = panchang_feature_tables[
        horizon_minutes
    ]

    df_onehot = encode_panchang_fixed_vocabulary(
        df_raw_panchang
    )

    panchang_onehot_tables[horizon_minutes] = df_onehot


# ------------------------------------------------------------
# 6. Verify each one-hot group activates exactly once per row
# ------------------------------------------------------------
#
# Every modelling row must contain:
#
#   exactly one Tithi
#   exactly one Paksha
#   exactly one Karana
#   exactly one Nakshatra
#   exactly one Yoga
#
# Therefore the 97-column Panchang block must contain exactly
# five active indicators per row.
# ------------------------------------------------------------

for horizon_minutes, df_onehot in panchang_onehot_tables.items():

    row_sums = df_onehot.sum(axis=1)

    assert (row_sums == 5).all(), (
        f"{horizon_minutes}-minute Panchang one-hot rows "
        "do not contain exactly five active indicators."
    )

    print(
        f"\n{horizon_minutes}-minute one-hot Panchang:"
    )
    print(f"  Shape: {df_onehot.shape}")
    print(
        f"  Active indicators per row: "
        f"{row_sums.min()} to {row_sums.max()}"
    )


# ------------------------------------------------------------
# 7. Construct the final F4 feature table
# ------------------------------------------------------------
#
# F4 is:
#
#     F3
#     +
#     97 one-hot Panchang indicators
#
# The five raw categorical columns are NOT included as separate
# model inputs.
# ------------------------------------------------------------

for horizon_minutes in FORECAST_HORIZONS:

    df_f3 = feature_family_tables[
        horizon_minutes
    ]["F3"]

    df_panchang_encoded = panchang_onehot_tables[
        horizon_minutes
    ]

    df_f4 = pd.concat(
        [
            df_f3,
            df_panchang_encoded,
        ],
        axis=1,
    )

    assert df_f4.shape[1] == 244, (
        f"{horizon_minutes}-minute F4 expected 244 "
        f"features, found {df_f4.shape[1]}."
    )

    assert df_f4.isna().sum().sum() == 0

    assert df_f4.index.equals(df_f3.index)

    feature_family_tables[
        horizon_minutes
    ]["F4"] = df_f4


# ------------------------------------------------------------
# 8. Final F0-F4 dimensionality check
# ------------------------------------------------------------

EXPECTED_FINAL_FEATURE_COUNTS = {
    "F0": 118,
    "F1": 133,
    "F2": 138,
    "F3": 147,
    "F4": 244,
}

print("\nFinal model-input feature dimensions:")

for horizon_minutes in FORECAST_HORIZONS:

    print(f"\n{horizon_minutes}-minute horizon:")

    for family_name, expected_count in (
        EXPECTED_FINAL_FEATURE_COUNTS.items()
    ):

        df_family = feature_family_tables[
            horizon_minutes
        ][family_name]

        assert df_family.shape[1] == expected_count

        print(
            f"  {family_name}: "
            f"{df_family.shape}"
        )


# ------------------------------------------------------------
# 9. Confirm final nested F3 -> F4 structure
# ------------------------------------------------------------

for horizon_minutes in FORECAST_HORIZONS:

    df_f3 = feature_family_tables[
        horizon_minutes
    ]["F3"]

    df_f4 = feature_family_tables[
        horizon_minutes
    ]["F4"]

    assert list(
        df_f4.columns[:147]
    ) == list(
        df_f3.columns
    )

print("\nFinal F3 -> F4 nesting check: PASSED")


# ------------------------------------------------------------
# 10. Inspect the first 60-minute Panchang encoding
# ------------------------------------------------------------

df_onehot_60 = panchang_onehot_tables[60]

first_active_columns = (
    df_onehot_60
    .iloc[0]
    .loc[lambda s: s == 1]
    .index
    .tolist()
)

print(
    "\nActive Panchang indicators for the first "
    "60-minute modelling row:"
)

for column in first_active_columns:
    print(" -", column)

Canonical Panchang vocabulary sizes:
  Tithi:      30
  Paksha:     2
  Karana:     11
  Nakshatra:  27
  Yoga:       27
  Total:      97

Canonical Karana categories:
('Kimstughna', 'Bava', 'Balava', 'Kaulava', 'Taitila', 'Gara', 'Vanija', 'Vishti', 'Shakuni', 'Chatushpada', 'Naga')

15-minute one-hot Panchang:
  Shape: (131325, 97)
  Active indicators per row: 5 to 5

30-minute one-hot Panchang:
  Shape: (131322, 97)
  Active indicators per row: 5 to 5

60-minute one-hot Panchang:
  Shape: (131316, 97)
  Active indicators per row: 5 to 5

Final model-input feature dimensions:

15-minute horizon:
  F0: (131325, 118)
  F1: (131325, 133)
  F2: (131325, 138)
  F3: (131325, 147)
  F4: (131325, 244)

30-minute horizon:
  F0: (131322, 118)
  F1: (131322, 133)
  F2: (131322, 138)
  F3: (131322, 147)
  F4: (131322, 244)

60-minute horizon:
  F0: (131316, 118)
  F1: (131316, 133)
  F2: (131316, 138)
  F3: (131316, 147)
  F4: (131316, 244)

Final F3 -> F4 nesting check: PASSED

Active Panchang 

In [16]:
# ============================================================
# Step 15: Build model-ready X / y / split bundles
# ============================================================

# ------------------------------------------------------------
# 1. Storage convention
# ------------------------------------------------------------
#
# modelling_bundles[horizon] contains:
#
#   "X"              -> dictionary of F0-F4 feature matrices
#   "y"              -> future P_Solar target
#   "metadata"       -> issue time, valid time, split, horizon
#   "train_mask"     -> chronological training rows
#   "validation_mask"-> chronological validation rows
#   "test_mask"      -> chronological test rows
#   "daylight_mask"  -> solar daylight at forecast-valid time
#
# Nothing is scaled or fitted here.
# ------------------------------------------------------------

modelling_bundles = {}


# ------------------------------------------------------------
# 2. Construct one bundle per forecast horizon
# ------------------------------------------------------------

for horizon_minutes in FORECAST_HORIZONS:

    df_base = base_modelling_tables[horizon_minutes]

    # --------------------------------------------------------
    # Target
    # --------------------------------------------------------
    #
    # Keep the target as a Series indexed by forecast issue time.
    # --------------------------------------------------------

    y = (
        df_base["target_P_Solar_kW"]
        .copy()
        .rename("target_P_Solar_kW")
    )


    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    metadata = df_base[
        [
            "issue_time",
            "valid_time",
            "horizon_minutes",
            "split",
        ]
    ].copy()


    # --------------------------------------------------------
    # Chronological split masks
    # --------------------------------------------------------

    train_mask = metadata["split"].eq("train")

    validation_mask = metadata["split"].eq("validation")

    test_mask = metadata["split"].eq("test")


    # --------------------------------------------------------
    # Daylight mask
    # --------------------------------------------------------
    #
    # Daylight is defined deterministically from the F2 solar
    # geometry at forecast-valid time:
    #
    #     solar elevation > 0 degrees
    #
    # The reusable F2 pipeline already expresses this through:
    #
    #     valid_solar_daylight
    #
    # This avoids defining daylight using the observed target.
    # --------------------------------------------------------

    daylight_mask = (
        solar_feature_tables[horizon_minutes][
            "valid_solar_daylight"
        ]
        .eq(1)
    )


    # --------------------------------------------------------
    # F0-F4 model-input matrices
    # --------------------------------------------------------

    X_tables = {
        family_name: (
            feature_family_tables[horizon_minutes][family_name]
            .copy()
        )
        for family_name in ["F0", "F1", "F2", "F3", "F4"]
    }


    # --------------------------------------------------------
    # 3. Alignment checks
    # --------------------------------------------------------

    canonical_index = df_base.index

    assert y.index.equals(canonical_index)

    assert metadata.index.equals(canonical_index)

    assert daylight_mask.index.equals(canonical_index)

    for family_name, X in X_tables.items():

        assert X.index.equals(canonical_index), (
            f"{horizon_minutes}-minute {family_name}: "
            "X/y index alignment failed."
        )

        assert len(X) == len(y)

        assert X.isna().sum().sum() == 0


    # Target must also be complete.
    assert y.isna().sum() == 0


    # Every row must belong to exactly one chronological split.
    split_membership_count = (
        train_mask.astype(int)
        + validation_mask.astype(int)
        + test_mask.astype(int)
    )

    assert (split_membership_count == 1).all(), (
        f"{horizon_minutes}-minute split membership error."
    )


    # --------------------------------------------------------
    # 4. Store the complete bundle
    # --------------------------------------------------------

    modelling_bundles[horizon_minutes] = {
        "X": X_tables,
        "y": y,
        "metadata": metadata,
        "train_mask": train_mask,
        "validation_mask": validation_mask,
        "test_mask": test_mask,
        "daylight_mask": daylight_mask,
    }


# ------------------------------------------------------------
# 5. Report the finalized modelling structure
# ------------------------------------------------------------

for horizon_minutes, bundle in modelling_bundles.items():

    print(f"\n{horizon_minutes}-minute horizon:")

    print(f"  Total rows:       {len(bundle['y'])}")
    print(
        f"  Train rows:       "
        f"{bundle['train_mask'].sum()}"
    )
    print(
        f"  Validation rows:  "
        f"{bundle['validation_mask'].sum()}"
    )
    print(
        f"  Test rows:        "
        f"{bundle['test_mask'].sum()}"
    )
    print(
        f"  Daylight rows:    "
        f"{bundle['daylight_mask'].sum()}"
    )

    print("  Feature shapes:")

    for family_name in ["F0", "F1", "F2", "F3", "F4"]:
        print(
            f"    {family_name}: "
            f"{bundle['X'][family_name].shape}"
        )


# ------------------------------------------------------------
# 6. Explicit 60-minute target alignment audit
# ------------------------------------------------------------

bundle_60 = modelling_bundles[60]

first_issue_60 = bundle_60["metadata"]["issue_time"].iloc[0]
first_valid_60 = bundle_60["metadata"]["valid_time"].iloc[0]
first_target_60 = bundle_60["y"].iloc[0]

expected_target_60 = df_clean.loc[
    first_valid_60,
    "P_Solar[kW]",
]

assert first_target_60 == expected_target_60

print("\n60-minute first-row audit:")
print(f"  Issue time:       {first_issue_60}")
print(f"  Valid time:       {first_valid_60}")
print(f"  Stored target:    {first_target_60}")
print(f"  Source P_Solar:   {expected_target_60}")

print("\nModel-ready bundle construction: PASSED")


15-minute horizon:
  Total rows:       131325
  Train rows:       87261
  Validation rows:  17568
  Test rows:        26496
  Daylight rows:    70543
  Feature shapes:
    F0: (131325, 118)
    F1: (131325, 133)
    F2: (131325, 138)
    F3: (131325, 147)
    F4: (131325, 244)

30-minute horizon:
  Total rows:       131322
  Train rows:       87258
  Validation rows:  17568
  Test rows:        26496
  Daylight rows:    70543
  Feature shapes:
    F0: (131322, 118)
    F1: (131322, 133)
    F2: (131322, 138)
    F3: (131322, 147)
    F4: (131322, 244)

60-minute horizon:
  Total rows:       131316
  Train rows:       87252
  Validation rows:  17568
  Test rows:        26496
  Daylight rows:    70543
  Feature shapes:
    F0: (131316, 118)
    F1: (131316, 133)
    F2: (131316, 138)
    F3: (131316, 147)
    F4: (131316, 244)

60-minute first-row audit:
  Issue time:       2018-06-02 00:00:00+00:00
  Valid time:       2018-06-02 01:00:00+00:00
  Stored target:    0.0
  Source P_Solar:  

In [17]:
# ============================================================
# Step 16: Evaluate non-ML forecasting baselines
# ============================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)


# ------------------------------------------------------------
# 1. Fixed PV capacity used for normalized error metrics
# ------------------------------------------------------------

PV_RATED_KW = 10.0


# ------------------------------------------------------------
# 2. Metric helper
# ------------------------------------------------------------

def regression_metrics(
    y_true: pd.Series,
    y_pred: pd.Series,
) -> dict:
    """
    Calculate the frozen regression metrics.

    Returns:
        MAE   : kW
        RMSE  : kW
        nMAE  : % of nominal 10 kW PV capacity
        nRMSE : % of nominal 10 kW PV capacity
        R2    : supplementary coefficient of determination
    """

    mae = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    nmae = (
        mae / PV_RATED_KW
    ) * 100.0

    nrmse = (
        rmse / PV_RATED_KW
    ) * 100.0

    r2 = r2_score(
        y_true,
        y_pred,
    )

    return {
        "MAE_kW": mae,
        "RMSE_kW": rmse,
        "nMAE_percent": nmae,
        "nRMSE_percent": nrmse,
        "R2": r2,
    }


# ------------------------------------------------------------
# 3. Construct baseline predictions
# ------------------------------------------------------------
#
# B1 — Persistence
#
#     P_hat(t+h) = P_Solar(t)
#
#
# B2 — Daily Persistence
#
#     P_hat(t+h) = P_Solar(t+h-24h)
#
#
# B3 — Training Mean
#
#     P_hat = mean training target
#
# B3 is only a diagnostic reference and is estimated using
# training targets only.
# ------------------------------------------------------------

baseline_prediction_tables = {}


for horizon_minutes, bundle in modelling_bundles.items():

    metadata = bundle["metadata"]
    y = bundle["y"]

    issue_index = metadata.index

    # --------------------------------------------------------
    # B1: Persistence
    # --------------------------------------------------------

    persistence = (
        df_clean
        .loc[
            issue_index,
            "P_Solar[kW]",
        ]
        .copy()
    )

    persistence.index = issue_index
    persistence.name = "Persistence"


    # --------------------------------------------------------
    # B2: Daily Persistence
    # --------------------------------------------------------
    #
    # Reference timestamp:
    #
    #     valid_time - 24 hours
    #
    # This differs intentionally from the F0 feature:
    #
    #     P_Solar(t - 24h)
    # --------------------------------------------------------

    daily_reference_times = pd.DatetimeIndex(
        metadata["valid_time"]
        - pd.Timedelta(hours=24)
    )

    daily_values = (
        df_clean["P_Solar[kW]"]
        .reindex(daily_reference_times)
    )

    assert daily_values.notna().all(), (
        f"{horizon_minutes}-minute daily-persistence "
        "reference contains missing values."
    )

    daily_persistence = pd.Series(
        daily_values.to_numpy(),
        index=issue_index,
        name="Daily Persistence",
    )


    # --------------------------------------------------------
    # B3: Training-set mean
    # --------------------------------------------------------

    training_mean_value = (
        y.loc[bundle["train_mask"]]
        .mean()
    )

    training_mean = pd.Series(
        training_mean_value,
        index=issue_index,
        name="Training Mean",
    )


    # --------------------------------------------------------
    # Store all three prediction series
    # --------------------------------------------------------

    df_predictions = pd.concat(
        [
            persistence,
            daily_persistence,
            training_mean,
        ],
        axis=1,
    )

    assert df_predictions.index.equals(issue_index)

    assert df_predictions.isna().sum().sum() == 0

    baseline_prediction_tables[
        horizon_minutes
    ] = df_predictions


# ------------------------------------------------------------
# 4. Evaluate validation and test periods
# ------------------------------------------------------------
#
# Each baseline is evaluated under two scopes:
#
#   all       -> every valid forecast timestamp
#   daylight  -> solar elevation > 0 at valid time
# ------------------------------------------------------------

baseline_result_rows = []


for horizon_minutes, bundle in modelling_bundles.items():

    y = bundle["y"]

    predictions = baseline_prediction_tables[
        horizon_minutes
    ]

    split_masks = {
        "validation": bundle["validation_mask"],
        "test": bundle["test_mask"],
    }

    scope_masks = {
        "all": pd.Series(
            True,
            index=y.index,
        ),
        "daylight": bundle["daylight_mask"],
    }


    for split_name, split_mask in split_masks.items():

        for scope_name, scope_mask in scope_masks.items():

            evaluation_mask = (
                split_mask
                & scope_mask
            )

            y_true = y.loc[
                evaluation_mask
            ]

            for baseline_name in predictions.columns:

                y_pred = predictions.loc[
                    evaluation_mask,
                    baseline_name,
                ]

                metrics = regression_metrics(
                    y_true,
                    y_pred,
                )

                baseline_result_rows.append(
                    {
                        "horizon_minutes": horizon_minutes,
                        "split": split_name,
                        "scope": scope_name,
                        "baseline": baseline_name,
                        "n_rows": len(y_true),
                        **metrics,
                    }
                )


# ------------------------------------------------------------
# 5. Assemble results table
# ------------------------------------------------------------

baseline_results = pd.DataFrame(
    baseline_result_rows
)

baseline_results = baseline_results.sort_values(
    [
        "horizon_minutes",
        "split",
        "scope",
        "baseline",
    ]
).reset_index(drop=True)


# ------------------------------------------------------------
# 6. Sanity checks
# ------------------------------------------------------------

assert len(baseline_results) == (
    3       # horizons
    * 2     # validation/test
    * 2     # all/daylight
    * 3     # baselines
)

assert np.isfinite(
    baseline_results[
        [
            "MAE_kW",
            "RMSE_kW",
            "nMAE_percent",
            "nRMSE_percent",
            "R2",
        ]
    ].to_numpy()
).all()

print("Baseline evaluation: PASSED")


# ------------------------------------------------------------
# 7. Show the primary 60-minute TEST results
# ------------------------------------------------------------

primary_baseline_results = (
    baseline_results.loc[
        (baseline_results["horizon_minutes"] == 60)
        & (baseline_results["split"] == "test")
    ]
    .copy()
)

print("\nPrimary 60-minute test baseline results:")

display(
    primary_baseline_results[
        [
            "scope",
            "baseline",
            "n_rows",
            "MAE_kW",
            "RMSE_kW",
            "nMAE_percent",
            "nRMSE_percent",
            "R2",
        ]
    ]
)


# ------------------------------------------------------------
# 8. Show persistence MAE across all three horizons
# ------------------------------------------------------------

persistence_summary = (
    baseline_results.loc[
        (baseline_results["split"] == "test")
        & (baseline_results["baseline"] == "Persistence")
    ]
    [
        [
            "horizon_minutes",
            "scope",
            "n_rows",
            "MAE_kW",
            "RMSE_kW",
        ]
    ]
    .sort_values(
        [
            "scope",
            "horizon_minutes",
        ]
    )
)

print("\nPersistence test performance across horizons:")

display(persistence_summary)

Baseline evaluation: PASSED

Primary 60-minute test baseline results:


,scope,baseline,n_rows,MAE_kW,RMSE_kW,nMAE_percent,nRMSE_percent,R2
24,all,Daily Persistence,26496,0.579710,1.171413,5.797097,11.714128,0.496376
25,all,Persistence,26496,0.520977,0.921371,5.209765,9.213714,0.688430
26,all,Training Mean,26496,1.230175,1.693070,12.301751,16.930699,-0.052049
27,daylight,Daily Persistence,17851,0.859875,1.427145,8.598750,14.271445,0.317626
28,daylight,Persistence,17851,0.766041,1.122071,7.660405,11.220714,0.578179
29,daylight,Training Mean,17851,1.420386,1.978652,14.203856,19.786521,-0.311673



Persistence test performance across horizons:


,horizon_minutes,scope,n_rows,MAE_kW,RMSE_kW
1,15,all,26496,0.291366,0.653352
13,30,all,26496,0.380360,0.762045
25,60,all,26496,0.520977,0.921371
4,15,daylight,17851,0.431529,0.795973
16,30,daylight,17851,0.562241,0.928341
28,60,daylight,17851,0.766041,1.122071


In [18]:
# ============================================================
# Step 17: Validate the Ridge pipeline on 60-minute F0
# ============================================================

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


# ------------------------------------------------------------
# 1. Select the primary horizon and baseline feature family
# ------------------------------------------------------------

HORIZON = 60
FEATURE_FAMILY = "F0"

bundle = modelling_bundles[HORIZON]

X = bundle["X"][FEATURE_FAMILY]
y = bundle["y"]

train_mask = bundle["train_mask"]
validation_mask = bundle["validation_mask"]


X_train = X.loc[train_mask]
y_train = y.loc[train_mask]

X_validation = X.loc[validation_mask]
y_validation = y.loc[validation_mask]


print("60-minute F0 Ridge experiment:")
print(f"  X_train:       {X_train.shape}")
print(f"  y_train:       {y_train.shape}")
print(f"  X_validation:  {X_validation.shape}")
print(f"  y_validation:  {y_validation.shape}")


# ------------------------------------------------------------
# 2. Frozen Ridge alpha grid
# ------------------------------------------------------------

RIDGE_ALPHA_GRID = [
    0.01,
    0.1,
    1.0,
    10.0,
    100.0,
]


# ------------------------------------------------------------
# 3. Tune alpha using chronological validation MAE
# ------------------------------------------------------------
#
# StandardScaler is inside the Pipeline.
#
# Therefore, for every candidate alpha:
#
#   scaler.fit()
#
# sees TRAINING data only.
#
# Validation data is transformed using the training-fitted
# scaler, which prevents preprocessing leakage.
# ------------------------------------------------------------

ridge_validation_rows = []


for alpha in RIDGE_ALPHA_GRID:

    ridge_pipeline = Pipeline(
        steps=[
            (
                "scaler",
                StandardScaler(),
            ),
            (
                "ridge",
                Ridge(alpha=alpha),
            ),
        ]
    )

    ridge_pipeline.fit(
        X_train,
        y_train,
    )

    validation_predictions = ridge_pipeline.predict(
        X_validation
    )

    validation_metrics = regression_metrics(
        y_validation,
        validation_predictions,
    )

    ridge_validation_rows.append(
        {
            "alpha": alpha,
            **validation_metrics,
        }
    )


# ------------------------------------------------------------
# 4. Assemble validation tuning results
# ------------------------------------------------------------

ridge_f0_60_validation = pd.DataFrame(
    ridge_validation_rows
).sort_values(
    "MAE_kW"
).reset_index(drop=True)


print("\n60-minute F0 Ridge validation results:")

display(ridge_f0_60_validation)


# ------------------------------------------------------------
# 5. Select the alpha with minimum validation MAE
# ------------------------------------------------------------

best_ridge_f0_60_alpha = (
    ridge_f0_60_validation
    .iloc[0]["alpha"]
)

best_ridge_f0_60_mae = (
    ridge_f0_60_validation
    .iloc[0]["MAE_kW"]
)


assert best_ridge_f0_60_alpha in RIDGE_ALPHA_GRID


print(
    "\nBest alpha:",
    best_ridge_f0_60_alpha,
)

print(
    "Best validation MAE:",
    best_ridge_f0_60_mae,
    "kW",
)


# ------------------------------------------------------------
# 6. Compare against 60-minute validation persistence
# ------------------------------------------------------------
#
# We compare validation with validation here.
#
# The TEST set remains completely untouched.
# ------------------------------------------------------------

validation_persistence_mae = (
    baseline_results.loc[
        (baseline_results["horizon_minutes"] == 60)
        & (baseline_results["split"] == "validation")
        & (baseline_results["scope"] == "all")
        & (baseline_results["baseline"] == "Persistence"),
        "MAE_kW",
    ]
    .iloc[0]
)


print(
    "\nValidation persistence MAE:",
    validation_persistence_mae,
    "kW",
)

print(
    "Ridge F0 validation MAE:",
    best_ridge_f0_60_mae,
    "kW",
)


print("\n60-minute F0 Ridge tuning: PASSED")

60-minute F0 Ridge experiment:
  X_train:       (87252, 118)
  y_train:       (87252,)
  X_validation:  (17568, 118)
  y_validation:  (17568,)

60-minute F0 Ridge validation results:


,alpha,MAE_kW,RMSE_kW,nMAE_percent,nRMSE_percent,R2
0,100.00,0.512713,0.863786,5.127134,8.637857,0.801459
1,10.00,0.512911,0.863378,5.129107,8.633782,0.801646
2,1.00,0.512946,0.863341,5.129463,8.633407,0.801664
3,0.10,0.512950,0.863337,5.129501,8.633371,0.801665
4,0.01,0.512950,0.863337,5.129504,8.633367,0.801666



Best alpha: 100.0
Best validation MAE: 0.5127134159791293 kW

Validation persistence MAE: 0.567365010004198 kW
Ridge F0 validation MAE: 0.5127134159791293 kW

60-minute F0 Ridge tuning: PASSED


In [19]:
# ============================================================
# Step 18: Ridge F0-F4 ablation for the 60-minute horizon
# ============================================================

# ------------------------------------------------------------
# 1. Use the primary 60-minute forecasting horizon
# ------------------------------------------------------------

HORIZON = 60

bundle = modelling_bundles[HORIZON]

y = bundle["y"]

train_mask = bundle["train_mask"]
validation_mask = bundle["validation_mask"]

y_train = y.loc[train_mask]
y_validation = y.loc[validation_mask]


# ------------------------------------------------------------
# 2. Evaluate every frozen feature family
# ------------------------------------------------------------
#
# For each feature family:
#
#   F0
#   F1
#   F2
#   F3
#   F4
#
# we apply the SAME:
#
#   - chronological training period;
#   - chronological validation period;
#   - StandardScaler -> Ridge pipeline;
#   - alpha grid;
#   - validation-MAE selection rule.
#
# This preserves the controlled feature-ablation design.
# ------------------------------------------------------------

ridge_60_tuning_rows = []


for feature_family in ["F0", "F1", "F2", "F3", "F4"]:

    X = bundle["X"][feature_family]

    X_train = X.loc[train_mask]
    X_validation = X.loc[validation_mask]

    print(
        f"\nTuning Ridge: "
        f"{HORIZON}-minute / {feature_family}"
    )

    print(
        f"  Train shape:      {X_train.shape}"
    )

    print(
        f"  Validation shape: {X_validation.shape}"
    )


    # --------------------------------------------------------
    # Test every alpha from the frozen candidate grid.
    # --------------------------------------------------------

    for alpha in RIDGE_ALPHA_GRID:

        model = Pipeline(
            steps=[
                (
                    "scaler",
                    StandardScaler(),
                ),
                (
                    "ridge",
                    Ridge(alpha=alpha),
                ),
            ]
        )

        # Fit using TRAINING data only.
        model.fit(
            X_train,
            y_train,
        )

        # Evaluate on the dedicated validation period.
        validation_predictions = model.predict(
            X_validation
        )

        metrics = regression_metrics(
            y_validation,
            validation_predictions,
        )

        ridge_60_tuning_rows.append(
            {
                "horizon_minutes": HORIZON,
                "feature_family": feature_family,
                "alpha": alpha,
                **metrics,
            }
        )


# ------------------------------------------------------------
# 3. Assemble all tuning results
# ------------------------------------------------------------

ridge_60_tuning_results = pd.DataFrame(
    ridge_60_tuning_rows
)


# ------------------------------------------------------------
# 4. Select the best alpha independently for each F0-F4 family
# ------------------------------------------------------------
#
# Selection criterion:
#
#     minimum validation MAE
#
# This follows the frozen protocol.
# ------------------------------------------------------------

ridge_60_best_results = (
    ridge_60_tuning_results
    .sort_values(
        [
            "feature_family",
            "MAE_kW",
        ]
    )
    .groupby(
        "feature_family",
        as_index=False,
    )
    .first()
)


# Restore the intended F0 -> F4 ordering.
family_order = [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]

ridge_60_best_results[
    "feature_family"
] = pd.Categorical(
    ridge_60_best_results["feature_family"],
    categories=family_order,
    ordered=True,
)

ridge_60_best_results = (
    ridge_60_best_results
    .sort_values("feature_family")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. Verify that F0 reproduces our previous single-case test
# ------------------------------------------------------------

f0_result = ridge_60_best_results.loc[
    ridge_60_best_results["feature_family"] == "F0"
].iloc[0]

assert np.isclose(
    f0_result["MAE_kW"],
    best_ridge_f0_60_mae,
    atol=1e-12,
    rtol=0.0,
)

assert (
    f0_result["alpha"]
    == best_ridge_f0_60_alpha
)

print(
    "\nF0 reproduction check: PASSED"
)


# ------------------------------------------------------------
# 6. Calculate incremental MAE changes
# ------------------------------------------------------------
#
# Positive values mean the new feature family reduced MAE.
#
# Examples:
#
#   F0 -> F1
#   F1 -> F2
#   F2 -> F3
#   F3 -> F4
# ------------------------------------------------------------

ridge_60_best_results[
    "previous_MAE_kW"
] = (
    ridge_60_best_results["MAE_kW"]
    .shift(1)
)

ridge_60_best_results[
    "incremental_MAE_improvement_percent"
] = (
    (
        ridge_60_best_results["previous_MAE_kW"]
        - ridge_60_best_results["MAE_kW"]
    )
    / ridge_60_best_results["previous_MAE_kW"]
    * 100.0
)


# ------------------------------------------------------------
# 7. Show the primary validation ablation table
# ------------------------------------------------------------

print(
    "\n60-minute Ridge validation ablation:"
)

display(
    ridge_60_best_results[
        [
            "feature_family",
            "alpha",
            "MAE_kW",
            "RMSE_kW",
            "nMAE_percent",
            "nRMSE_percent",
            "R2",
            "incremental_MAE_improvement_percent",
        ]
    ]
)


# ------------------------------------------------------------
# 8. Explicitly report the scientifically important F3 -> F4
#    validation comparison
# ------------------------------------------------------------

f3_mae = (
    ridge_60_best_results.loc[
        ridge_60_best_results["feature_family"] == "F3",
        "MAE_kW",
    ]
    .iloc[0]
)

f4_mae = (
    ridge_60_best_results.loc[
        ridge_60_best_results["feature_family"] == "F4",
        "MAE_kW",
    ]
    .iloc[0]
)

f3_to_f4_improvement = (
    (f3_mae - f4_mae)
    / f3_mae
    * 100.0
)

print("\nPrimary F3 -> F4 validation comparison:")
print(f"  F3 MAE: {f3_mae:.6f} kW")
print(f"  F4 MAE: {f4_mae:.6f} kW")
print(
    f"  Relative MAE change: "
    f"{f3_to_f4_improvement:.4f}%"
)


print(
    "\n60-minute Ridge F0-F4 validation ablation: PASSED"
)


Tuning Ridge: 60-minute / F0
  Train shape:      (87252, 118)
  Validation shape: (17568, 118)

Tuning Ridge: 60-minute / F1
  Train shape:      (87252, 133)
  Validation shape: (17568, 133)

Tuning Ridge: 60-minute / F2
  Train shape:      (87252, 138)
  Validation shape: (17568, 138)

Tuning Ridge: 60-minute / F3
  Train shape:      (87252, 147)
  Validation shape: (17568, 147)

Tuning Ridge: 60-minute / F4
  Train shape:      (87252, 244)
  Validation shape: (17568, 244)

F0 reproduction check: PASSED

60-minute Ridge validation ablation:


,feature_family,alpha,MAE_kW,RMSE_kW,nMAE_percent,nRMSE_percent,R2,incremental_MAE_improvement_percent
0,F0,100.00,0.512713,0.863786,5.127134,8.637857,0.801459,NaN
1,F1,100.00,0.523861,0.826821,5.238609,8.268214,0.818088,-2.174205
2,F2,100.00,0.512424,0.812477,5.124236,8.124769,0.824345,2.183260
3,F3,0.01,0.511464,0.807592,5.114637,8.075917,0.826451,0.187320
4,F4,0.01,0.509421,0.802295,5.094205,8.022955,0.828720,0.399480



Primary F3 -> F4 validation comparison:
  F3 MAE: 0.511464 kW
  F4 MAE: 0.509421 kW
  Relative MAE change: 0.3995%

60-minute Ridge F0-F4 validation ablation: PASSED


In [20]:
# ============================================================
# Step 19: Validate Histogram Gradient Boosting on 60-min F0
# ============================================================

from itertools import product
from sklearn.ensemble import HistGradientBoostingRegressor


# ------------------------------------------------------------
# 1. Select the primary horizon and F0 feature family
# ------------------------------------------------------------

HORIZON = 60
FEATURE_FAMILY = "F0"

bundle = modelling_bundles[HORIZON]

X = bundle["X"][FEATURE_FAMILY]
y = bundle["y"]

train_mask = bundle["train_mask"]
validation_mask = bundle["validation_mask"]

X_train = X.loc[train_mask]
y_train = y.loc[train_mask]

X_validation = X.loc[validation_mask]
y_validation = y.loc[validation_mask]


print("60-minute F0 HGBR experiment:")
print(f"  X_train:       {X_train.shape}")
print(f"  y_train:       {y_train.shape}")
print(f"  X_validation:  {X_validation.shape}")
print(f"  y_validation:  {y_validation.shape}")


# ------------------------------------------------------------
# 2. Freeze the compact HGBR hyperparameter grid
# ------------------------------------------------------------
#
# These values are defined BEFORE examining any HGBR result.
#
# The grid is intentionally modest:
#
#   learning_rate      2 values
#   max_leaf_nodes     2 values
#   l2_regularization  2 values
#
# Total:
#
#   2 x 2 x 2 = 8 configurations
#
# max_iter is fixed at 300.
#
# IMPORTANT:
# early_stopping=False prevents scikit-learn from internally
# creating its own validation subset, which could otherwise
# complicate the chronological validation protocol.
# ------------------------------------------------------------

HGBR_LEARNING_RATES = [
    0.05,
    0.10,
]

HGBR_MAX_LEAF_NODES = [
    15,
    31,
]

HGBR_L2_REGULARIZATION = [
    0.0,
    1.0,
]

HGBR_MAX_ITER = 300


HGBR_PARAMETER_GRID = list(
    product(
        HGBR_LEARNING_RATES,
        HGBR_MAX_LEAF_NODES,
        HGBR_L2_REGULARIZATION,
    )
)

assert len(HGBR_PARAMETER_GRID) == 8

print(
    "\nNumber of HGBR candidate configurations:",
    len(HGBR_PARAMETER_GRID),
)


# ------------------------------------------------------------
# 3. Tune using chronological validation MAE
# ------------------------------------------------------------
#
# Histogram Gradient Boosting does not require feature
# standardization, so the cleaned/model-ready features are
# supplied directly.
# ------------------------------------------------------------

hgbr_f0_60_rows = []


for (
    learning_rate,
    max_leaf_nodes,
    l2_regularization,
) in HGBR_PARAMETER_GRID:

    model = HistGradientBoostingRegressor(
        loss="squared_error",
        learning_rate=learning_rate,
        max_iter=HGBR_MAX_ITER,
        max_leaf_nodes=max_leaf_nodes,
        l2_regularization=l2_regularization,
        early_stopping=False,
        random_state=42,
    )

    model.fit(
        X_train,
        y_train,
    )

    validation_predictions = model.predict(
        X_validation
    )

    validation_metrics = regression_metrics(
        y_validation,
        validation_predictions,
    )

    hgbr_f0_60_rows.append(
        {
            "learning_rate": learning_rate,
            "max_leaf_nodes": max_leaf_nodes,
            "l2_regularization": l2_regularization,
            "max_iter": HGBR_MAX_ITER,
            **validation_metrics,
        }
    )


# ------------------------------------------------------------
# 4. Assemble and rank the validation results
# ------------------------------------------------------------

hgbr_f0_60_validation = pd.DataFrame(
    hgbr_f0_60_rows
).sort_values(
    "MAE_kW"
).reset_index(drop=True)


print("\n60-minute F0 HGBR validation results:")

display(hgbr_f0_60_validation)


# ------------------------------------------------------------
# 5. Select the minimum-validation-MAE configuration
# ------------------------------------------------------------

best_hgbr_f0_60 = hgbr_f0_60_validation.iloc[0]

best_hgbr_f0_60_mae = best_hgbr_f0_60["MAE_kW"]


print("\nBest HGBR configuration:")

print(
    "  learning_rate:",
    best_hgbr_f0_60["learning_rate"],
)

print(
    "  max_leaf_nodes:",
    int(best_hgbr_f0_60["max_leaf_nodes"]),
)

print(
    "  l2_regularization:",
    best_hgbr_f0_60["l2_regularization"],
)

print(
    "  max_iter:",
    int(best_hgbr_f0_60["max_iter"]),
)

print(
    "  validation MAE:",
    best_hgbr_f0_60_mae,
    "kW",
)


# ------------------------------------------------------------
# 6. Compare with the same validation persistence benchmark
# ------------------------------------------------------------

print(
    "\nValidation persistence MAE:",
    validation_persistence_mae,
    "kW",
)

print(
    "HGBR F0 validation MAE:",
    best_hgbr_f0_60_mae,
    "kW",
)


print("\n60-minute F0 HGBR tuning: PASSED")

60-minute F0 HGBR experiment:
  X_train:       (87252, 118)
  y_train:       (87252,)
  X_validation:  (17568, 118)
  y_validation:  (17568,)

Number of HGBR candidate configurations: 8

60-minute F0 HGBR validation results:


,learning_rate,max_leaf_nodes,l2_regularization,max_iter,MAE_kW,RMSE_kW,nMAE_percent,nRMSE_percent,R2
0,0.05,15,0.0,300,0.462910,0.838344,4.629102,8.383442,0.812982
1,0.05,31,0.0,300,0.462942,0.841652,4.629424,8.416519,0.811504
2,0.05,31,1.0,300,0.465358,0.838765,4.653577,8.387654,0.812794
3,0.05,15,1.0,300,0.467596,0.843028,4.675962,8.430280,0.810887
4,0.10,31,0.0,300,0.479113,0.860510,4.791134,8.605103,0.802962
5,0.10,31,1.0,300,0.480191,0.860720,4.801907,8.607196,0.802866
6,0.10,15,0.0,300,0.480458,0.860484,4.804576,8.604838,0.802974
7,0.10,15,1.0,300,0.486818,0.875319,4.868179,8.753190,0.796122



Best HGBR configuration:
  learning_rate: 0.05
  max_leaf_nodes: 15
  l2_regularization: 0.0
  max_iter: 300
  validation MAE: 0.4629102099361732 kW

Validation persistence MAE: 0.567365010004198 kW
HGBR F0 validation MAE: 0.4629102099361732 kW

60-minute F0 HGBR tuning: PASSED


In [21]:
# ============================================================
# Step 20: HGBR F0-F4 ablation for the 60-minute horizon
# ============================================================

HORIZON = 60

bundle = modelling_bundles[HORIZON]

y = bundle["y"]

train_mask = bundle["train_mask"]
validation_mask = bundle["validation_mask"]

y_train = y.loc[train_mask]
y_validation = y.loc[validation_mask]


# ------------------------------------------------------------
# 1. Tune HGBR separately for each frozen feature family
# ------------------------------------------------------------

hgbr_60_tuning_rows = []


for feature_family in ["F0", "F1", "F2", "F3", "F4"]:

    X = bundle["X"][feature_family]

    X_train = X.loc[train_mask]
    X_validation = X.loc[validation_mask]

    print(
        f"\nTuning HGBR: "
        f"{HORIZON}-minute / {feature_family}"
    )

    print(
        f"  Train shape:      {X_train.shape}"
    )

    print(
        f"  Validation shape: {X_validation.shape}"
    )


    # --------------------------------------------------------
    # Apply the same frozen 8-configuration grid to every
    # feature family.
    # --------------------------------------------------------

    for (
        learning_rate,
        max_leaf_nodes,
        l2_regularization,
    ) in HGBR_PARAMETER_GRID:

        model = HistGradientBoostingRegressor(
            loss="squared_error",
            learning_rate=learning_rate,
            max_iter=HGBR_MAX_ITER,
            max_leaf_nodes=max_leaf_nodes,
            l2_regularization=l2_regularization,
            early_stopping=False,
            random_state=42,
        )

        model.fit(
            X_train,
            y_train,
        )

        validation_predictions = model.predict(
            X_validation
        )

        metrics = regression_metrics(
            y_validation,
            validation_predictions,
        )

        hgbr_60_tuning_rows.append(
            {
                "horizon_minutes": HORIZON,
                "feature_family": feature_family,
                "learning_rate": learning_rate,
                "max_leaf_nodes": max_leaf_nodes,
                "l2_regularization": l2_regularization,
                "max_iter": HGBR_MAX_ITER,
                **metrics,
            }
        )


# ------------------------------------------------------------
# 2. Assemble all validation results
# ------------------------------------------------------------

hgbr_60_tuning_results = pd.DataFrame(
    hgbr_60_tuning_rows
)


# ------------------------------------------------------------
# 3. Select the lowest-validation-MAE configuration for
#    each feature family
# ------------------------------------------------------------

hgbr_60_best_results = (
    hgbr_60_tuning_results
    .sort_values(
        [
            "feature_family",
            "MAE_kW",
        ]
    )
    .groupby(
        "feature_family",
        as_index=False,
    )
    .first()
)


# Restore F0 -> F4 ordering.
family_order = [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]

hgbr_60_best_results[
    "feature_family"
] = pd.Categorical(
    hgbr_60_best_results["feature_family"],
    categories=family_order,
    ordered=True,
)

hgbr_60_best_results = (
    hgbr_60_best_results
    .sort_values("feature_family")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 4. Confirm F0 reproduces Cell 32
# ------------------------------------------------------------

f0_hgbr_result = hgbr_60_best_results.loc[
    hgbr_60_best_results["feature_family"] == "F0"
].iloc[0]

assert np.isclose(
    f0_hgbr_result["MAE_kW"],
    best_hgbr_f0_60_mae,
    atol=1e-12,
    rtol=0.0,
)

assert np.isclose(
    f0_hgbr_result["learning_rate"],
    best_hgbr_f0_60["learning_rate"],
)

assert int(
    f0_hgbr_result["max_leaf_nodes"]
) == int(
    best_hgbr_f0_60["max_leaf_nodes"]
)

assert np.isclose(
    f0_hgbr_result["l2_regularization"],
    best_hgbr_f0_60["l2_regularization"],
)

print("\nF0 reproduction check: PASSED")


# ------------------------------------------------------------
# 5. Calculate incremental MAE changes
# ------------------------------------------------------------
#
# Positive value:
#     new feature family reduced MAE
#
# Negative value:
#     new feature family increased MAE
# ------------------------------------------------------------

hgbr_60_best_results[
    "previous_MAE_kW"
] = (
    hgbr_60_best_results["MAE_kW"]
    .shift(1)
)

hgbr_60_best_results[
    "incremental_MAE_improvement_percent"
] = (
    (
        hgbr_60_best_results["previous_MAE_kW"]
        - hgbr_60_best_results["MAE_kW"]
    )
    / hgbr_60_best_results["previous_MAE_kW"]
    * 100.0
)


# ------------------------------------------------------------
# 6. Display the 60-minute HGBR validation ablation
# ------------------------------------------------------------

print(
    "\n60-minute HGBR validation ablation:"
)

display(
    hgbr_60_best_results[
        [
            "feature_family",
            "learning_rate",
            "max_leaf_nodes",
            "l2_regularization",
            "MAE_kW",
            "RMSE_kW",
            "nMAE_percent",
            "nRMSE_percent",
            "R2",
            "incremental_MAE_improvement_percent",
        ]
    ]
)


# ------------------------------------------------------------
# 7. Explicit F3 -> F4 comparison
# ------------------------------------------------------------

f3_hgbr_mae = (
    hgbr_60_best_results.loc[
        hgbr_60_best_results["feature_family"] == "F3",
        "MAE_kW",
    ]
    .iloc[0]
)

f4_hgbr_mae = (
    hgbr_60_best_results.loc[
        hgbr_60_best_results["feature_family"] == "F4",
        "MAE_kW",
    ]
    .iloc[0]
)

f3_to_f4_hgbr_improvement = (
    (f3_hgbr_mae - f4_hgbr_mae)
    / f3_hgbr_mae
    * 100.0
)

print("\nPrimary F3 -> F4 HGBR validation comparison:")
print(f"  F3 MAE: {f3_hgbr_mae:.6f} kW")
print(f"  F4 MAE: {f4_hgbr_mae:.6f} kW")
print(
    f"  Relative MAE change: "
    f"{f3_to_f4_hgbr_improvement:.4f}%"
)


print(
    "\n60-minute HGBR F0-F4 validation ablation: PASSED"
)


Tuning HGBR: 60-minute / F0
  Train shape:      (87252, 118)
  Validation shape: (17568, 118)

Tuning HGBR: 60-minute / F1
  Train shape:      (87252, 133)
  Validation shape: (17568, 133)

Tuning HGBR: 60-minute / F2
  Train shape:      (87252, 138)
  Validation shape: (17568, 138)

Tuning HGBR: 60-minute / F3
  Train shape:      (87252, 147)
  Validation shape: (17568, 147)

Tuning HGBR: 60-minute / F4
  Train shape:      (87252, 244)
  Validation shape: (17568, 244)

F0 reproduction check: PASSED

60-minute HGBR validation ablation:


,feature_family,learning_rate,max_leaf_nodes,l2_regularization,MAE_kW,RMSE_kW,nMAE_percent,nRMSE_percent,R2,incremental_MAE_improvement_percent
0,F0,0.05,15,0.0,0.462910,0.838344,4.629102,8.383442,0.812982,NaN
1,F1,0.05,15,1.0,0.363070,0.718482,3.630705,7.184819,0.862637,21.567843
2,F2,0.05,15,0.0,0.368494,0.728825,3.684943,7.288255,0.858654,-1.493868
3,F3,0.05,15,0.0,0.383858,0.736462,3.838584,7.364617,0.855676,-4.169440
4,F4,0.05,15,1.0,0.396150,0.746822,3.961498,7.468216,0.851587,-3.202072



Primary F3 -> F4 HGBR validation comparison:
  F3 MAE: 0.383858 kW
  F4 MAE: 0.396150 kW
  Relative MAE change: -3.2021%

60-minute HGBR F0-F4 validation ablation: PASSED


In [22]:
# ============================================================
# Step 21: Ridge F0-F4 validation ablation for 15 and 30 min
# ============================================================

# ------------------------------------------------------------
# 1. Supporting forecast horizons
# ------------------------------------------------------------

SUPPORTING_HORIZONS = [
    15,
    30,
]


# ------------------------------------------------------------
# 2. Tune Ridge for every F0-F4 family at each supporting
#    horizon using the SAME frozen alpha grid.
# ------------------------------------------------------------

ridge_supporting_tuning_rows = []


for horizon_minutes in SUPPORTING_HORIZONS:

    bundle = modelling_bundles[horizon_minutes]

    y = bundle["y"]

    train_mask = bundle["train_mask"]
    validation_mask = bundle["validation_mask"]

    y_train = y.loc[train_mask]
    y_validation = y.loc[validation_mask]

    print(
        f"\n========================================"
    )
    print(
        f"Ridge horizon: {horizon_minutes} minutes"
    )
    print(
        f"========================================"
    )


    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        X = bundle["X"][feature_family]

        X_train = X.loc[train_mask]
        X_validation = X.loc[validation_mask]

        print(
            f"\nTuning Ridge: "
            f"{horizon_minutes}-minute / {feature_family}"
        )

        print(
            f"  Train shape:      {X_train.shape}"
        )

        print(
            f"  Validation shape: {X_validation.shape}"
        )


        for alpha in RIDGE_ALPHA_GRID:

            model = Pipeline(
                steps=[
                    (
                        "scaler",
                        StandardScaler(),
                    ),
                    (
                        "ridge",
                        Ridge(alpha=alpha),
                    ),
                ]
            )

            model.fit(
                X_train,
                y_train,
            )

            validation_predictions = model.predict(
                X_validation
            )

            metrics = regression_metrics(
                y_validation,
                validation_predictions,
            )

            ridge_supporting_tuning_rows.append(
                {
                    "horizon_minutes": horizon_minutes,
                    "feature_family": feature_family,
                    "alpha": alpha,
                    **metrics,
                }
            )


# ------------------------------------------------------------
# 3. Assemble complete supporting-horizon tuning results
# ------------------------------------------------------------

ridge_supporting_tuning_results = pd.DataFrame(
    ridge_supporting_tuning_rows
)


# ------------------------------------------------------------
# 4. Select minimum-validation-MAE alpha for every:
#
#       horizon × feature family
# ------------------------------------------------------------

ridge_supporting_best_results = (
    ridge_supporting_tuning_results
    .sort_values(
        [
            "horizon_minutes",
            "feature_family",
            "MAE_kW",
        ]
    )
    .groupby(
        [
            "horizon_minutes",
            "feature_family",
        ],
        as_index=False,
    )
    .first()
)


# ------------------------------------------------------------
# 5. Restore explicit F0 -> F4 ordering
# ------------------------------------------------------------

ridge_supporting_best_results[
    "feature_family"
] = pd.Categorical(
    ridge_supporting_best_results["feature_family"],
    categories=[
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ],
    ordered=True,
)

ridge_supporting_best_results = (
    ridge_supporting_best_results
    .sort_values(
        [
            "horizon_minutes",
            "feature_family",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. Calculate incremental MAE change independently within
#    each forecast horizon.
# ------------------------------------------------------------
#
# Positive:
#     newly added feature family reduced MAE.
#
# Negative:
#     newly added feature family increased MAE.
# ------------------------------------------------------------

ridge_supporting_best_results[
    "previous_MAE_kW"
] = (
    ridge_supporting_best_results
    .groupby(
        "horizon_minutes",
        observed=False,
    )["MAE_kW"]
    .shift(1)
)

ridge_supporting_best_results[
    "incremental_MAE_improvement_percent"
] = (
    (
        ridge_supporting_best_results["previous_MAE_kW"]
        - ridge_supporting_best_results["MAE_kW"]
    )
    / ridge_supporting_best_results["previous_MAE_kW"]
    * 100.0
)


# ------------------------------------------------------------
# 7. Display one ablation table per supporting horizon
# ------------------------------------------------------------

for horizon_minutes in SUPPORTING_HORIZONS:

    horizon_results = (
        ridge_supporting_best_results.loc[
            ridge_supporting_best_results[
                "horizon_minutes"
            ].eq(horizon_minutes)
        ]
        .copy()
    )

    print(
        f"\n{horizon_minutes}-minute Ridge "
        "validation ablation:"
    )

    display(
        horizon_results[
            [
                "feature_family",
                "alpha",
                "MAE_kW",
                "RMSE_kW",
                "nMAE_percent",
                "nRMSE_percent",
                "R2",
                "incremental_MAE_improvement_percent",
            ]
        ]
    )


# ------------------------------------------------------------
# 8. Explicit F3 -> F4 summary across all three horizons
# ------------------------------------------------------------

ridge_all_best_results = pd.concat(
    [
        ridge_supporting_best_results,
        ridge_60_best_results.assign(
            horizon_minutes=60
        ),
    ],
    ignore_index=True,
)


ridge_f3_f4_summary_rows = []


for horizon_minutes in [
    15,
    30,
    60,
]:

    horizon_results = ridge_all_best_results.loc[
        ridge_all_best_results[
            "horizon_minutes"
        ].eq(horizon_minutes)
    ]

    f3_mae = (
        horizon_results.loc[
            horizon_results[
                "feature_family"
            ].eq("F3"),
            "MAE_kW",
        ]
        .iloc[0]
    )

    f4_mae = (
        horizon_results.loc[
            horizon_results[
                "feature_family"
            ].eq("F4"),
            "MAE_kW",
        ]
        .iloc[0]
    )

    relative_change = (
        (f3_mae - f4_mae)
        / f3_mae
        * 100.0
    )

    ridge_f3_f4_summary_rows.append(
        {
            "horizon_minutes": horizon_minutes,
            "F3_MAE_kW": f3_mae,
            "F4_MAE_kW": f4_mae,
            "F3_to_F4_MAE_improvement_percent": (
                relative_change
            ),
        }
    )


ridge_f3_f4_summary = pd.DataFrame(
    ridge_f3_f4_summary_rows
)


print(
    "\nRidge F3 -> F4 validation summary "
    "across forecast horizons:"
)

display(ridge_f3_f4_summary)


print(
    "\nSupporting-horizon Ridge ablation: PASSED"
)


Ridge horizon: 15 minutes

Tuning Ridge: 15-minute / F0
  Train shape:      (87261, 118)
  Validation shape: (17568, 118)

Tuning Ridge: 15-minute / F1
  Train shape:      (87261, 133)
  Validation shape: (17568, 133)

Tuning Ridge: 15-minute / F2
  Train shape:      (87261, 138)
  Validation shape: (17568, 138)

Tuning Ridge: 15-minute / F3
  Train shape:      (87261, 147)
  Validation shape: (17568, 147)

Tuning Ridge: 15-minute / F4
  Train shape:      (87261, 244)
  Validation shape: (17568, 244)

Ridge horizon: 30 minutes

Tuning Ridge: 30-minute / F0
  Train shape:      (87258, 118)
  Validation shape: (17568, 118)

Tuning Ridge: 30-minute / F1
  Train shape:      (87258, 133)
  Validation shape: (17568, 133)

Tuning Ridge: 30-minute / F2
  Train shape:      (87258, 138)
  Validation shape: (17568, 138)

Tuning Ridge: 30-minute / F3
  Train shape:      (87258, 147)
  Validation shape: (17568, 147)

Tuning Ridge: 30-minute / F4
  Train shape:      (87258, 244)
  Validation shape:

,feature_family,alpha,MAE_kW,RMSE_kW,nMAE_percent,nRMSE_percent,R2,incremental_MAE_improvement_percent
0,F0,100.00,0.300900,0.610773,3.009003,6.107731,0.900735,NaN
1,F1,100.00,0.307891,0.601782,3.078908,6.017818,0.903636,-2.323195
2,F2,100.00,0.304236,0.598573,3.042355,5.985730,0.904661,1.187200
3,F3,0.01,0.307647,0.598610,3.076465,5.986096,0.904649,-1.121172
4,F4,0.01,0.306413,0.596949,3.064127,5.969486,0.905177,0.401053



30-minute Ridge validation ablation:


,feature_family,alpha,MAE_kW,RMSE_kW,nMAE_percent,nRMSE_percent,R2,incremental_MAE_improvement_percent
5,F0,100.00,0.382907,0.708195,3.829074,7.081951,0.866542,NaN
6,F1,100.00,0.393408,0.691751,3.934083,6.917511,0.872668,-2.742405
7,F2,100.00,0.386216,0.684763,3.862164,6.847634,0.875228,1.828087
8,F3,0.01,0.389081,0.684193,3.890811,6.841925,0.875435,-0.741720
9,F4,0.01,0.388180,0.681316,3.881804,6.813156,0.876481,0.231492



Ridge F3 -> F4 validation summary across forecast horizons:


,horizon_minutes,F3_MAE_kW,F4_MAE_kW,F3_to_F4_MAE_improvement_percent
0,15,0.307647,0.306413,0.401053
1,30,0.389081,0.388180,0.231492
2,60,0.511464,0.509421,0.399480



Supporting-horizon Ridge ablation: PASSED


In [23]:
# ============================================================
# Step 22: HGBR F0-F4 validation ablation for 15 and 30 min
# ============================================================

# ------------------------------------------------------------
# 1. Supporting forecast horizons
# ------------------------------------------------------------

SUPPORTING_HORIZONS = [
    15,
    30,
]


# ------------------------------------------------------------
# 2. Tune HGBR for every F0-F4 family at each supporting
#    horizon using the SAME frozen 8-configuration grid.
#
# Total model fits:
#
#   2 horizons
#   x 5 feature families
#   x 8 HGBR configurations
#   = 80 fits
#
# This may take noticeably longer than Ridge.
# ------------------------------------------------------------

hgbr_supporting_tuning_rows = []


for horizon_minutes in SUPPORTING_HORIZONS:

    bundle = modelling_bundles[horizon_minutes]

    y = bundle["y"]

    train_mask = bundle["train_mask"]
    validation_mask = bundle["validation_mask"]

    y_train = y.loc[train_mask]
    y_validation = y.loc[validation_mask]

    print("\n========================================")
    print(
        f"HGBR horizon: {horizon_minutes} minutes"
    )
    print("========================================")


    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        X = bundle["X"][feature_family]

        X_train = X.loc[train_mask]
        X_validation = X.loc[validation_mask]

        print(
            f"\nTuning HGBR: "
            f"{horizon_minutes}-minute / {feature_family}"
        )

        print(
            f"  Train shape:      {X_train.shape}"
        )

        print(
            f"  Validation shape: {X_validation.shape}"
        )


        # ----------------------------------------------------
        # Apply exactly the same frozen grid used for the
        # 60-minute HGBR experiment.
        # ----------------------------------------------------

        for (
            learning_rate,
            max_leaf_nodes,
            l2_regularization,
        ) in HGBR_PARAMETER_GRID:

            model = HistGradientBoostingRegressor(
                loss="squared_error",
                learning_rate=learning_rate,
                max_iter=HGBR_MAX_ITER,
                max_leaf_nodes=max_leaf_nodes,
                l2_regularization=l2_regularization,
                early_stopping=False,
                random_state=42,
            )

            model.fit(
                X_train,
                y_train,
            )

            validation_predictions = model.predict(
                X_validation
            )

            metrics = regression_metrics(
                y_validation,
                validation_predictions,
            )

            hgbr_supporting_tuning_rows.append(
                {
                    "horizon_minutes": horizon_minutes,
                    "feature_family": feature_family,
                    "learning_rate": learning_rate,
                    "max_leaf_nodes": max_leaf_nodes,
                    "l2_regularization": l2_regularization,
                    "max_iter": HGBR_MAX_ITER,
                    **metrics,
                }
            )


# ------------------------------------------------------------
# 3. Assemble the complete supporting-horizon tuning results
# ------------------------------------------------------------

hgbr_supporting_tuning_results = pd.DataFrame(
    hgbr_supporting_tuning_rows
)


# ------------------------------------------------------------
# 4. Select minimum-validation-MAE configuration for each:
#
#       horizon x feature family
# ------------------------------------------------------------

hgbr_supporting_best_results = (
    hgbr_supporting_tuning_results
    .sort_values(
        [
            "horizon_minutes",
            "feature_family",
            "MAE_kW",
        ]
    )
    .groupby(
        [
            "horizon_minutes",
            "feature_family",
        ],
        as_index=False,
    )
    .first()
)


# ------------------------------------------------------------
# 5. Restore explicit F0 -> F4 ordering
# ------------------------------------------------------------

hgbr_supporting_best_results[
    "feature_family"
] = pd.Categorical(
    hgbr_supporting_best_results["feature_family"],
    categories=[
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ],
    ordered=True,
)

hgbr_supporting_best_results = (
    hgbr_supporting_best_results
    .sort_values(
        [
            "horizon_minutes",
            "feature_family",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. Calculate incremental MAE change independently within
#    each forecast horizon.
#
# Positive:
#     added feature family reduced MAE.
#
# Negative:
#     added feature family increased MAE.
# ------------------------------------------------------------

hgbr_supporting_best_results[
    "previous_MAE_kW"
] = (
    hgbr_supporting_best_results
    .groupby(
        "horizon_minutes",
        observed=False,
    )["MAE_kW"]
    .shift(1)
)

hgbr_supporting_best_results[
    "incremental_MAE_improvement_percent"
] = (
    (
        hgbr_supporting_best_results["previous_MAE_kW"]
        - hgbr_supporting_best_results["MAE_kW"]
    )
    / hgbr_supporting_best_results["previous_MAE_kW"]
    * 100.0
)


# ------------------------------------------------------------
# 7. Display one validation-ablation table per horizon
# ------------------------------------------------------------

for horizon_minutes in SUPPORTING_HORIZONS:

    horizon_results = (
        hgbr_supporting_best_results.loc[
            hgbr_supporting_best_results[
                "horizon_minutes"
            ].eq(horizon_minutes)
        ]
        .copy()
    )

    print(
        f"\n{horizon_minutes}-minute HGBR "
        "validation ablation:"
    )

    display(
        horizon_results[
            [
                "feature_family",
                "learning_rate",
                "max_leaf_nodes",
                "l2_regularization",
                "MAE_kW",
                "RMSE_kW",
                "nMAE_percent",
                "nRMSE_percent",
                "R2",
                "incremental_MAE_improvement_percent",
            ]
        ]
    )


# ------------------------------------------------------------
# 8. Combine supporting horizons with the already-completed
#    60-minute HGBR results
# ------------------------------------------------------------

hgbr_all_best_results = pd.concat(
    [
        hgbr_supporting_best_results,
        hgbr_60_best_results.assign(
            horizon_minutes=60
        ),
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# 9. Build explicit F3 -> F4 summary across all horizons
# ------------------------------------------------------------

hgbr_f3_f4_summary_rows = []


for horizon_minutes in [
    15,
    30,
    60,
]:

    horizon_results = hgbr_all_best_results.loc[
        hgbr_all_best_results[
            "horizon_minutes"
        ].eq(horizon_minutes)
    ]

    f3_mae = (
        horizon_results.loc[
            horizon_results[
                "feature_family"
            ].eq("F3"),
            "MAE_kW",
        ]
        .iloc[0]
    )

    f4_mae = (
        horizon_results.loc[
            horizon_results[
                "feature_family"
            ].eq("F4"),
            "MAE_kW",
        ]
        .iloc[0]
    )

    relative_change = (
        (f3_mae - f4_mae)
        / f3_mae
        * 100.0
    )

    hgbr_f3_f4_summary_rows.append(
        {
            "horizon_minutes": horizon_minutes,
            "F3_MAE_kW": f3_mae,
            "F4_MAE_kW": f4_mae,
            "F3_to_F4_MAE_improvement_percent": (
                relative_change
            ),
        }
    )


hgbr_f3_f4_summary = pd.DataFrame(
    hgbr_f3_f4_summary_rows
)


print(
    "\nHGBR F3 -> F4 validation summary "
    "across forecast horizons:"
)

display(hgbr_f3_f4_summary)


print(
    "\nSupporting-horizon HGBR ablation: PASSED"
)


HGBR horizon: 15 minutes

Tuning HGBR: 15-minute / F0
  Train shape:      (87261, 118)
  Validation shape: (17568, 118)

Tuning HGBR: 15-minute / F1
  Train shape:      (87261, 133)
  Validation shape: (17568, 133)

Tuning HGBR: 15-minute / F2
  Train shape:      (87261, 138)
  Validation shape: (17568, 138)

Tuning HGBR: 15-minute / F3
  Train shape:      (87261, 147)
  Validation shape: (17568, 147)

Tuning HGBR: 15-minute / F4
  Train shape:      (87261, 244)
  Validation shape: (17568, 244)

HGBR horizon: 30 minutes

Tuning HGBR: 30-minute / F0
  Train shape:      (87258, 118)
  Validation shape: (17568, 118)

Tuning HGBR: 30-minute / F1
  Train shape:      (87258, 133)
  Validation shape: (17568, 133)

Tuning HGBR: 30-minute / F2
  Train shape:      (87258, 138)
  Validation shape: (17568, 138)

Tuning HGBR: 30-minute / F3
  Train shape:      (87258, 147)
  Validation shape: (17568, 147)

Tuning HGBR: 30-minute / F4
  Train shape:      (87258, 244)
  Validation shape: (17568, 244

,feature_family,learning_rate,max_leaf_nodes,l2_regularization,MAE_kW,RMSE_kW,nMAE_percent,nRMSE_percent,R2,incremental_MAE_improvement_percent
0,F0,0.05,15,1.0,0.293510,0.620730,2.935098,6.207299,0.897472,NaN
1,F1,0.05,15,1.0,0.268899,0.591675,2.688993,5.916749,0.906845,8.384907
2,F2,0.05,15,1.0,0.274653,0.598131,2.746531,5.981306,0.904802,-2.139763
3,F3,0.05,15,1.0,0.278706,0.597951,2.787057,5.979508,0.904859,-1.475529
4,F4,0.05,15,1.0,0.276521,0.596176,2.765205,5.961761,0.905423,0.784054



30-minute HGBR validation ablation:


,feature_family,learning_rate,max_leaf_nodes,l2_regularization,MAE_kW,RMSE_kW,nMAE_percent,nRMSE_percent,R2,incremental_MAE_improvement_percent
5,F0,0.05,15,1.0,0.357288,0.707711,3.572880,7.077115,0.866725,NaN
6,F1,0.05,15,1.0,0.316789,0.647159,3.167894,6.471589,0.888555,11.334993
7,F2,0.05,15,0.0,0.319658,0.648053,3.196581,6.480527,0.888247,-0.905555
8,F3,0.05,15,1.0,0.332030,0.658878,3.320298,6.588777,0.884483,-3.870297
9,F4,0.05,15,0.0,0.326199,0.655069,3.261992,6.550687,0.885814,1.756060



HGBR F3 -> F4 validation summary across forecast horizons:


,horizon_minutes,F3_MAE_kW,F4_MAE_kW,F3_to_F4_MAE_improvement_percent
0,15,0.278706,0.276521,0.784054
1,30,0.332030,0.326199,1.756060
2,60,0.383858,0.396150,-3.202072



Supporting-horizon HGBR ablation: PASSED


In [24]:
# ============================================================
# Step 23: Freeze all validation-selected configurations
# ============================================================

# ------------------------------------------------------------
# 1. Standardize the selected Ridge configurations
# ------------------------------------------------------------

ridge_selected = (
    ridge_all_best_results[
        [
            "horizon_minutes",
            "feature_family",
            "alpha",
            "MAE_kW",
            "RMSE_kW",
            "nMAE_percent",
            "nRMSE_percent",
            "R2",
        ]
    ]
    .copy()
)

ridge_selected["model"] = "Ridge"


# ------------------------------------------------------------
# 2. Standardize the selected HGBR configurations
# ------------------------------------------------------------

hgbr_selected = (
    hgbr_all_best_results[
        [
            "horizon_minutes",
            "feature_family",
            "learning_rate",
            "max_leaf_nodes",
            "l2_regularization",
            "max_iter",
            "MAE_kW",
            "RMSE_kW",
            "nMAE_percent",
            "nRMSE_percent",
            "R2",
        ]
    ]
    .copy()
)

hgbr_selected["model"] = "HGBR"


# ------------------------------------------------------------
# 3. Validate that every required experiment is represented
# ------------------------------------------------------------
#
# Required:
#
#   3 horizons
#   x 5 feature families
#   = 15 selected configurations per model
# ------------------------------------------------------------

EXPECTED_HORIZONS = {
    15,
    30,
    60,
}

EXPECTED_FAMILIES = {
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
}


assert len(ridge_selected) == 15
assert len(hgbr_selected) == 15


for selected_table, model_name in [
    (ridge_selected, "Ridge"),
    (hgbr_selected, "HGBR"),
]:

    assert set(
        selected_table["horizon_minutes"]
    ) == EXPECTED_HORIZONS

    assert set(
        selected_table["feature_family"].astype(str)
    ) == EXPECTED_FAMILIES

    combination_count = (
        selected_table[
            [
                "horizon_minutes",
                "feature_family",
            ]
        ]
        .drop_duplicates()
        .shape[0]
    )

    assert combination_count == 15, (
        f"{model_name}: expected 15 unique "
        "horizon/family combinations."
    )


print("Selected-configuration coverage check: PASSED")


# ------------------------------------------------------------
# 4. Build immutable-style configuration registry
# ------------------------------------------------------------
#
# This dictionary will be used later for final model fitting.
#
# Keys:
#
#   ("Ridge", horizon, family)
#   ("HGBR", horizon, family)
#
# Values contain ONLY the hyperparameters selected from
# validation performance.
# ------------------------------------------------------------

selected_model_configs = {}


for _, row in ridge_selected.iterrows():

    key = (
        "Ridge",
        int(row["horizon_minutes"]),
        str(row["feature_family"]),
    )

    selected_model_configs[key] = {
        "alpha": float(row["alpha"]),
    }


for _, row in hgbr_selected.iterrows():

    key = (
        "HGBR",
        int(row["horizon_minutes"]),
        str(row["feature_family"]),
    )

    selected_model_configs[key] = {
        "learning_rate": float(
            row["learning_rate"]
        ),
        "max_leaf_nodes": int(
            row["max_leaf_nodes"]
        ),
        "l2_regularization": float(
            row["l2_regularization"]
        ),
        "max_iter": int(
            row["max_iter"]
        ),
    }


assert len(selected_model_configs) == 30

print(
    "Frozen configuration registry:",
    len(selected_model_configs),
    "model/horizon/feature combinations",
)


# ------------------------------------------------------------
# 5. Build a compact human-readable freeze table
# ------------------------------------------------------------

freeze_rows = []


for horizon_minutes in [
    15,
    30,
    60,
]:

    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        # ----------------------------------------------------
        # Ridge
        # ----------------------------------------------------

        ridge_row = ridge_selected.loc[
            (
                ridge_selected[
                    "horizon_minutes"
                ].eq(horizon_minutes)
            )
            & (
                ridge_selected[
                    "feature_family"
                ].astype(str).eq(
                    feature_family
                )
            )
        ].iloc[0]

        freeze_rows.append(
            {
                "model": "Ridge",
                "horizon_minutes": horizon_minutes,
                "feature_family": feature_family,
                "selected_parameters": (
                    f"alpha={float(ridge_row['alpha']):g}"
                ),
                "validation_MAE_kW": (
                    ridge_row["MAE_kW"]
                ),
            }
        )


        # ----------------------------------------------------
        # HGBR
        # ----------------------------------------------------

        hgbr_row = hgbr_selected.loc[
            (
                hgbr_selected[
                    "horizon_minutes"
                ].eq(horizon_minutes)
            )
            & (
                hgbr_selected[
                    "feature_family"
                ].astype(str).eq(
                    feature_family
                )
            )
        ].iloc[0]

        freeze_rows.append(
            {
                "model": "HGBR",
                "horizon_minutes": horizon_minutes,
                "feature_family": feature_family,
                "selected_parameters": (
                    "learning_rate="
                    f"{float(hgbr_row['learning_rate']):g}, "
                    "max_leaf_nodes="
                    f"{int(hgbr_row['max_leaf_nodes'])}, "
                    "l2_regularization="
                    f"{float(hgbr_row['l2_regularization']):g}, "
                    "max_iter="
                    f"{int(hgbr_row['max_iter'])}"
                ),
                "validation_MAE_kW": (
                    hgbr_row["MAE_kW"]
                ),
            }
        )


frozen_selection_table = pd.DataFrame(
    freeze_rows
)


# ------------------------------------------------------------
# 6. Sort for easy auditing
# ------------------------------------------------------------

model_order = pd.CategoricalDtype(
    categories=[
        "Ridge",
        "HGBR",
    ],
    ordered=True,
)

family_order_dtype = pd.CategoricalDtype(
    categories=[
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ],
    ordered=True,
)

frozen_selection_table["model"] = (
    frozen_selection_table["model"]
    .astype(model_order)
)

frozen_selection_table["feature_family"] = (
    frozen_selection_table["feature_family"]
    .astype(family_order_dtype)
)

frozen_selection_table = (
    frozen_selection_table
    .sort_values(
        [
            "horizon_minutes",
            "model",
            "feature_family",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 7. Final checks
# ------------------------------------------------------------

assert len(frozen_selection_table) == 30

assert (
    frozen_selection_table[
        "validation_MAE_kW"
    ]
    .notna()
    .all()
)

print("\nValidation-selected configurations are now frozen.")

display(frozen_selection_table)


# ------------------------------------------------------------
# 8. Explicit primary-horizon freeze
# ------------------------------------------------------------

print("\nPrimary 60-minute frozen configurations:")

display(
    frozen_selection_table.loc[
        frozen_selection_table[
            "horizon_minutes"
        ].eq(60)
    ]
)


print(
    "\nPre-test model-selection freeze: PASSED"
)

Selected-configuration coverage check: PASSED
Frozen configuration registry: 30 model/horizon/feature combinations

Validation-selected configurations are now frozen.


,model,horizon_minutes,feature_family,selected_parameters,validation_MAE_kW
0,Ridge,15,F0,alpha=100,0.300900
1,Ridge,15,F1,alpha=100,0.307891
2,Ridge,15,F2,alpha=100,0.304236
3,Ridge,15,F3,alpha=0.01,0.307647
4,Ridge,15,F4,alpha=0.01,0.306413
5,HGBR,15,F0,"learning_rate=0.05, max_leaf_nodes=15, l2_regu...",0.293510
6,HGBR,15,F1,"learning_rate=0.05, max_leaf_nodes=15, l2_regu...",0.268899
7,HGBR,15,F2,"learning_rate=0.05, max_leaf_nodes=15, l2_regu...",0.274653
8,HGBR,15,F3,"learning_rate=0.05, max_leaf_nodes=15, l2_regu...",0.278706
9,HGBR,15,F4,"learning_rate=0.05, max_leaf_nodes=15, l2_regu...",0.276521



Primary 60-minute frozen configurations:


,model,horizon_minutes,feature_family,selected_parameters,validation_MAE_kW
20,Ridge,60,F0,alpha=100,0.512713
21,Ridge,60,F1,alpha=100,0.523861
22,Ridge,60,F2,alpha=100,0.512424
23,Ridge,60,F3,alpha=0.01,0.511464
24,Ridge,60,F4,alpha=0.01,0.509421
25,HGBR,60,F0,"learning_rate=0.05, max_leaf_nodes=15, l2_regu...",0.462910
26,HGBR,60,F1,"learning_rate=0.05, max_leaf_nodes=15, l2_regu...",0.363070
27,HGBR,60,F2,"learning_rate=0.05, max_leaf_nodes=15, l2_regu...",0.368494
28,HGBR,60,F3,"learning_rate=0.05, max_leaf_nodes=15, l2_regu...",0.383858
29,HGBR,60,F4,"learning_rate=0.05, max_leaf_nodes=15, l2_regu...",0.396150



Pre-test model-selection freeze: PASSED


In [25]:
# ============================================================
# Step 24: Final refit on train + validation and test evaluation
# ============================================================

# ------------------------------------------------------------
# IMPORTANT:
#
# Hyperparameter selection is now complete and frozen.
#
# This cell:
#
#   1. combines training + validation;
#   2. refits each selected model configuration;
#   3. evaluates once on the chronological test period;
#   4. records all-time and daylight-only metrics.
#
# NO hyperparameter selection occurs in this cell.
# ------------------------------------------------------------


# ------------------------------------------------------------
# 1. Storage for final predictions and test metrics
# ------------------------------------------------------------

final_prediction_tables = {}

final_test_result_rows = []


# ------------------------------------------------------------
# 2. Loop over all three forecast horizons
# ------------------------------------------------------------

for horizon_minutes in [
    15,
    30,
    60,
]:

    bundle = modelling_bundles[horizon_minutes]

    y = bundle["y"]

    # --------------------------------------------------------
    # Final pre-test fitting set:
    #
    #     training + validation
    #
    # Test remains separate.
    # --------------------------------------------------------

    fit_mask = (
        bundle["train_mask"]
        | bundle["validation_mask"]
    )

    test_mask = bundle["test_mask"]

    daylight_test_mask = (
        test_mask
        & bundle["daylight_mask"]
    )

    y_fit = y.loc[fit_mask]
    y_test = y.loc[test_mask]

    print("\n========================================")
    print(
        f"Final test evaluation: "
        f"{horizon_minutes}-minute horizon"
    )
    print("========================================")

    print(
        f"  Train + validation rows: "
        f"{fit_mask.sum()}"
    )

    print(
        f"  Test rows:               "
        f"{test_mask.sum()}"
    )

    print(
        f"  Daylight test rows:      "
        f"{daylight_test_mask.sum()}"
    )


    # Store final predictions for this horizon.
    horizon_predictions = pd.DataFrame(
        index=y.loc[test_mask].index
    )

    horizon_predictions["target_P_Solar_kW"] = (
        y.loc[test_mask]
    )


    # ========================================================
    # 3. RIDGE — refit frozen F0-F4 configurations
    # ========================================================

    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        X = bundle["X"][feature_family]

        X_fit = X.loc[fit_mask]
        X_test = X.loc[test_mask]

        config = selected_model_configs[
            (
                "Ridge",
                horizon_minutes,
                feature_family,
            )
        ]

        alpha = config["alpha"]


        # ----------------------------------------------------
        # StandardScaler is refitted using the complete
        # pre-test period only:
        #
        #     train + validation
        #
        # Test data never participates in scaler fitting.
        # ----------------------------------------------------

        ridge_model = Pipeline(
            steps=[
                (
                    "scaler",
                    StandardScaler(),
                ),
                (
                    "ridge",
                    Ridge(
                        alpha=alpha
                    ),
                ),
            ]
        )

        ridge_model.fit(
            X_fit,
            y_fit,
        )

        test_predictions = ridge_model.predict(
            X_test
        )

        prediction_series = pd.Series(
            test_predictions,
            index=X_test.index,
            name=f"Ridge_{feature_family}",
        )

        horizon_predictions[
            f"Ridge_{feature_family}"
        ] = prediction_series


        # ----------------------------------------------------
        # Evaluate:
        #
        #   1. all test timestamps;
        #   2. daylight-only test timestamps.
        # ----------------------------------------------------

        evaluation_scopes = {
            "all": test_mask,
            "daylight": daylight_test_mask,
        }

        for scope_name, scope_mask in (
            evaluation_scopes.items()
        ):

            scope_index = y.index[
                scope_mask
            ]

            y_true = y.loc[
                scope_index
            ]

            y_pred = prediction_series.loc[
                scope_index
            ]

            metrics = regression_metrics(
                y_true,
                y_pred,
            )

            final_test_result_rows.append(
                {
                    "model": "Ridge",
                    "horizon_minutes": horizon_minutes,
                    "feature_family": feature_family,
                    "scope": scope_name,
                    "n_rows": len(y_true),
                    "selected_parameters": (
                        f"alpha={alpha:g}"
                    ),
                    **metrics,
                }
            )


        print(
            f"  Ridge {feature_family}: completed"
        )


    # ========================================================
    # 4. HGBR — refit frozen F0-F4 configurations
    # ========================================================

    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        X = bundle["X"][feature_family]

        X_fit = X.loc[fit_mask]
        X_test = X.loc[test_mask]

        config = selected_model_configs[
            (
                "HGBR",
                horizon_minutes,
                feature_family,
            )
        ]


        # ----------------------------------------------------
        # Use the exact validation-selected configuration.
        #
        # No test-driven adjustment is allowed here.
        # ----------------------------------------------------

        hgbr_model = HistGradientBoostingRegressor(
            loss="squared_error",
            learning_rate=(
                config["learning_rate"]
            ),
            max_iter=(
                config["max_iter"]
            ),
            max_leaf_nodes=(
                config["max_leaf_nodes"]
            ),
            l2_regularization=(
                config["l2_regularization"]
            ),
            early_stopping=False,
            random_state=42,
        )

        hgbr_model.fit(
            X_fit,
            y_fit,
        )

        test_predictions = hgbr_model.predict(
            X_test
        )

        prediction_series = pd.Series(
            test_predictions,
            index=X_test.index,
            name=f"HGBR_{feature_family}",
        )

        horizon_predictions[
            f"HGBR_{feature_family}"
        ] = prediction_series


        evaluation_scopes = {
            "all": test_mask,
            "daylight": daylight_test_mask,
        }

        for scope_name, scope_mask in (
            evaluation_scopes.items()
        ):

            scope_index = y.index[
                scope_mask
            ]

            y_true = y.loc[
                scope_index
            ]

            y_pred = prediction_series.loc[
                scope_index
            ]

            metrics = regression_metrics(
                y_true,
                y_pred,
            )

            final_test_result_rows.append(
                {
                    "model": "HGBR",
                    "horizon_minutes": horizon_minutes,
                    "feature_family": feature_family,
                    "scope": scope_name,
                    "n_rows": len(y_true),
                    "selected_parameters": (
                        "learning_rate="
                        f"{config['learning_rate']:g}, "
                        "max_leaf_nodes="
                        f"{config['max_leaf_nodes']}, "
                        "l2_regularization="
                        f"{config['l2_regularization']:g}, "
                        "max_iter="
                        f"{config['max_iter']}"
                    ),
                    **metrics,
                }
            )


        print(
            f"  HGBR  {feature_family}: completed"
        )


    # --------------------------------------------------------
    # 5. Add test metadata to the stored prediction table
    # --------------------------------------------------------

    test_metadata = (
        bundle["metadata"]
        .loc[test_mask]
        [
            [
                "issue_time",
                "valid_time",
            ]
        ]
    )

    horizon_predictions.insert(
        0,
        "valid_time",
        test_metadata["valid_time"],
    )

    horizon_predictions.insert(
        0,
        "issue_time",
        test_metadata["issue_time"],
    )

    horizon_predictions[
        "daylight"
    ] = (
        bundle["daylight_mask"]
        .loc[test_mask]
        .astype(bool)
    )


    # Final alignment sanity check.
    assert horizon_predictions.index.equals(
        y_test.index
    )

    assert (
        horizon_predictions[
            "target_P_Solar_kW"
        ]
        .equals(y_test)
    )

    final_prediction_tables[
        horizon_minutes
    ] = horizon_predictions


# ------------------------------------------------------------
# 6. Assemble complete final ML test-results table
# ------------------------------------------------------------

final_test_results = pd.DataFrame(
    final_test_result_rows
)


# ------------------------------------------------------------
# 7. Verify expected result count
# ------------------------------------------------------------
#
#   2 models
#   x 3 horizons
#   x 5 feature families
#   x 2 evaluation scopes
#   = 60 result rows
# ------------------------------------------------------------

assert len(final_test_results) == 60

assert final_test_results[
    [
        "MAE_kW",
        "RMSE_kW",
        "nMAE_percent",
        "nRMSE_percent",
        "R2",
    ]
].notna().all().all()

assert np.isfinite(
    final_test_results[
        [
            "MAE_kW",
            "RMSE_kW",
            "nMAE_percent",
            "nRMSE_percent",
            "R2",
        ]
    ].to_numpy()
).all()

print(
    "\nFinal ML test-result construction: PASSED"
)


# ------------------------------------------------------------
# 8. Sort the final results for interpretation
# ------------------------------------------------------------

final_test_results["model"] = pd.Categorical(
    final_test_results["model"],
    categories=[
        "Ridge",
        "HGBR",
    ],
    ordered=True,
)

final_test_results["feature_family"] = pd.Categorical(
    final_test_results["feature_family"],
    categories=[
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ],
    ordered=True,
)

final_test_results["scope"] = pd.Categorical(
    final_test_results["scope"],
    categories=[
        "all",
        "daylight",
    ],
    ordered=True,
)

final_test_results = (
    final_test_results
    .sort_values(
        [
            "horizon_minutes",
            "model",
            "scope",
            "feature_family",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 9. Display the PRIMARY 60-minute all-time test result
# ------------------------------------------------------------

primary_60_test = (
    final_test_results.loc[
        (
            final_test_results[
                "horizon_minutes"
            ].eq(60)
        )
        & (
            final_test_results[
                "scope"
            ].eq("all")
        )
    ]
    .copy()
)

print(
    "\nPrimary 60-minute ALL-TIME test results:"
)

display(
    primary_60_test[
        [
            "model",
            "feature_family",
            "n_rows",
            "MAE_kW",
            "RMSE_kW",
            "nMAE_percent",
            "nRMSE_percent",
            "R2",
        ]
    ]
)


# ------------------------------------------------------------
# 10. Display the PRIMARY 60-minute DAYLIGHT test result
# ------------------------------------------------------------

primary_60_daylight_test = (
    final_test_results.loc[
        (
            final_test_results[
                "horizon_minutes"
            ].eq(60)
        )
        & (
            final_test_results[
                "scope"
            ].eq("daylight")
        )
    ]
    .copy()
)

print(
    "\nPrimary 60-minute DAYLIGHT test results:"
)

display(
    primary_60_daylight_test[
        [
            "model",
            "feature_family",
            "n_rows",
            "MAE_kW",
            "RMSE_kW",
            "nMAE_percent",
            "nRMSE_percent",
            "R2",
        ]
    ]
)


# ------------------------------------------------------------
# 11. Build the final F3 -> F4 TEST comparison
# ------------------------------------------------------------

f3_f4_test_rows = []


for model_name in [
    "Ridge",
    "HGBR",
]:

    for horizon_minutes in [
        15,
        30,
        60,
    ]:

        for scope_name in [
            "all",
            "daylight",
        ]:

            subset = final_test_results.loc[
                (
                    final_test_results[
                        "model"
                    ].eq(model_name)
                )
                & (
                    final_test_results[
                        "horizon_minutes"
                    ].eq(horizon_minutes)
                )
                & (
                    final_test_results[
                        "scope"
                    ].eq(scope_name)
                )
            ]

            f3_mae = (
                subset.loc[
                    subset[
                        "feature_family"
                    ].eq("F3"),
                    "MAE_kW",
                ]
                .iloc[0]
            )

            f4_mae = (
                subset.loc[
                    subset[
                        "feature_family"
                    ].eq("F4"),
                    "MAE_kW",
                ]
                .iloc[0]
            )

            relative_change = (
                (f3_mae - f4_mae)
                / f3_mae
                * 100.0
            )

            f3_f4_test_rows.append(
                {
                    "model": model_name,
                    "horizon_minutes": horizon_minutes,
                    "scope": scope_name,
                    "F3_MAE_kW": f3_mae,
                    "F4_MAE_kW": f4_mae,
                    "F3_to_F4_MAE_improvement_percent": (
                        relative_change
                    ),
                }
            )


final_f3_f4_test_summary = pd.DataFrame(
    f3_f4_test_rows
)


print(
    "\nFinal F3 -> F4 TEST comparison:"
)

display(
    final_f3_f4_test_summary
    .sort_values(
        [
            "model",
            "scope",
            "horizon_minutes",
        ]
    )
    .reset_index(drop=True)
)


print(
    "\nFinal frozen-model test evaluation: PASSED"
)


Final test evaluation: 15-minute horizon
  Train + validation rows: 104829
  Test rows:               26496
  Daylight test rows:      17851
  Ridge F0: completed
  Ridge F1: completed
  Ridge F2: completed
  Ridge F3: completed
  Ridge F4: completed
  HGBR  F0: completed
  HGBR  F1: completed
  HGBR  F2: completed
  HGBR  F3: completed
  HGBR  F4: completed

Final test evaluation: 30-minute horizon
  Train + validation rows: 104826
  Test rows:               26496
  Daylight test rows:      17851
  Ridge F0: completed
  Ridge F1: completed
  Ridge F2: completed
  Ridge F3: completed
  Ridge F4: completed
  HGBR  F0: completed
  HGBR  F1: completed
  HGBR  F2: completed
  HGBR  F3: completed
  HGBR  F4: completed

Final test evaluation: 60-minute horizon
  Train + validation rows: 104820
  Test rows:               26496
  Daylight test rows:      17851
  Ridge F0: completed
  Ridge F1: completed
  Ridge F2: completed
  Ridge F3: completed
  Ridge F4: completed
  HGBR  F0: completed
  

,model,feature_family,n_rows,MAE_kW,RMSE_kW,nMAE_percent,nRMSE_percent,R2
40,Ridge,F0,26496,0.474499,0.826878,4.744986,8.268783,0.749060
41,Ridge,F1,26496,0.455582,0.760020,4.555824,7.600195,0.788000
42,Ridge,F2,26496,0.439199,0.746812,4.391990,7.468115,0.795304
43,Ridge,F3,26496,0.527476,0.777570,5.274765,7.775697,0.778096
44,Ridge,F4,26496,0.562776,0.802506,5.627765,8.025063,0.763635
50,HGBR,F0,26496,0.426420,0.775054,4.264199,7.750540,0.779530
51,HGBR,F1,26496,0.345082,0.684492,3.450815,6.844925,0.828041
52,HGBR,F2,26496,0.347983,0.689167,3.479825,6.891666,0.825685
53,HGBR,F3,26496,0.351776,0.692404,3.517761,6.924036,0.824044
54,HGBR,F4,26496,0.351646,0.692137,3.516459,6.921374,0.824179



Primary 60-minute DAYLIGHT test results:


,model,feature_family,n_rows,MAE_kW,RMSE_kW,nMAE_percent,nRMSE_percent,R2
45,Ridge,F0,17851,0.656932,1.003224,6.569316,10.032244,0.662804
46,Ridge,F1,17851,0.590232,0.914012,5.902323,9.140120,0.720108
47,Ridge,F2,17851,0.571447,0.898589,5.714468,8.985887,0.729474
48,Ridge,F3,17851,0.609240,0.904667,6.092396,9.046671,0.725802
49,Ridge,F4,17851,0.634608,0.921441,6.346084,9.214405,0.715540
55,HGBR,F0,17851,0.622643,0.943915,6.226431,9.439148,0.701494
56,HGBR,F1,17851,0.506001,0.833808,5.060007,8.338081,0.767073
57,HGBR,F2,17851,0.513339,0.839597,5.133394,8.395968,0.763828
58,HGBR,F3,17851,0.518207,0.843477,5.182071,8.434769,0.761640
59,HGBR,F4,17851,0.517860,0.843053,5.178603,8.430528,0.761880



Final F3 -> F4 TEST comparison:


,model,horizon_minutes,scope,F3_MAE_kW,F4_MAE_kW,F3_to_F4_MAE_improvement_percent
0,HGBR,15,all,0.265345,0.263828,0.571617
1,HGBR,30,all,0.307225,0.307053,0.056085
2,HGBR,60,all,0.351776,0.351646,0.036997
3,HGBR,15,daylight,0.391272,0.389439,0.468340
4,HGBR,30,daylight,0.452722,0.452502,0.048688
5,HGBR,60,daylight,0.518207,0.517860,0.066918
6,Ridge,15,all,0.318521,0.329466,-3.436065
7,Ridge,30,all,0.401663,0.420881,-4.784612
8,Ridge,60,all,0.527476,0.562776,-6.692235
9,Ridge,15,daylight,0.414016,0.421066,-1.702771



Final frozen-model test evaluation: PASSED


# 13. Primary Out-of-Sample Test Findings

## Final Test Evaluation

After completing hyperparameter selection using the chronological validation
period, all selected configurations were frozen.

Each model was then refitted using:

$$
\text{training} + \text{validation}
$$

and evaluated once on the previously untouched test period:

`2019-06-01` to `2019-08-31`

No model configuration, feature definition, lag structure, or evaluation
criterion was changed after observing the test results.

---

## Primary 60-Minute All-Time Results

The 60-minute horizon is the primary forecast horizon defined by the
experimental protocol.

The persistence benchmark achieved:

| Method | MAE (kW) |
|---|---:|
| Persistence | 0.520977 |

The final Ridge results were:

| Feature family | MAE (kW) |
|---|---:|
| F0 | 0.474499 |
| F1 | 0.455582 |
| F2 | 0.439199 |
| F3 | 0.527476 |
| F4 | 0.562776 |

The final Histogram Gradient Boosting results were:

| Feature family | MAE (kW) |
|---|---:|
| F0 | 0.426420 |
| F1 | 0.345082 |
| F2 | 0.347983 |
| F3 | 0.351776 |
| F4 | 0.351646 |

Within the tested HGBR configurations, F1 produced the lowest 60-minute
test MAE.

Within Ridge, F2 produced the lowest 60-minute test MAE.

---

## Primary F3 → F4 Research Comparison

The central research comparison is:

$$
F3 \rightarrow F4
$$

because F3 already contains the continuous solar and lunar astronomical
variables underlying the Panchang representation.

For the 60-minute all-time test set:

### Ridge

$$
\text{MAE}_{F3}
=
0.527476\text{ kW}
$$

$$
\text{MAE}_{F4}
=
0.562776\text{ kW}
$$

The relative MAE change was:

$$
-6.69\%
$$

Thus, adding the F4 Panchang representation increased test MAE under Ridge.

### Histogram Gradient Boosting

$$
\text{MAE}_{F3}
=
0.351776\text{ kW}
$$

$$
\text{MAE}_{F4}
=
0.351646\text{ kW}
$$

The relative MAE improvement was approximately:

$$
0.037\%
$$

This is a numerically positive but extremely small difference.

---

## F3 → F4 Across Forecast Horizons

For all test timestamps, the relative F3 → F4 MAE changes were:

| Model | 15 min | 30 min | 60 min |
|---|---:|---:|---:|
| Ridge | -3.44% | -4.78% | -6.69% |
| HGBR | +0.57% | +0.06% | +0.04% |

Positive values indicate lower MAE after adding F4.

Negative values indicate higher MAE after adding F4.

The Ridge results therefore show degradation at all three horizons.

HGBR shows small positive changes, but the magnitude decreases from
approximately 0.57% at 15 minutes to approximately 0.04% at 60 minutes.

---

## Daylight-Only F3 → F4 Comparison

The daylight-only evaluation leads to a similar conclusion.

For the primary 60-minute horizon:

### Ridge

$$
0.609240
\rightarrow
0.634608\text{ kW}
$$

corresponding to approximately:

$$
-4.16\%
$$

relative MAE change.

### HGBR

$$
0.518207
\rightarrow
0.517860\text{ kW}
$$

corresponding to approximately:

$$
+0.067\%
$$

relative MAE change.

Thus, restricting evaluation to physically relevant daylight periods does
not produce a substantial F4 advantage.

---

## Validation Versus Test Behaviour

An important result is that validation behaviour did not always generalize
to the final test period.

For example, Ridge showed small positive F3 → F4 validation improvements at
all three horizons:

- approximately 0.40% at 15 minutes;
- approximately 0.23% at 30 minutes;
- approximately 0.40% at 60 minutes.

However, the final test period showed deterioration under Ridge at all three
horizons.

Similarly, the HGBR validation results were mixed across horizons, whereas
the final test results showed only very small positive F3 → F4 differences.

This reinforces the importance of retaining a completely untouched
chronological test period.

---

## Interpretation of the Current Evidence

The current results do **not** support a claim that discrete Panchang
features provide a robust forecasting improvement beyond the continuous
astronomical variables already contained in F3.

The evidence is model-dependent:

- Ridge shows a clear test-set deterioration after adding F4;
- HGBR shows only very small numerical F4 improvements over F3.

Furthermore, at the primary 60-minute horizon, the lowest HGBR test MAE is
obtained with F1 rather than F3 or F4.

Therefore no general forecasting advantage should currently be attributed
to the Panchang representation.

---

## What the Results Do Show

The experiment nevertheless demonstrates meaningful differences between
feature families.

In particular, deterministic future-time information can improve forecasting
relative to purely historical inputs.

For the nonlinear HGBR model, adding Gregorian future-time features produced
a substantial improvement over F0.

At the primary 60-minute horizon:

$$
\text{HGBR F0 MAE}
=
0.426420\text{ kW}
$$

whereas:

$$
\text{HGBR F1 MAE}
=
0.345082\text{ kW}
$$

This indicates that deterministic temporal structure is useful in the
forecasting problem even though progressively more detailed astronomical and
Panchang representations do not necessarily produce further improvement.

---

## Statistical Caution

The F3 and F4 forecasts are generated for the same timestamps and therefore
their forecast errors are paired and temporally dependent.

Small numerical differences, particularly the HGBR F3 → F4 differences,
should not be interpreted as meaningful without uncertainty analysis.

The next stage should therefore quantify the uncertainty in paired forecast
error differences using a time-series-aware procedure rather than treating
individual 5-minute observations as independent samples.

No claim of statistically meaningful F4 improvement will be made before that
analysis is completed.

# 14. Paired Block-Bootstrap Uncertainty Analysis

## Purpose

The final F3 and F4 forecasts are produced for exactly the same forecast-valid
timestamps.

Therefore their forecast errors are naturally paired.

The central uncertainty question is not simply whether:

$$
\text{MAE}_{F4}
<
\text{MAE}_{F3}
$$

for one observed test sample.

Instead, we want to estimate how stable the paired F3 → F4 error difference
is when temporal dependence in the test data is respected.

---

## Paired Absolute-Error Difference

For each forecast-valid timestamp $i$, define:

$$
e_{F3,i}
=
|y_i-\hat{y}_{F3,i}|
$$

and:

$$
e_{F4,i}
=
|y_i-\hat{y}_{F4,i}|
$$

The paired error difference is then:

$$
d_i
=
e_{F3,i}
-
e_{F4,i}
$$

The sign convention is therefore:

- $d_i > 0$: F4 has lower absolute error than F3;
- $d_i < 0$: F4 has higher absolute error than F3.

The corresponding mean paired difference is:

$$
\Delta\text{MAE}
=
\text{MAE}_{F3}
-
\text{MAE}_{F4}
$$

Thus:

$$
\Delta\text{MAE} > 0
$$

indicates an F4 improvement.

---

## Why Individual 5-Minute Rows Are Not Bootstrapped Independently

Adjacent 5-minute solar-forecast errors are not statistically independent.

They may share:

- the same weather regime;
- the same cloud system;
- similar solar geometry;
- common daily irradiance patterns;
- overlapping historical predictor windows.

Therefore a conventional independent-observation bootstrap would break the
time-series dependence structure and could produce overly narrow uncertainty
intervals.

---

## Bootstrap Block Definition

The primary uncertainty analysis will use:

**one UTC calendar day as one bootstrap block**

based on the forecast-valid timestamp.

The final test period is:

`2019-06-01` to `2019-08-31`

which contains:

$$
92
$$

calendar days.

Each bootstrap sample will therefore resample whole test days rather than
individual 5-minute observations.

This preserves the complete within-day sequence of paired F3 and F4 errors.

---

## Bootstrap Procedure

For each combination of:

- model:
  - Ridge;
  - HGBR;
- forecast horizon:
  - 15 minutes;
  - 30 minutes;
  - 60 minutes;
- evaluation scope:
  - all timestamps;
  - daylight-only timestamps;

the following procedure will be used.

1. Compute the paired F3 and F4 absolute errors on the fixed test set.

2. Group those paired errors by forecast-valid UTC calendar day.

3. Draw 92 daily blocks with replacement.

4. Combine the selected blocks and calculate:

$$
\Delta\text{MAE}
=
\text{MAE}_{F3}
-
\text{MAE}_{F4}
$$

5. Also calculate the relative F3 → F4 MAE improvement:

$$
\Delta_{\text{relative}}
=
\frac{
\text{MAE}_{F3}
-
\text{MAE}_{F4}
}{
\text{MAE}_{F3}
}
\times 100
$$

6. Repeat the daily-block resampling:

$$
10{,}000
$$

times.

A fixed random seed will be used for reproducibility.

---

## Confidence Intervals

A two-sided 95% percentile bootstrap interval will be calculated from the:

$$
2.5^{\text{th}}
$$

and:

$$
97.5^{\text{th}}
$$

percentiles of the bootstrap distribution.

The principal interval will be reported for:

$$
\Delta\text{MAE}
$$

in kW.

The relative percentage change will be reported as a supplementary quantity.

---

## Interpretation

The interval will be interpreted using the previously defined sign
convention.

### Entire interval above zero

If:

$$
\text{CI}_{95\%}(\Delta\text{MAE}) > 0
$$

then the sampled test-period evidence is consistent with F4 having lower MAE
than F3 under the selected model and horizon.

### Interval contains zero

If the interval crosses:

$$
0
$$

then the observed F3 → F4 difference is not clearly separated from zero
under this block-resampling analysis.

### Entire interval below zero

If:

$$
\text{CI}_{95\%}(\Delta\text{MAE}) < 0
$$

then the sampled test-period evidence is consistent with F4 having higher MAE
than F3.

No conclusion will be based only on the sign of the point estimate.

---

## All-Time and Daylight Evaluation

The bootstrap will be performed separately for:

1. all test timestamps;
2. daylight-only test timestamps.

The daylight definition remains:

$$
\text{solar elevation} > 0^\circ
$$

at the forecast-valid time.

The daylight subset is therefore determined independently of the observed PV
target.

---

## Paired Resampling Requirement

For every bootstrap replicate, the same sampled daily blocks must be used
simultaneously for F3 and F4.

F3 and F4 must never be resampled independently.

This preserves the paired experimental design and directly measures the
uncertainty in their forecast-error difference.

---

## Scope of the Bootstrap Inference

The final trained models will remain fixed during this analysis.

The bootstrap therefore quantifies uncertainty associated with the
chronological **test-period forecast errors conditional on the fitted
models**.

It does not quantify every possible source of uncertainty associated with:

- retraining the model on different historical datasets;
- hyperparameter selection;
- alternative train / validation / test periods;
- alternative feature definitions.

Those questions would require additional repeated or rolling-origin
experiments.

---

## Frozen Bootstrap Settings

The primary block-bootstrap analysis is therefore fixed as:

- paired F3 versus F4 comparison;
- block unit: one forecast-valid UTC calendar day;
- test period only;
- 92 daily blocks;
- 10,000 bootstrap replicates;
- fixed random seed;
- 95% percentile confidence interval;
- separate all-time and daylight analyses;
- no model retraining during bootstrap.

These settings will not be changed in response to the resulting confidence
intervals.

In [26]:
# ============================================================
# Step 25: Paired daily-block bootstrap for F3 vs F4
# ============================================================

# ------------------------------------------------------------
# 1. Frozen bootstrap settings
# ------------------------------------------------------------

N_BOOTSTRAP = 10_000
BOOTSTRAP_RANDOM_SEED = 42

BOOTSTRAP_MODELS = [
    "Ridge",
    "HGBR",
]

BOOTSTRAP_HORIZONS = [
    15,
    30,
    60,
]

BOOTSTRAP_SCOPES = [
    "all",
    "daylight",
]


# One reproducible random-number generator for the complete
# bootstrap analysis.
rng = np.random.default_rng(
    BOOTSTRAP_RANDOM_SEED
)


# ------------------------------------------------------------
# 2. Storage for bootstrap summaries
# ------------------------------------------------------------

bootstrap_summary_rows = []

# Keep the complete bootstrap distributions as well, in case
# we later need plots or additional diagnostics.
bootstrap_distributions = {}


# ------------------------------------------------------------
# 3. Run one paired daily-block bootstrap for every:
#
#       model x horizon x scope
#
# Total analyses:
#
#       2 x 3 x 2 = 12
# ------------------------------------------------------------

for model_name in BOOTSTRAP_MODELS:

    for horizon_minutes in BOOTSTRAP_HORIZONS:

        # Final test predictions generated in Cell 37.
        df_predictions = (
            final_prediction_tables[
                horizon_minutes
            ]
            .copy()
        )


        # ----------------------------------------------------
        # Verify that this is genuinely the fixed TEST set.
        # ----------------------------------------------------

        assert len(df_predictions) == 26_496

        assert (
            df_predictions["valid_time"].min()
            == pd.Timestamp(
                "2019-06-01 00:00:00",
                tz="UTC",
            )
        )

        assert (
            df_predictions["valid_time"].max()
            == pd.Timestamp(
                "2019-08-31 23:55:00",
                tz="UTC",
            )
        )


        # ----------------------------------------------------
        # Target and paired F3/F4 predictions
        # ----------------------------------------------------

        target_column = "target_P_Solar_kW"

        f3_column = (
            f"{model_name}_F3"
        )

        f4_column = (
            f"{model_name}_F4"
        )


        for scope_name in BOOTSTRAP_SCOPES:

            # ------------------------------------------------
            # Select evaluation scope.
            # ------------------------------------------------

            if scope_name == "all":

                df_scope = (
                    df_predictions
                    .copy()
                )

            elif scope_name == "daylight":

                df_scope = (
                    df_predictions.loc[
                        df_predictions[
                            "daylight"
                        ]
                    ]
                    .copy()
                )

            else:
                raise ValueError(
                    f"Unexpected scope: {scope_name}"
                )


            # ------------------------------------------------
            # 4. Calculate paired absolute errors
            # ------------------------------------------------
            #
            # Positive paired difference means F4 is better:
            #
            #     |error F3| - |error F4| > 0
            # ------------------------------------------------

            df_scope["abs_error_F3"] = np.abs(
                df_scope[target_column]
                - df_scope[f3_column]
            )

            df_scope["abs_error_F4"] = np.abs(
                df_scope[target_column]
                - df_scope[f4_column]
            )

            df_scope["paired_error_difference"] = (
                df_scope["abs_error_F3"]
                - df_scope["abs_error_F4"]
            )


            # ------------------------------------------------
            # 5. Assign each forecast-valid timestamp to its
            #    UTC calendar-day block.
            # ------------------------------------------------

            df_scope["valid_date"] = (
                df_scope["valid_time"]
                .dt.floor("D")
            )


            # ------------------------------------------------
            # 6. Collapse each day into sufficient statistics
            #
            # For each daily block, retain:
            #
            #   number of observations
            #   sum of F3 absolute errors
            #   sum of F4 absolute errors
            #
            # If a day is sampled multiple times, the complete
            # block is therefore repeated multiple times.
            #
            # This is mathematically equivalent to concatenating
            # the selected daily rows and recalculating MAE, but
            # is much faster for 10,000 bootstrap replicates.
            # ------------------------------------------------

            daily_blocks = (
                df_scope
                .groupby(
                    "valid_date",
                    sort=True,
                )
                .agg(
                    n_rows=(
                        target_column,
                        "size",
                    ),
                    abs_error_F3_sum=(
                        "abs_error_F3",
                        "sum",
                    ),
                    abs_error_F4_sum=(
                        "abs_error_F4",
                        "sum",
                    ),
                )
            )


            # ------------------------------------------------
            # 7. Verify the natural daily-block structure
            # ------------------------------------------------

            n_days = len(daily_blocks)

            assert n_days == 92, (
                f"{model_name} / "
                f"{horizon_minutes} min / "
                f"{scope_name}: "
                f"expected 92 daily blocks, "
                f"found {n_days}."
            )


            # ------------------------------------------------
            # 8. Calculate the observed test-set point estimate
            # ------------------------------------------------

            observed_f3_mae = (
                df_scope[
                    "abs_error_F3"
                ]
                .mean()
            )

            observed_f4_mae = (
                df_scope[
                    "abs_error_F4"
                ]
                .mean()
            )

            observed_delta_mae = (
                observed_f3_mae
                - observed_f4_mae
            )

            observed_relative_change = (
                observed_delta_mae
                / observed_f3_mae
                * 100.0
            )


            # ------------------------------------------------
            # 9. Verify that the observed point estimate
            #    reproduces Cell 37 exactly.
            # ------------------------------------------------

            expected_row = (
                final_f3_f4_test_summary.loc[
                    (
                        final_f3_f4_test_summary[
                            "model"
                        ].eq(model_name)
                    )
                    & (
                        final_f3_f4_test_summary[
                            "horizon_minutes"
                        ].eq(horizon_minutes)
                    )
                    & (
                        final_f3_f4_test_summary[
                            "scope"
                        ].eq(scope_name)
                    )
                ]
                .iloc[0]
            )

            assert np.isclose(
                observed_f3_mae,
                expected_row["F3_MAE_kW"],
                atol=1e-12,
                rtol=0.0,
            )

            assert np.isclose(
                observed_f4_mae,
                expected_row["F4_MAE_kW"],
                atol=1e-12,
                rtol=0.0,
            )

            assert np.isclose(
                observed_relative_change,
                expected_row[
                    "F3_to_F4_MAE_improvement_percent"
                ],
                atol=1e-10,
                rtol=0.0,
            )


            # ------------------------------------------------
            # 10. Convert daily sufficient statistics to NumPy
            #     arrays for efficient resampling.
            # ------------------------------------------------

            daily_counts = (
                daily_blocks[
                    "n_rows"
                ]
                .to_numpy(
                    dtype=np.int64
                )
            )

            daily_f3_error_sums = (
                daily_blocks[
                    "abs_error_F3_sum"
                ]
                .to_numpy(
                    dtype=float
                )
            )

            daily_f4_error_sums = (
                daily_blocks[
                    "abs_error_F4_sum"
                ]
                .to_numpy(
                    dtype=float
                )
            )


            # ------------------------------------------------
            # 11. Draw whole daily blocks with replacement
            #
            # Shape:
            #
            #     10,000 bootstrap samples x 92 sampled days
            #
            # Each row represents one complete bootstrap
            # replicate.
            # ------------------------------------------------

            sampled_day_indices = rng.integers(
                low=0,
                high=n_days,
                size=(
                    N_BOOTSTRAP,
                    n_days,
                ),
            )


            # ------------------------------------------------
            # 12. Reconstruct pooled MAE for each bootstrap
            #     replicate.
            # ------------------------------------------------

            bootstrap_row_counts = (
                daily_counts[
                    sampled_day_indices
                ]
                .sum(axis=1)
            )

            bootstrap_f3_error_sums = (
                daily_f3_error_sums[
                    sampled_day_indices
                ]
                .sum(axis=1)
            )

            bootstrap_f4_error_sums = (
                daily_f4_error_sums[
                    sampled_day_indices
                ]
                .sum(axis=1)
            )

            bootstrap_f3_mae = (
                bootstrap_f3_error_sums
                / bootstrap_row_counts
            )

            bootstrap_f4_mae = (
                bootstrap_f4_error_sums
                / bootstrap_row_counts
            )


            # ------------------------------------------------
            # 13. Paired F3 -> F4 MAE difference
            # ------------------------------------------------

            bootstrap_delta_mae = (
                bootstrap_f3_mae
                - bootstrap_f4_mae
            )

            bootstrap_relative_change = (
                bootstrap_delta_mae
                / bootstrap_f3_mae
                * 100.0
            )


            # ------------------------------------------------
            # 14. Calculate frozen 95% percentile intervals
            # ------------------------------------------------

            delta_ci_lower, delta_ci_upper = (
                np.percentile(
                    bootstrap_delta_mae,
                    [
                        2.5,
                        97.5,
                    ],
                )
            )

            relative_ci_lower, relative_ci_upper = (
                np.percentile(
                    bootstrap_relative_change,
                    [
                        2.5,
                        97.5,
                    ],
                )
            )


            # ------------------------------------------------
            # 15. Store complete distributions
            # ------------------------------------------------

            distribution_key = (
                model_name,
                horizon_minutes,
                scope_name,
            )

            bootstrap_distributions[
                distribution_key
            ] = pd.DataFrame(
                {
                    "delta_MAE_kW": (
                        bootstrap_delta_mae
                    ),
                    "relative_MAE_change_percent": (
                        bootstrap_relative_change
                    ),
                }
            )


            # ------------------------------------------------
            # 16. Store compact summary
            # ------------------------------------------------

            bootstrap_summary_rows.append(
                {
                    "model": model_name,
                    "horizon_minutes": (
                        horizon_minutes
                    ),
                    "scope": scope_name,
                    "n_test_rows": len(
                        df_scope
                    ),
                    "n_daily_blocks": n_days,
                    "F3_MAE_kW": (
                        observed_f3_mae
                    ),
                    "F4_MAE_kW": (
                        observed_f4_mae
                    ),
                    "delta_MAE_kW": (
                        observed_delta_mae
                    ),
                    "delta_MAE_CI95_lower_kW": (
                        delta_ci_lower
                    ),
                    "delta_MAE_CI95_upper_kW": (
                        delta_ci_upper
                    ),
                    "relative_MAE_change_percent": (
                        observed_relative_change
                    ),
                    "relative_CI95_lower_percent": (
                        relative_ci_lower
                    ),
                    "relative_CI95_upper_percent": (
                        relative_ci_upper
                    ),
                }
            )


# ------------------------------------------------------------
# 17. Assemble the final bootstrap summary table
# ------------------------------------------------------------

bootstrap_summary = pd.DataFrame(
    bootstrap_summary_rows
)


# ------------------------------------------------------------
# 18. Final structural checks
# ------------------------------------------------------------

assert len(bootstrap_summary) == 12

assert len(bootstrap_distributions) == 12

assert (
    bootstrap_summary[
        "n_daily_blocks"
    ]
    .eq(92)
    .all()
)

assert np.isfinite(
    bootstrap_summary[
        [
            "F3_MAE_kW",
            "F4_MAE_kW",
            "delta_MAE_kW",
            "delta_MAE_CI95_lower_kW",
            "delta_MAE_CI95_upper_kW",
            "relative_MAE_change_percent",
            "relative_CI95_lower_percent",
            "relative_CI95_upper_percent",
        ]
    ].to_numpy()
).all()


print(
    "Paired daily-block bootstrap: PASSED"
)


# ------------------------------------------------------------
# 19. Sort for interpretation
# ------------------------------------------------------------

bootstrap_summary = (
    bootstrap_summary
    .sort_values(
        [
            "model",
            "scope",
            "horizon_minutes",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 20. Display compact final uncertainty table
# ------------------------------------------------------------

print(
    "\nF3 -> F4 paired daily-block bootstrap results:"
)

display(
    bootstrap_summary[
        [
            "model",
            "horizon_minutes",
            "scope",
            "n_test_rows",
            "n_daily_blocks",
            "delta_MAE_kW",
            "delta_MAE_CI95_lower_kW",
            "delta_MAE_CI95_upper_kW",
            "relative_MAE_change_percent",
            "relative_CI95_lower_percent",
            "relative_CI95_upper_percent",
        ]
    ]
)


# ------------------------------------------------------------
# 21. Primary 60-minute result only
# ------------------------------------------------------------

print(
    "\nPrimary 60-minute bootstrap results:"
)

display(
    bootstrap_summary.loc[
        bootstrap_summary[
            "horizon_minutes"
        ].eq(60),
        [
            "model",
            "scope",
            "F3_MAE_kW",
            "F4_MAE_kW",
            "delta_MAE_kW",
            "delta_MAE_CI95_lower_kW",
            "delta_MAE_CI95_upper_kW",
            "relative_MAE_change_percent",
            "relative_CI95_lower_percent",
            "relative_CI95_upper_percent",
        ],
    ]
)

Paired daily-block bootstrap: PASSED

F3 -> F4 paired daily-block bootstrap results:


,model,horizon_minutes,scope,n_test_rows,n_daily_blocks,delta_MAE_kW,delta_MAE_CI95_lower_kW,delta_MAE_CI95_upper_kW,relative_MAE_change_percent,relative_CI95_lower_percent,relative_CI95_upper_percent
0,HGBR,15,all,26496,92,0.001517,-0.000185,0.003390,0.571617,-0.067338,1.316337
1,HGBR,30,all,26496,92,0.000172,-0.002692,0.002801,0.056085,-0.876010,0.928335
2,HGBR,60,all,26496,92,0.000130,-0.004125,0.004331,0.036997,-1.172098,1.233895
3,HGBR,15,daylight,17851,92,0.001832,-0.000683,0.004620,0.468340,-0.170313,1.212107
4,HGBR,30,daylight,17851,92,0.000220,-0.003807,0.004054,0.048688,-0.844997,0.920729
5,HGBR,60,daylight,17851,92,0.000347,-0.005377,0.005944,0.066918,-1.046433,1.138562
6,Ridge,15,all,26496,92,-0.010945,-0.014890,-0.006984,-3.436065,-4.794128,-2.141506
7,Ridge,30,all,26496,92,-0.019218,-0.026405,-0.012215,-4.784612,-6.689357,-3.006919
8,Ridge,60,all,26496,92,-0.035300,-0.048475,-0.022212,-6.692235,-9.297395,-4.186024
9,Ridge,15,daylight,17851,92,-0.007050,-0.009758,-0.004403,-1.702771,-2.430308,-1.046600



Primary 60-minute bootstrap results:


,model,scope,F3_MAE_kW,F4_MAE_kW,delta_MAE_kW,delta_MAE_CI95_lower_kW,delta_MAE_CI95_upper_kW,relative_MAE_change_percent,relative_CI95_lower_percent,relative_CI95_upper_percent
2,HGBR,all,0.351776,0.351646,0.000130,-0.004125,0.004331,0.036997,-1.172098,1.233895
5,HGBR,daylight,0.518207,0.517860,0.000347,-0.005377,0.005944,0.066918,-1.046433,1.138562
8,Ridge,all,0.527476,0.562776,-0.035300,-0.048475,-0.022212,-6.692235,-9.297395,-4.186024
11,Ridge,daylight,0.609240,0.634608,-0.025369,-0.035190,-0.015709,-4.164006,-5.879373,-2.527742


# 15. Paired Block-Bootstrap Results

## Bootstrap Completion

The frozen paired daily-block bootstrap was applied to the final test-period
F3 and F4 forecasts.

The analysis used:

- 92 forecast-valid UTC calendar-day blocks;
- 10,000 bootstrap replicates;
- paired F3 and F4 forecasts;
- fixed trained models;
- separate all-time and daylight analyses;
- a 95% percentile confidence interval.

The paired difference was defined as:

$$
\Delta\text{MAE}
=
\text{MAE}_{F3}
-
\text{MAE}_{F4}
$$

Therefore:

- positive values favor F4;
- negative values favor F3;
- intervals containing zero do not clearly separate F3 and F4 under this
  resampling analysis.

---

## HGBR Results

The HGBR F3 → F4 test-set point estimates were numerically positive at all
three forecast horizons.

For all test timestamps:

| Horizon | $\Delta$MAE (kW) | 95% CI (kW) | Relative change |
|---|---:|---:|---:|
| 15 min | +0.001517 | [-0.000185, +0.003390] | +0.572% |
| 30 min | +0.000172 | [-0.002692, +0.002801] | +0.056% |
| 60 min | +0.000130 | [-0.004125, +0.004331] | +0.037% |

All three 95% confidence intervals contain zero.

Thus, although the HGBR point estimates show slightly lower F4 MAE, the
paired daily-block bootstrap does not clearly separate those differences
from zero.

---

## HGBR Daylight-Only Results

The daylight-only HGBR results show the same pattern:

| Horizon | $\Delta$MAE (kW) | 95% CI (kW) | Relative change |
|---|---:|---:|---:|
| 15 min | +0.001832 | [-0.000683, +0.004620] | +0.468% |
| 30 min | +0.000220 | [-0.003807, +0.004054] | +0.049% |
| 60 min | +0.000347 | [-0.005377, +0.005944] | +0.067% |

Again, every interval contains zero.

Restricting evaluation to daylight periods therefore does not reveal a
clearly separated F4 advantage under HGBR.

---

## Ridge Results

The Ridge results differ substantially.

For all test timestamps:

| Horizon | $\Delta$MAE (kW) | 95% CI (kW) | Relative change |
|---|---:|---:|---:|
| 15 min | -0.010945 | [-0.014890, -0.006984] | -3.436% |
| 30 min | -0.019218 | [-0.026405, -0.012215] | -4.785% |
| 60 min | -0.035300 | [-0.048475, -0.022212] | -6.692% |

All three confidence intervals lie entirely below zero.

Under the defined sign convention, this means that F4 produced higher MAE
than F3 throughout the Ridge test comparisons.

The degradation also increases in magnitude as the forecast horizon becomes
longer.

---

## Ridge Daylight-Only Results

The daylight-only Ridge results remain negative:

| Horizon | $\Delta$MAE (kW) | 95% CI (kW) | Relative change |
|---|---:|---:|---:|
| 15 min | -0.007050 | [-0.009758, -0.004403] | -1.703% |
| 30 min | -0.012941 | [-0.018077, -0.007912] | -2.607% |
| 60 min | -0.025369 | [-0.035190, -0.015709] | -4.164% |

Every 95% interval again lies entirely below zero.

Thus the Ridge deterioration after adding F4 is not explained simply by the
inclusion of nighttime observations.

---

## Primary 60-Minute Result

The 60-minute horizon was defined in advance as the primary forecasting
horizon.

### HGBR

For all timestamps:

$$
\Delta\text{MAE}
=
+0.000130\text{ kW}
$$

with:

$$
95\%\ \text{CI}
=
[-0.004125,\;+0.004331]\text{ kW}
$$

For daylight timestamps:

$$
\Delta\text{MAE}
=
+0.000347\text{ kW}
$$

with:

$$
95\%\ \text{CI}
=
[-0.005377,\;+0.005944]\text{ kW}
$$

Both intervals contain zero.

The very small positive HGBR point estimate therefore should not be
interpreted as a clearly established Panchang forecasting improvement.

### Ridge

For all timestamps:

$$
\Delta\text{MAE}
=
-0.035300\text{ kW}
$$

with:

$$
95\%\ \text{CI}
=
[-0.048475,\;-0.022212]\text{ kW}
$$

For daylight timestamps:

$$
\Delta\text{MAE}
=
-0.025369\text{ kW}
$$

with:

$$
95\%\ \text{CI}
=
[-0.035190,\;-0.015709]\text{ kW}
$$

Both intervals lie below zero, indicating higher F4 MAE than F3 under the
Ridge model in this test period.

---

## Overall Interpretation

The uncertainty analysis strengthens the conclusion from the raw test
metrics.

The current experiment does not provide robust evidence that discrete
Panchang features improve photovoltaic forecasting beyond the continuous
astronomical variables already contained in F3.

The results instead show two different model-dependent patterns:

1. **HGBR**

   F4 produces very small numerical improvements over F3, but all paired
   daily-block 95% confidence intervals cross zero.

2. **Ridge**

   F4 consistently produces higher MAE than F3, with all paired daily-block
   confidence intervals lying below zero.

Therefore the available evidence does not support rejecting the project's
null hypothesis in favor of a general Panchang forecasting benefit.

This conclusion is specific to:

- the SOLETE photovoltaic dataset;
- the selected 5-minute resolution;
- the defined historical input window;
- the 15-, 30-, and 60-minute forecast horizons;
- the selected Ridge and HGBR models;
- the canonical Panchang representation used here;
- the fixed chronological test period.

It should not be generalized beyond the tested experimental conditions
without additional evidence.

---

## Methodological Implication

The experiment also illustrates why point estimates alone can be misleading.

For example, the 60-minute HGBR F4 result was numerically better than F3 by
approximately:

$$
0.037\%
$$

but its 95% block-bootstrap interval ranged from approximately:

$$
-1.17\%
$$

to:

$$
+1.23\%
$$

in relative MAE change.

The uncertainty interval is therefore much larger than the observed point
difference.

This makes uncertainty analysis essential when evaluating small feature-
ablation improvements in temporally dependent forecasting data.

In [27]:
# ============================================================
# Step 26: Save the completed primary experiment
# ============================================================

import json


# ------------------------------------------------------------
# 1. Create a dedicated output directory
# ------------------------------------------------------------

PRIMARY_RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "notebook09_primary_experiment"
)

PRIMARY_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Primary results directory:")
print(PRIMARY_RESULTS_DIR)


# ------------------------------------------------------------
# 2. Save the principal result tables
# ------------------------------------------------------------
#
# CSV is used for the compact result tables so they remain easy
# to inspect outside Python and easy to include in later report
# generation.
# ------------------------------------------------------------

result_tables_to_save = {
    "baseline_results.csv": baseline_results,
    "ridge_validation_selected.csv": ridge_all_best_results,
    "hgbr_validation_selected.csv": hgbr_all_best_results,
    "frozen_model_selection.csv": frozen_selection_table,
    "final_test_results.csv": final_test_results,
    "final_f3_f4_test_summary.csv": final_f3_f4_test_summary,
    "paired_block_bootstrap_summary.csv": bootstrap_summary,
}


for filename, dataframe in result_tables_to_save.items():

    output_path = (
        PRIMARY_RESULTS_DIR
        / filename
    )

    dataframe.to_csv(
        output_path,
        index=False,
    )


# ------------------------------------------------------------
# 3. Save final test predictions for every forecast horizon
# ------------------------------------------------------------
#
# These prediction files preserve the individual timestamp-level
# forecasts used to calculate the final metrics.
#
# They include:
#
#   - issue time
#   - valid time
#   - daylight flag
#   - observed target
#   - Ridge F0-F4 predictions
#   - HGBR F0-F4 predictions
# ------------------------------------------------------------

for horizon_minutes, df_predictions in (
    final_prediction_tables.items()
):

    output_path = (
        PRIMARY_RESULTS_DIR
        / (
            f"final_test_predictions_"
            f"{horizon_minutes}min.csv"
        )
    )

    # Preserve the issue-time dataframe index explicitly.
    df_to_save = df_predictions.copy()

    df_to_save.index.name = "issue_time_index"

    df_to_save.to_csv(
        output_path,
        index=True,
    )


# ------------------------------------------------------------
# 4. Save the complete bootstrap distributions
# ------------------------------------------------------------
#
# There are:
#
#   2 models
#   x 3 horizons
#   x 2 scopes
#   x 10,000 bootstrap replicates
#
# Rather than creating 12 separate files, combine them into one
# table and compress it with gzip.
# ------------------------------------------------------------

bootstrap_distribution_tables = []


for (
    model_name,
    horizon_minutes,
    scope_name,
), df_distribution in bootstrap_distributions.items():

    df_temp = df_distribution.copy()

    df_temp.insert(
        0,
        "bootstrap_replicate",
        np.arange(
            1,
            len(df_temp) + 1,
        ),
    )

    df_temp.insert(
        0,
        "scope",
        scope_name,
    )

    df_temp.insert(
        0,
        "horizon_minutes",
        horizon_minutes,
    )

    df_temp.insert(
        0,
        "model",
        model_name,
    )

    bootstrap_distribution_tables.append(
        df_temp
    )


bootstrap_distributions_combined = pd.concat(
    bootstrap_distribution_tables,
    ignore_index=True,
)


assert len(
    bootstrap_distributions_combined
) == (
    2
    * 3
    * 2
    * N_BOOTSTRAP
)


bootstrap_distributions_combined.to_csv(
    PRIMARY_RESULTS_DIR
    / "paired_block_bootstrap_distributions.csv.gz",
    index=False,
    compression="gzip",
)


# ------------------------------------------------------------
# 5. Save a machine-readable protocol record
# ------------------------------------------------------------
#
# This file records the major frozen experimental decisions
# alongside the numerical outputs.
# ------------------------------------------------------------

experiment_protocol = {
    "notebook": (
        "09_modelling_dataset_and_protocol.ipynb"
    ),

    "target": "P_Solar[kW]",

    "resolution_minutes": 5,

    "forecast_horizons_minutes": [
        15,
        30,
        60,
    ],

    "primary_horizon_minutes": 60,

    "recent_history_lags_steps": list(
        range(13)
    ),

    "daily_pv_lag_steps": 288,

    "f0_history_variables": list(
        F0_HISTORY_SOURCES.values()
    ),

    "feature_counts": {
        "F0": 118,
        "F1": 133,
        "F2": 138,
        "F3": 147,
        "F4": 244,
    },

    "split_boundaries_utc": {
        "train_start": (
            "2018-06-01T00:00:00+00:00"
        ),
        "validation_start": (
            "2019-04-01T00:00:00+00:00"
        ),
        "test_start": (
            "2019-06-01T00:00:00+00:00"
        ),
        "test_end_exclusive": (
            "2019-09-01T00:00:00+00:00"
        ),
    },

    "split_assignment": (
        "forecast-valid time"
    ),

    "deterministic_feature_time": (
        "forecast-valid time t+h"
    ),

    "solete_astronomy_offset_minutes": 2,

    "models": [
        "Ridge",
        "HistGradientBoostingRegressor",
    ],

    "ridge_alpha_grid": [
        float(value)
        for value in RIDGE_ALPHA_GRID
    ],

    "hgbr_grid": {
        "learning_rate": [
            float(value)
            for value in HGBR_LEARNING_RATES
        ],
        "max_leaf_nodes": [
            int(value)
            for value in HGBR_MAX_LEAF_NODES
        ],
        "l2_regularization": [
            float(value)
            for value in HGBR_L2_REGULARIZATION
        ],
        "max_iter": int(
            HGBR_MAX_ITER
        ),
        "early_stopping": False,
        "random_state": 42,
    },

    "primary_metric": "MAE_kW",

    "pv_rated_kw": float(
        PV_RATED_KW
    ),

    "daylight_definition": (
        "forecast-valid solar elevation > 0 degrees"
    ),

    "panchang_encoding": (
        "full fixed-vocabulary one-hot encoding"
    ),

    "panchang_indicator_count": 97,

    "bootstrap": {
        "comparison": "F3_minus_F4_MAE",
        "block_unit": (
            "forecast-valid UTC calendar day"
        ),
        "daily_blocks": 92,
        "replicates": int(
            N_BOOTSTRAP
        ),
        "random_seed": int(
            BOOTSTRAP_RANDOM_SEED
        ),
        "confidence_interval": (
            "95% percentile"
        ),
        "model_retraining": False,
    },
}


with open(
    PRIMARY_RESULTS_DIR
    / "experiment_protocol.json",
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        experiment_protocol,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 6. Save canonical Panchang vocabularies
# ------------------------------------------------------------
#
# This makes the exact F4 categorical definition recoverable
# without having to infer it from the encoded column names.
# ------------------------------------------------------------

panchang_vocabulary = {
    "tithi": list(
        TITHI_CATEGORIES
    ),
    "paksha": list(
        PAKSHA_CATEGORIES
    ),
    "karana": list(
        KARANA_CATEGORIES
    ),
    "nakshatra": list(
        NAKSHATRA_CATEGORIES
    ),
    "yoga": list(
        YOGA_CATEGORIES
    ),
}


with open(
    PRIMARY_RESULTS_DIR
    / "panchang_vocabulary.json",
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        panchang_vocabulary,
        file,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 7. Build and save a result-file manifest
# ------------------------------------------------------------

saved_files = sorted(
    path
    for path in PRIMARY_RESULTS_DIR.iterdir()
    if path.is_file()
)


manifest_rows = []


for path in saved_files:

    manifest_rows.append(
        {
            "filename": path.name,
            "size_bytes": path.stat().st_size,
            "size_kb": (
                path.stat().st_size
                / 1024
            ),
        }
    )


results_manifest = pd.DataFrame(
    manifest_rows
)


results_manifest.to_csv(
    PRIMARY_RESULTS_DIR
    / "results_manifest.csv",
    index=False,
)


# Rebuild manifest so that the manifest file itself also
# appears in the displayed audit.
saved_files = sorted(
    path
    for path in PRIMARY_RESULTS_DIR.iterdir()
    if path.is_file()
)


# ------------------------------------------------------------
# 8. Final persistence checks
# ------------------------------------------------------------

required_files = [
    "baseline_results.csv",
    "ridge_validation_selected.csv",
    "hgbr_validation_selected.csv",
    "frozen_model_selection.csv",
    "final_test_results.csv",
    "final_f3_f4_test_summary.csv",
    "paired_block_bootstrap_summary.csv",
    "paired_block_bootstrap_distributions.csv.gz",
    "final_test_predictions_15min.csv",
    "final_test_predictions_30min.csv",
    "final_test_predictions_60min.csv",
    "experiment_protocol.json",
    "panchang_vocabulary.json",
    "results_manifest.csv",
]


for filename in required_files:

    assert (
        PRIMARY_RESULTS_DIR
        / filename
    ).is_file(), (
        f"Missing saved result file: {filename}"
    )


print(
    "\nPrimary-experiment persistence check: PASSED"
)

print(
    f"\nSaved {len(required_files)} "
    "primary experiment artifacts."
)

print("\nSaved files:")

for filename in required_files:
    print(" -", filename)

Primary results directory:
/Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook09_primary_experiment

Primary-experiment persistence check: PASSED

Saved 14 primary experiment artifacts.

Saved files:
 - baseline_results.csv
 - ridge_validation_selected.csv
 - hgbr_validation_selected.csv
 - frozen_model_selection.csv
 - final_test_results.csv
 - final_f3_f4_test_summary.csv
 - paired_block_bootstrap_summary.csv
 - paired_block_bootstrap_distributions.csv.gz
 - final_test_predictions_15min.csv
 - final_test_predictions_30min.csv
 - final_test_predictions_60min.csv
 - experiment_protocol.json
 - panchang_vocabulary.json
 - results_manifest.csv


# 16. Primary Experiment Completion Checkpoint

## Experiment Status

The predefined primary forecasting experiment is now complete.

The completed workflow includes:

1. loading the canonical 5-minute SOLETE dataset;
2. applying the validated observational preprocessing pipeline;
3. constructing the fixed F0 historical / autoregressive feature block;
4. generating 15-, 30-, and 60-minute future PV targets;
5. assigning chronological train / validation / test periods by
   forecast-valid time;
6. generating forecast-valid Gregorian features;
7. generating independently computed forecast-valid solar astronomy;
8. generating continuous lunar / Sun–Moon astronomical features;
9. generating canonical Lahiri-based Panchang features;
10. applying the fixed 97-column Panchang one-hot representation;
11. constructing the nested F0–F4 modelling matrices;
12. evaluating persistence and daily-persistence baselines;
13. tuning Ridge Regression using the chronological validation period;
14. tuning Histogram Gradient Boosting using the same validation protocol;
15. freezing all selected model configurations before test evaluation;
16. refitting the frozen models using training + validation data;
17. performing one final evaluation on the untouched chronological test set;
18. evaluating both all-time and daylight-only performance;
19. performing a paired 10,000-replicate daily-block bootstrap for the
    F3 → F4 comparison;
20. persisting the complete primary experimental outputs.

---

## Primary Scientific Finding

The primary experiment does not provide robust evidence that the discrete
Panchang representation improves photovoltaic forecasting beyond the
continuous astronomical information already contained in F3.

For the primary 60-minute horizon:

### HGBR

The F3 → F4 all-time MAE change was approximately:

$$
+0.037\%
$$

but the paired daily-block 95% confidence interval crossed zero:

$$
[-1.17\%,\;+1.23\%]
$$

Therefore the small numerical F4 improvement is not clearly separated from
zero under the predefined uncertainty analysis.

### Ridge

The F3 → F4 all-time MAE change was approximately:

$$
-6.69\%
$$

with a 95% confidence interval entirely below zero:

$$
[-9.30\%,\;-4.19\%]
$$

Thus F4 produced higher test MAE than F3 under Ridge.

---

## Additional Forecasting Finding

The deterministic feature families were not uniformly beneficial.

For the primary 60-minute HGBR experiment, the lowest test MAE among F0–F4
was obtained by:

**F1 — historical observations + Gregorian future-time features**

with:

$$
\text{MAE}
=
0.345082\text{ kW}
$$

compared with:

$$
\text{F0 MAE}
=
0.426420\text{ kW}
$$

This shows that deterministic future-time structure can provide useful
forecasting information even though progressively richer astronomical and
Panchang representations do not necessarily provide additional gains.

---

## Persisted Results

The complete primary experiment has been saved under:

`results/notebook09_primary_experiment/`

The saved artifacts include:

- baseline metrics;
- validation-selected Ridge configurations;
- validation-selected HGBR configurations;
- the frozen model-selection table;
- final test metrics;
- F3 → F4 test summaries;
- paired block-bootstrap summaries;
- all 10,000 bootstrap replicates;
- timestamp-level final test predictions for all three horizons;
- the machine-readable experimental protocol;
- the canonical Panchang vocabulary;
- a results manifest.

These files allow the primary conclusions and future figures to be
reconstructed without repeating the complete model-training process.

---

## Experimental Boundary

Everything up to this point belongs to the **predefined primary experiment**.

Any subsequent investigation, including:

- alternative lag windows;
- expanded hyperparameter searches;
- different Panchang encodings;
- alternative chronological splits;
- rolling-origin validation;
- 1-minute resolution experiments;
- wind-power forecasting;
- VMD decomposition;
- neural networks;
- alternative model families;
- additional astronomical features;

must be labeled as a:

**sensitivity analysis, robustness analysis, secondary experiment, or
follow-up experiment**

rather than being incorporated retrospectively into the primary experiment.

This distinction is important because the primary protocol was frozen before
the final test results were observed.

---

## Primary Experiment Complete

The main F0–F4 photovoltaic forecasting experiment is now complete.

The next stage should focus on reproducibility checks, sensitivity analyses,
secondary targets, or advanced forecasting methods without altering the
reported primary result.